### Initialization

In [ ]:
from qubox.cQED_experiments import cQED_Experiment
from qubox.pulse_manager import PulseOp
from qubox.analysis import *
from qubox.tools.waveforms import *
from qubox.gates_legacy import *
from qubox.analysis.analysis_tools import *
from qubox.macros.measure_macro import measureMacro
from qubox.program_GUI import run_gui
from qubox.gates.contexts import HardwareContext
import numpy as np
import matplotlib.pyplot as plt
from qualang_tools.units import unit
from qubox.tools.generators import register_rotations_from_ref_iq
from qubox.analysis.cQED_plottings import display_fock_populations
"""
with open("requirements.txt", "w") as f:
    for dist in pkg_resources.working_set:+
        line = f"{dist.project_name}=={dist.version}\n"
        f.write(line)print("Saved all installed packages to requirements.txt")
"""
experiment_path = "data/seq_1_device"
experiment = cQED_Experiment(
    experiment_path,
    qop_ip="10.157.36.68",
    cluster_name="Cluster_2",
    oct_cal_path="./",
    override_octave_json_mode="on",
    output_mode="on",
    load_devices=["signalcore_pump"]
    
)
#load_devices=["signalcore_ro"]
u = unit()
Gate.set_attributes(experiment.pulseOpMngr, experiment.attributes)
#experiment.quaProgMngr.set_exec_mode("simulate")
pulseInfo = experiment.pulseOpMngr.get_pulseOp_by_element_op(element="resonator", op="readout")
experiment.load_measureMacro_state()

pom = experiment.pulseOpMngr
qpm = experiment.quaProgMngr
#sc = experiment.device_manager.get("signalcore_pump", connect=False)
#experiment.quaProgMngr.set_spa_pump(sc)

### Mixer Calibration

In [ ]:
attr = experiment.attributes
progMngr = experiment.quaProgMngr
elements = [attr.ro_el, attr.qb_el, attr.st_el]
el_los = [progMngr.get_element_lo(el) for el in elements]
el_ifs = [progMngr.calculate_el_if_fq(el, fq) for el, fq in zip(elements, [attr.ro_fq, attr.qb_fq, attr.st_fq])]
progMngr.calibration_el_octave(el=elements, target_LO=el_los, target_IF=el_ifs)
#progMngr.calibration_el_octave(el=elements, target_LO=el_los, target_IF=[-50e6, -50e6, -50e6])

### Cont. waves

In [ ]:
elements = ["resonator", "qubit", "storage", "qubit2"]
el_freqs = [-50e6, -50e6, -50e6, -50e6]
pulses = ["const", "const" , "const", "const"]
temp = experiment.quaProgMngr.run_continuous_wave(elements, el_freqs, pulses, gain=0.1)

In [ ]:
elements = ["qubit"]
el_freqs = [-159e6]
pulses = [ "const"]
temp = experiment.quaProgMngr.run_continuous_wave(elements, el_freqs, pulses, gain=1)

In [ ]:
experiment.quaProgMngr.set_octave_gain("qubit2", -19)

In [ ]:
experiment.quaProgMngr.halt_job()

# Simple Readout Experiments

### Time of Flight

In [ ]:
sc = experiment.device_manager.get("signalcore_pump", connect=False)
sc.do_set_output_status(True)
sc.do_set_frequency(2*experiment.attributes.ro_fq+10e6)
sc.do_set_power(10)

In [ ]:

ro_fq_hz      = 8597e6   # or: experiment.attributes.ro_fq
#ro_fq_hz      = 8_600_000_000.0
ro_therm_clks = 3_000
n_avg         = 10000

# Run
rr = experiment.readout_trace(ro_fq_hz, ro_therm_clks=ro_therm_clks, n_avg=n_avg)

if rr.mode == "simulate":
    print("Simulation run — raw ADC streams not produced; nothing to analyze.")
else:
    out = rr.output
    adc1, adc2, adc1_mean, adc2_mean = out.extract(
        "adc1", "adc2", "adc1_mean", "adc2_mean"
    )

    dt_ns = 1.0  # change if your acquisition sampling differs
    t_ns  = np.arange(len(adc1)) * dt_ns

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(t_ns, adc1, label=f"ADC1 (I), mean = {adc1_mean:.6f} V")
    plt.plot(t_ns, adc2, label=f"ADC2 (Q), mean = {adc2_mean:.6f} V")
    plt.title("Readout Raw Traces")
    plt.ylabel("Signal [V]")
    plt.xlabel("Time [ns]")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Suggested offset correction (I, Q) = ({-adc1_mean:.6g}, {-adc2_mean:.6g}) V")


In [ ]:
experiment.quaProgMngr.serialize_program()

### resonator spectroscopy

In [ ]:
experiment.set_readout_SPA_pump_power(9)
experiment.set_readout_SPA_pump_frequency(experiment.attributes.ro_fq*2+10e6)

In [ ]:
readout_op = "readout_test"
length     = 1000
total_length = length 

I_wf = 0.005

pom = experiment.pulseOpMngr  # convenience alias

pom.create_measurement_pulse(
    element=experiment.attributes.ro_el,
    op=readout_op,
    length=total_length,
    pulse_name=f"{readout_op}_pulse",   # "readout_test_pulse"
    I_samples=I_wf,
    digital_marker="ON",
    int_weights_mapping=None,  
    int_weights_defs=None,
    persist=False,                      # VOLATILE (like your old persist=False)
    override=True,                      # overwrite if you re-run this
)

experiment.burn_pulses()

#run_gui(experiment.resonator_spectroscopy)
n_avg      = 10000
freq_start = 8560 * u.MHz
freq_end   = 8640 * u.MHz
df         = 200 * u.kHz
result = experiment.resonator_spectroscopy(readout_op, freq_start, freq_end, df, n_avg=n_avg)
if result.mode == "simulate":
    print("Simulation run, no data to analyze.")
else:
    frequencies, magnitude = result.output.extract("frequencies", "magnitude")
    p0 = [np.average([freq_start, freq_end])*1e-6, 1, -1e-4, 0]
    custom_plot_opts = {"figsize": (10, 6), "xlabel": "frequency (MHz)", "ylabel": "R", "title": "Resonator Spectroscopy Fit", "legend_fontsize": 16}
    fit_result = fitting.generalized_fit(frequencies*1e-6, magnitude, cQED_models.resonator_spec_model, p0, plotting=True, plot_options=custom_plot_opts, param_format="{:.4f}")

In [ ]:
experiment.quaProgMngr.serialize_program()

In [ ]:
experiment.attributes.ro_fq = fit_result[0][0] * 1e6  # in Hz
experiment.save_attributes()
experiment.attributes.ro_fq
measureMacro.set_drive_frequency(experiment.attributes.ro_fq)

### resonator amplitude cheveron

In [ ]:
experiment.quaProgMngr.set_simulate_params(True, duration=4000, plot=True)

In [ ]:
# --- define/readout pulse for the resonator ---
readout_op = "readout_test"
amplitude  = 0.3  # Volts peak of your readout I waveform

pom = experiment.pulseOpMngr  # convenience alias

pom.create_measurement_pulse(
    element="resonator",      # same as before
    op=readout_op,            # "readout_test"
    length=1000,
    # Let the manager auto-name the waveforms; just give samples:
    I_samples=amplitude,      # scalar -> constant waveform at 0.1
    Q_samples=0.0,            # scalar -> constant waveform at 0.0 (zero on Q)
    digital_marker="ON",
    int_weights_mapping=None,  # default cos/sin/minus_sin triplet for this length
    int_weights_defs=None,
    persist=False,             # volatile, like your old persist=False
    override=True,             # overwrite if it already exists
)

# push into the active QM config (equivalent to burn=True)
experiment.burn_pulses()

# --- sweep knobs ---
rf_begin = 8590 * u.MHz
rf_end   = 8610 * u.MHz
df         = 100 * u.kHz

# run the experiment (returns RunResult)
rr = experiment.resonator_power_spectroscopy(readout_op, rf_begin, rf_end, df, g_min=1e-1, g_max=1.2, N_a=100, n_avg=500)

if rr.mode == "simulate":
    print("Simulation run — no stream data; skipping heatmaps.")
else:
    out = rr.output
    freqs, gains, S, phases = out.extract("frequencies", "gains", "S", "Phases")          # shapes: (Nf,), (Na,)
    # default_proc (or your pipeline) should have produced these:

    # x-axis in MHz (if your 'frequencies' are in Hz)
    f_mhz = np.asarray(freqs) * 1e-6

    # y-axis in millivolts (actual applied voltage = amplitude * gain)
    v_mV = (amplitude * np.asarray(gains)) * 1e3

    # helper: convert centers → edges for pcolormesh
    def centers_to_edges(x):
        x = np.asarray(x)
        dx = np.diff(x)
        left  = x[0]  - dx[0] / 2.0
        right = x[-1] + dx[-1] / 2.0 if len(dx) else x[0] + 0.5
        mids  = (x[:-1] + x[1:]) / 2.0
        return np.concatenate(([left], mids, [right]))

    f_edges  = centers_to_edges(f_mhz)
    v_edges  = centers_to_edges(v_mV)

    # If your S_volts came out (Nf, Na), flip it to (Na, Nf)
    Zmag = np.abs(np.asarray(S))
    if Zmag.shape == (len(f_mhz), len(v_mV)):
        Zmag = Zmag.T
    Zph  = np.asarray(phases)
    if Zph.shape == (len(f_mhz), len(v_mV)):
        Zph = Zph.T

    # --- |S| heatmap ---
    plt.figure(figsize=(9, 6))
    pcm = plt.pcolormesh(f_edges, v_edges, Zmag, shading="auto")
    plt.colorbar(pcm, label="|S|")
    plt.xlabel("Frequency [MHz]")
    plt.ylabel("Applied readout trace")
    plt.title("Resonator Power Spectroscopy — Magnitude")
    plt.tight_layout()
    plt.show()

    # --- phase heatmap ---
    plt.figure(figsize=(9, 6))
    pcm = plt.pcolormesh(f_edges, v_edges, Zph, shading="auto")
    plt.colorbar(pcm, label="Phase [rad]")
    plt.xlabel("Frequency [MHz]")
    plt.ylabel("Applied readout trace")
    plt.title("Resonator Power Spectroscopy — Phase")
    plt.tight_layout()
    plt.show()


# Qubit Experiments/Characterization

## Basic Characterization

### qubit spectroscopy

In [ ]:
n_avg      = 10000
freq_start = 6130 * u.MHz
freq_end   = 6170 * u.MHz
df         = 500 * u.kHz

qb_gain = 1.0
qb_len  = None
pulse = "x180"

rr = experiment.qubit_spectroscopy(
    pulse, freq_start, freq_end, df,
    qb_gain=qb_gain, qb_len=qb_len, n_avg=n_avg
)

if rr.mode == "simulate":
    print("Simulation run — no stream data; skipping analysis/plot.")
else:
    out = rr.output
    freqs, S, phases = out.extract("frequencies", "S", "Phases")

    # axes in MHz
    f_mhz = np.asarray(freqs) * 1e-6
    mag   = np.abs(np.asarray(S))

    ro_fq_mhz = experiment.attributes.ro_fq * 1e-6

    fig, ax1 = plt.subplots(figsize=(10, 6))
    # magnitude on left axis
    ax1.plot(f_mhz, mag, label="|S|", linewidth=1.5)
    ax1.set_xlabel("Frequency [MHz]")
    ax1.set_ylabel("adc trace (arb. units)")
    ax1.tick_params(axis="y")

    # phase on right axis (dashed)
    ax2 = ax1.twinx()
    ax2.plot(f_mhz, np.asarray(phases), label="Phase", linestyle="--")
    ax2.set_ylabel("Phase [rad]")
    ax2.tick_params(axis="y")

    # compose a single legend
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="best")

    plt.title(
        f"Qubit Spectroscopy — ro_fq={ro_fq_mhz:.3f} MHz, "
        f"qb_gain={qb_gain}, qb_len={qb_len/u.us:.3f} µs"
    )
    fig.tight_layout()
    plt.show()

    # optional: peak find on phase dip (as in your call)
    peak_results = algorithms.find_peaks(f_mhz, -np.asarray(phases), 3, min_distance=5)
    print(peak_results)

In [ ]:
experiment.attributes.fock_fqs

In [ ]:
experiment.attributes.qb_fq

### pi/N pulse waveform definition (const)

In [ ]:
experiment.pulseOpMngr.display_op("qubit", "x180", freq_range=(-200, 200))

In [ ]:
def const_square_waveforms(
    amplitude_I: float,
    amplitude_Q: float,
    length: int,
):
    """
    Build a pair (I_wf, Q_wf) for a square (constant) pulse.

    amplitude_I : value on I channel (in IQ mixer units)
    amplitude_Q : value on Q channel
    length      : number of clock cycles / samples

    Returns:
        I_wf, Q_wf  (lists of length `length`)
    """
    I_wf = [amplitude_I] * length
    Q_wf = [amplitude_Q] * length
    return I_wf, Q_wf

# --- calibration numbers for the constant pulses ---
const_r180_amp = 0.02488 / 2 * 0.8129   # calibrated "pi" level for square pulse
const_len      = 200                    # length in clocks/samples
const_r90_amp  = const_r180_amp / 2.0

# --- build the raw waveforms ---
I_x180,  Q_x180  = const_square_waveforms(const_r180_amp,      0.0,           const_len)

pom = experiment.pulseOpMngr
res = register_rotations_from_ref_iq(pom, I_x180, Q_x180, rotations="all", make_r0="True", prefix="const_", override=True, persist=True)
experiment.burn_pulses()
experiment.save_pulses()


### pi/N pulse waveform definition (gaussian)

In [ ]:
0.08875937412663888 * 1.0121457489878543*1.2428


In [ ]:
import numpy as np

attr = experiment.attributes
rlen  = 16
sigma = rlen / 6

r180_amp = 0.08875937412663888 * 1.0121457489878543*1.2428
drag_coeff = -0.0428

gauss_r180, drag_r180 = drag_gaussian_pulse_waveforms(
    r180_amp, rlen, sigma, drag_coeff, attr.anharmonicity
)

pom = experiment.pulseOpMngr

# Optional: per-gate fine tweaks (start with none)


d_lambda_map = {
    "x90": -1.389434e+06,
    "y90": -1.389434e+06,
    "xn90":-1.389434e+06,
    "yn90":-1.389434e+06,
    "x180":-1173860.4004705912,
    "y180":-1173860.4004705912,
}
d_alpha_map = {
     "y90": 0.022700,
     "yn90": 0.022700,  
     "x90":0.022700,  
     "xn90": 0.022700,   
     "x180":0.006529648670673627,
     "y180":0.006529648670673627,
}
d_omega_map = {
     "y90": 0,
     "yn90": 0,  
     "x90":0,  
     "xn90": 0,  
}

res = register_rotations_from_ref_iq(
    pom,
    gauss_r180,        # ref_I  (your x180 template I)
    drag_r180,         # ref_Q  (your x180 template Q)
    element="qubit",   # or whatever your qubit element name is
    rotations="all",
    make_r0=True,      # <-- bool, not "True"
    override=True,
    persist=True,
    d_lambda_map=d_lambda_map,
    d_alpha_map=d_alpha_map,
    d_omega_map=d_omega_map,
)

experiment.burn_pulses()
experiment.save_pulses()


### power rabi

In [ ]:
max_gain = 1.2
dg       = 0.04     
rr = experiment.power_rabi(max_gain, dg, op="sel_ref_r180", truncate_clks=None, n_avg=5000)

if rr.mode == "simulate":
    print("Simulation run — no stream data; skipping fit.")
else:
    out = rr.output
    gains, S = out.extract(
        "gains", "S"
    )
    C0, V0, eta, phi = -1, max(S.real) - min(S.real), np.pi, 0
    p0 = (C0, V0, eta, phi)
    fit_params = fitting.generalized_fit(
        gains,
        S.real,
        cQED_models.sinusoid_pe_model,
        p0,
        plotting=True,
        plot_options={
            "figsize": (12, 5),
            r"xlabel": r"qubit amplitude",
            "ylabel": "amplitudes",
            "title": "Power Rabi",
            "legend_fontsize": 16,
        },
        param_format="{:.4f}",
    )
    C, V , eta, phi = fit_params[0]
    a_pi = (np.pi - phi) / eta
    a_pi_2 = (np.pi / 2 - phi) / eta
    print(f"Calculated a_pi: {a_pi:.4f}, a_pi/2: {a_pi_2:.4f}")

    A0, g_pi0, offset0 = max(S.real) - min(S.real), 1, 1,
    p0 = (A0, g_pi0, offset0)
    
    fit_params = fitting.generalized_fit(
        gains,
        S.real,
        cQED_models.power_rabi_model,
        p0,
        plotting=True,
        plot_options={
            "figsize": (12, 5),
            r"xlabel": r"qubit amplitude",
            "ylabel": "amplitudes",
            "title": "Power Rabi",
            "legend_fontsize": 16,
        },
        param_format="{:.4f}",
    )
cQED_plottings.plot_IQ(S, s=1)

In [ ]:
print(max(S))
measureMacro.compute_Pe_from_S(max(S))


### single qb sequential rotations

In [ ]:

rotations = ["sel_x180"]
n_shots     = 200_000

rr = experiment.sequential_qb_rotations(rotations, apply_avg=False, n_shots=n_shots)

out = rr.output
S, phases, state_flag = out.extract(
    "S", "Phases", "state_flag"
)
cQED_plottings.plot_IQ(S, s=1)
S.mean().real

In [ ]:
S.real

In [ ]:
S.mean().real

### IQ blob

In [ ]:
rr = experiment.iq_blob("x180", n_runs=50000)

S_g, S_e = rr.output.extract("S_g", "S_e")
out = analysis_tools.two_state_discriminator(S_g, S_e, b_plot=True)

In [ ]:
model_type = "1d"
w_g_given_g, w_e_given_g = measureMacro.compute_posterior_weights(S_g, model_type=model_type)
w_g_given_e, w_e_given_e = measureMacro.compute_posterior_weights(S_e, model_type=model_type)
print(w_g_given_g.mean(), w_e_given_g.mean())
print(w_g_given_e.mean(), w_e_given_e.mean())

In [ ]:
measureMacro.compute_posterior_weights(-20*8.43517482e-05+1.43293291e-04j)

In [ ]:
pos_g, pos_e = measureMacro._post_select_config.posterior_weights(S_g)
print(pos_g.mean())
print(pos_e.mean())



In [ ]:
pos_g, pos_e = measureMacro._post_select_config.posterior_weights(S_e)
print(pos_g.mean())
print(pos_e.mean())

### qubit |e> -> |f> spectroscopy

In [ ]:
n_avg      = 10000
freq_start = 5894.5 * u.MHz
freq_end   = 5895.5 * u.MHz
df         = 50 * u.kHz

qb_gain = 0.1
qb_len  = 5 * u.us
pulse = "saturation"
rr = experiment.qubit_spectroscopy_ef(
    pulse, freq_start, freq_end, df, qb_gain=qb_gain, qb_len=qb_len, n_avg=n_avg
)

if rr.mode == "simulate":
    print("Simulation run — no stream data; skipping plot.")
else:
    out = rr.output
    frequencies, S, phases = out.extract(
        "frequencies", "S", "Phases"
    )
    ro_fq = experiment.attributes.ro_fq

    f_mhz = np.asarray(frequencies) * 1e-6

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(f_mhz, np.abs(S), label="Amplitude", linewidth=1.6)
    ax1.set_xlabel("Frequency [MHz]")
    ax1.set_ylabel("Amplitude")
    ax1.tick_params(axis="y")

    ax2 = ax1.twinx()
    ax2.plot(f_mhz, np.asarray(phases), label="Phase", linestyle="--")
    ax2.set_ylabel("Phase [rad]")
    ax2.tick_params(axis="y")

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="best")

    plt.title(
        f"Qubit |e⟩ → |f⟩ Spectroscopy — ro_fq={ro_fq:.3f} MHz, "
        f"qb_gain={qb_gain}, qb_len={qb_len/u.us:.3f} µs"
    )
    fig.tight_layout()
    plt.show()

    min_index, value = algorithms.argmin_general(np.abs(S))
    qb_fq_ef = frequencies[min_index]
    print("qubit e->f frequency", qb_fq_ef)

### T1 relaxation

In [ ]:
# ---------- T1 relaxation ----------
delay_end = 80 * u.us
dt             = 500 

rr = experiment.T1_relaxation(delay_end, dt, n_avg=2000)

if rr.mode == "simulate":
    print("Simulation run — no stream data; skipping T1 fit.")
else:
    out = rr.output
    delays, S, phases = out.extract("delays", "S", "Phases",)
    T1_fit_parms = fitting.generalized_fit(
        delays * 1e-3,   # ns -> µs (assuming 'delays' are in ns as per your earlier convention)
        S.real,
        cQED_models.T1_relaxation_model,
        [0, 10, 0],
        plotting=True,
        plot_options={"figsize": (10, 6), r"xlabel": r"delays ($\mu s$)", "ylabel": "ampltiudes",
                      "title": "T1 relaxation", "legend_fontsize": 16},
        param_format="{:.4f}",
    )
A, T1, offset = T1_fit_parms[0]
T1_in_ns = T1 * 1e3  # µs -> ns
experiment.attributes.qb_T1_relax = T1_in_ns
T1_in_us = T1
experiment.attributes.qb_therm_clks = int(2*experiment.attributes.qb_T1_relax)
experiment.save_attributes()

In [ ]:
measureMacro._ro_disc_params

### T1 as a function of second low pump readout output

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from qubox.logging_config import temporarily_set_levels

# --- quiet down qubox + qm logs during the sweep ---
qubox_logger = logging.getLogger("qubox")
qm_logger    = logging.getLogger("qm")

with temporarily_set_levels([qubox_logger, qm_logger], logging.WARNING):

    # ----------------------------
    # Setup
    # ----------------------------
    sc = experiment.device_manager.get("signalcore_ro", connect=False)
    sc.do_set_output_status(True)

    base_f = float(experiment.attributes.ro_fq)   # Hz

    delay_end = 50 * u.us
    dt = 1000
    n_avg = 2500

    powers_in_dbm    = np.linspace(-20, 10, 30)
    detunings_hz     = np.linspace(-40e6, +40e6, 20)
    fit_p0           = [0, 10, 0]     # [A, T1_us, offset]
    plot_single_fits = False

    # ----------------------------
    # Storage for raw traces (for debugging fits)
    # ----------------------------
    trace_store = {
        "meta": {
            "base_f_Hz": base_f,
            "delay_end": delay_end,
            "dt": dt,
            "n_avg": n_avg,
            "fit_p0": fit_p0,
            "powers_in_dbm": powers_in_dbm,
            "detunings_hz": detunings_hz,
        },
        "points": []
    }

    # ----------------------------
    # Sweep: detuning x power
    # ----------------------------
    T1_map_us = np.full((len(detunings_hz), len(powers_in_dbm)), np.nan, dtype=float)

    total_pts = len(detunings_hz) * len(powers_in_dbm)
    pbar = tqdm(total=total_pts, desc="T1 sweep (detuning × power)", unit="pt")

    try:
        for i, detuning in enumerate(detunings_hz):
            f = base_f + float(detuning)
            sc.do_set_frequency(f)

            for j, power in enumerate(powers_in_dbm):
                sc.do_set_power(float(power))

                # Update progress bar display (keeps notebook output clean)
                pbar.set_postfix(
                    det_MHz=f"{detuning/1e6:+.2f}",
                    P_dBm=f"{power:.1f}",
                    refresh=False
                )

                rr = experiment.T1_relaxation(delay_end, dt, n_avg=n_avg)
                if getattr(rr, "mode", None) == "simulate":
                    trace_store["points"].append(
                        dict(
                            detuning_hz=float(detuning),
                            power_dbm=float(power),
                            frequency_hz=float(f),
                            mode="simulate",
                            delays=None,
                            delays_us=None,
                            S=None,
                            phases=None,
                            fit=None,
                        )
                    )
                    pbar.update(1)
                    continue

                out = rr.output
                delays, S, phases = out.extract("delays", "S", "Phases")

                # Your convention: delays in ns -> convert to µs for fit
                delays_us = delays * 1e-3
                y = np.real(S)

                T1_fit_parms = fitting.generalized_fit(
                    delays_us,
                    y,
                    cQED_models.T1_relaxation_model,
                    fit_p0,
                    plotting=plot_single_fits,
                    plot_options={
                        "figsize": (10, 6),
                        "xlabel": r"delays ($\mu s$)",
                        "ylabel": "amplitudes",
                        "title": "T1 relaxation",
                        "legend_fontsize": 16,
                    },
                    param_format="{:.4f}",
                )

                A, T1_us, offset = T1_fit_parms[0]
                T1_map_us[i, j] = float(T1_us)

                # Save trace for debugging later
                trace_store["points"].append(
                    dict(
                        detuning_hz=float(detuning),
                        power_dbm=float(power),
                        frequency_hz=float(f),
                        mode=getattr(rr, "mode", "run"),
                        delays=np.array(delays, copy=True),
                        delays_us=np.array(delays_us, copy=True),
                        S=np.array(S, copy=True),
                        phases=np.array(phases, copy=True) if phases is not None else None,
                        fit=(float(A), float(T1_us), float(offset)),
                    )
                )

                pbar.set_postfix(
                    det_MHz=f"{detuning/1e6:+.2f}",
                    P_dBm=f"{power:.1f}",
                    T1_us=f"{T1_us:.2f}",
                    refresh=False
                )
                pbar.update(1)

    finally:
        pbar.close()

    # ----------------------------
    # Save all traces to disk (NPZ)
    # ----------------------------
    np.savez_compressed("T1_traces_detuning_power.npz", trace_store=trace_store)
    print("\nSaved raw traces to: T1_traces_detuning_power.npz")
    print("Reload with: d=np.load('T1_traces_detuning_power.npz', allow_pickle=True); trace_store=d['trace_store'].item()")

    # ----------------------------
    # Plot 1: heatmap
    # ----------------------------
    fig, ax = plt.subplots(figsize=(11, 6))
    im = ax.imshow(
        T1_map_us,
        aspect="auto",
        origin="lower",
        extent=[
            float(powers_in_dbm[0]), float(powers_in_dbm[-1]),
            float(detunings_hz[0]/1e6), float(detunings_hz[-1]/1e6),
        ],
    )
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("T1 (µs)")
    ax.set_xlabel("Power (dBm)")
    ax.set_ylabel("Detuning from ro_fq (MHz)")
    ax.set_title("T1 vs power and detuning")
    ax.minorticks_on()
    plt.show()

    # ----------------------------
    # Plot 2: linecuts
    # ----------------------------
    fig, ax = plt.subplots(figsize=(11, 6))
    for i, detuning in enumerate(detunings_hz):
        ax.plot(
            powers_in_dbm,
            T1_map_us[i, :],
            marker="o",
            markersize=3,
            linewidth=1,
            label=f"{detuning/1e6:+.1f} MHz",
        )
    ax.set_xlabel("Power (dBm)")
    ax.set_ylabel("T1 (µs)")
    ax.set_title("T1 vs power (colored by detuning)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(title="Detuning", ncols=2, fontsize=9)
    plt.show()


# ----------------------------
# (Optional) helper: quick-look a saved trace later
# ----------------------------
def plot_saved_trace(trace_store, detuning_hz, power_dbm, tol_det_hz=1.0, tol_pow_db=1e-6):
    """
    Find the matching point (within tolerances) and plot Re(S) vs delays_us.
    """
    target = None
    for p in trace_store["points"]:
        if p["S"] is None:
            continue
        if abs(p["detuning_hz"] - detuning_hz) <= tol_det_hz and abs(p["power_dbm"] - power_dbm) <= tol_pow_db:
            target = p
            break

    if target is None:
        raise ValueError("No matching trace found. Check tolerances / values.")

    x = target["delays_us"]
    y = np.real(target["S"])
    fit = target["fit"]

    plt.figure(figsize=(9, 5))
    plt.plot(x, y, "o-", markersize=3, linewidth=1, label="Re(S)")
    plt.xlabel(r"delay ($\mu s$)")
    plt.ylabel("Re(S)")
    plt.title(f"Trace: det={target['detuning_hz']/1e6:+.2f} MHz, P={target['power_dbm']:.2f} dBm")
    plt.grid(True, alpha=0.3)

    if fit is not None:
        A, T1_us, offset = fit
        try:
            t = np.linspace(np.min(x), np.max(x), 400)
            yfit = cQED_models.T1_relaxation_model(t, A, T1_us, offset)
            plt.plot(t, yfit, "-", linewidth=2, label=f"fit T1={T1_us:.3f} µs")
        except Exception:
            pass

    plt.legend()
    plt.show()


### T2 ramesy

In [ ]:
qb_detune = 0.2 * u.MHz
delay_end = 40 * u.us
dt        = 100
runres = experiment.T2_ramsey(
    qb_detune, delay_end, dt, r90="x90",n_avg=4000
)

delays, S = runres.output.extract("delays", "S")

delays_us = delays * 1e-3  # ns -> µs
fit_p0 = [0, 20, 1, qb_detune*1e-6, 0, 0]
T2_fit_params = fitting.generalized_fit(
    delays_us,
    S.real,
    cQED_models.T2_ramsey_model,
    fit_p0,
    plotting=True,
    plot_options={
        "figsize": (12, 6),
        "xlabel": r"delays ($\mu s$)",
        "ylabel": "amplitudes",
        "title": "T2 Ramsey",
        "legend_fontsize": 16,
    },
    param_format="{:.6f}",
)

A, T2_us, n, fitted_det_MHz, phi, offset = T2_fit_params[0]


In [ ]:
qb_detune_Hz = qb_detune
fitted_det_Hz = float(fitted_det_MHz) * 1e6
experiment.attributes.qb_fq = experiment.attributes.qb_fq + (qb_detune_Hz - fitted_det_Hz)
experiment.save_attributes()

In [ ]:
sc = experiment.device_manager.get("signalcore_ro", connect=False)
sc.do_set_power(-20.0)
detuning = -20e6
sc.do_set_frequency(experiment.attributes.ro_fq + detuning)
sc.do_set_output_status(True)

In [ ]:
import logging, time
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from qubox.logging_config import temporarily_set_levels

# Assumes you already have these in your environment:
#   fitting, cQED_models, and units 'u' (e.g., from qubox import units as u)

qubox_logger = logging.getLogger("qubox")
qm_logger    = logging.getLogger("qm")


def run_reference_ramsey(
    experiment,
    sc,
    qb_detune,
    delay_end,
    dt,
    n_avg_ref=20000,
    fit_p0=None,
    plot=True,
    plot_title="Reference T2 Ramsey (SignalCore OFF)",
):
    """
    Runs a reference Ramsey with SignalCore output OFF.
    Fits using: T2_ramsey_model(t, A, T2, n, f_det, phi, offset)
    Returns dict with fitted params + raw trace.
    """
    sc.do_set_output_status(False)

    rr = experiment.T2_ramsey(qb_detune, delay_end, dt, n_avg=n_avg_ref)
    if getattr(rr, "mode", None) == "simulate":
        raise RuntimeError("Reference Ramsey returned simulate mode; no data to fit.")

    out = rr.output
    delays, S, phases = out.extract("delays", "S", "Phases")

    delays_us = delays * 1e-3   # ns -> µs
    y = np.real(S)

    if fit_p0 is None:
        qb_detune_MHz = float(qb_detune * 1e-6)
        fit_p0 = [0, 25, 1, qb_detune_MHz, 0, 0]  # [A, T2_us, n, det_MHz, phi, offset]

    T2_fit_params = fitting.generalized_fit(
        delays_us,
        y,
        cQED_models.T2_ramsey_model,
        fit_p0,
        plotting=bool(plot),
        plot_options={
            "figsize": (12, 6),
            "xlabel": r"delays ($\mu s$)",
            "ylabel": "amplitudes",
            "title": plot_title,
            "legend_fontsize": 16,
        } if plot else None,
        param_format="{:.6f}",
    )
    A, T2_us, n, fitted_det_MHz, phi, offset = T2_fit_params[0]

    print(f"[REF] Fitted detuning: {float(fitted_det_MHz):+.6f} MHz")
    print(f"[REF] Fitted T2*:      {float(T2_us):.3f} µs")
    print(f"[REF] Fitted n:        {float(n):.4f}")
    print(f"[REF] Fitted phi:      {float(phi):+.4f}")
    print(f"[REF] Fitted offset:   {float(offset):+.6f}")
    print(f"[REF] Fitted A:        {float(A):+.6f}")

    return dict(
        A=float(A),
        T2_us=float(T2_us),
        n=float(n),
        f_det_MHz=float(fitted_det_MHz),
        phi=float(phi),
        offset=float(offset),
        delays=np.array(delays, copy=True),
        delays_us=np.array(delays_us, copy=True),
        S=np.array(S, copy=True),
        phases=np.array(phases, copy=True) if phases is not None else None,
        n_avg_ref=int(n_avg_ref),
        fit_p0_used=list(fit_p0),
    )


# ============================================================
# 0) Setup
# ============================================================
sc = experiment.device_manager.get("signalcore_ro", connect=False)

base_f = float(experiment.attributes.ro_fq)   # Hz

# Ramsey settings
delay_end = 20 * u.us
dt        = 48 * u.ns
n_avg     = 1000

qb_detune = 0.2 * u.MHz
qb_detune_MHz = float(qb_detune * 1e-6)

# Sweep knobs (SC still set in dBm, but we will PLOT in linear power)
powers_in_dbm = np.linspace(-20, 0, 21)
detunings_hz  = np.linspace(-25e6, +250e6, 11)

settle_s = 0.02

# ============================================================
# 1) Reference Ramsey OUTSIDE temporarily_set_levels
# ============================================================
sc.do_set_frequency(base_f)
sc.do_set_power(float(powers_in_dbm[0]))
time.sleep(settle_s)

ref_fit_p0 = [0, 25, 1, qb_detune_MHz, 0, 0]
ref = run_reference_ramsey(
    experiment=experiment,
    sc=sc,
    qb_detune=qb_detune,
    delay_end=delay_end,
    dt=dt,
    n_avg_ref=5000,
    fit_p0=ref_fit_p0,
    plot=True,
    plot_title="Reference T2 Ramsey (SC OFF, high avg)",
)

# "Global" guess vector from reference (only det guess will be overwritten each iteration)
ref_guess = [
    ref["A"],
    ref["T2_us"],
    ref["n"],
    ref["f_det_MHz"],  # overwritten each iteration
    ref["phi"],
    ref["offset"],
]
det_guess0_MHz = float(ref["f_det_MHz"])

# ============================================================
# 2) Sweep INSIDE temporarily_set_levels
# ============================================================
with temporarily_set_levels([qubox_logger, qm_logger], logging.WARNING):

    sc.do_set_output_status(True)
    time.sleep(settle_s)

    verbose_print = True
    print_on_fail = True

    USE_DET_CLAMP = True
    DET_CLAMP_HALFSPAN_MHz = 10.0

    USE_ROW_ANCHOR = True
    row_anchor_det_MHz = det_guess0_MHz

    det_eff_map_MHz = np.full((len(detunings_hz), len(powers_in_dbm)), np.nan, dtype=float)
    dq_map_MHz      = np.full((len(detunings_hz), len(powers_in_dbm)), np.nan, dtype=float)
    T2_map_us       = np.full((len(detunings_hz), len(powers_in_dbm)), np.nan, dtype=float)

    trace_store = {
        "meta": {
            "base_f_Hz": base_f,
            "delay_end": delay_end,
            "dt": dt,
            "n_avg": n_avg,
            "qb_detune_MHz": qb_detune_MHz,
            "powers_in_dbm": powers_in_dbm,
            "detunings_hz": detunings_hz,
            "settle_s": settle_s,
            "reference_fit": ref,
            "ref_guess_used_for_all_runs_except_det": list(ref_guess),
            "USE_DET_CLAMP": USE_DET_CLAMP,
            "DET_CLAMP_HALFSPAN_MHz": DET_CLAMP_HALFSPAN_MHz,
            "USE_ROW_ANCHOR": USE_ROW_ANCHOR,
        },
        "points": []
    }

    total_pts = len(detunings_hz) * len(powers_in_dbm)
    pbar = tqdm(total=total_pts, desc="Ramsey detuning sweep (ro detuning × power)", unit="pt")

    try:
        for i, ro_det in enumerate(detunings_hz):
            ro_f = base_f + float(ro_det)
            sc.do_set_frequency(ro_f)
            time.sleep(settle_s)

            det_guess_MHz = float(row_anchor_det_MHz) if USE_ROW_ANCHOR else float(det_guess0_MHz)
            last_good_det_guess_MHz = det_guess_MHz

            for j, power in enumerate(powers_in_dbm):
                sc.do_set_power(float(power))
                time.sleep(settle_s)

                fit_p0 = list(ref_guess)
                fit_p0[3] = float(det_guess_MHz)

                pbar.set_postfix(
                    ro_det_MHz=f"{ro_det/1e6:+.2f}",
                    P_dBm=f"{power:+.1f}",
                    det_guess=f"{fit_p0[3]:+.3f}",
                    refresh=False
                )

                if verbose_print:
                    pbar.write(
                        f"[SET] ro_det={ro_det/1e6:+.2f} MHz | ro_f={ro_f/1e6:.6f} MHz | "
                        f"P={power:+.2f} dBm | det_guess={fit_p0[3]:+.6f} MHz"
                    )

                rr = experiment.T2_ramsey(qb_detune, delay_end, dt, n_avg=n_avg)

                # Default values for trace store
                delays = delays_us = S = phases = None
                fit_tuple = None
                fit_ok = False

                if getattr(rr, "mode", None) == "simulate":
                    if verbose_print:
                        pbar.write(f"[SIM] ro_f={ro_f/1e6:.6f} MHz | P={power:+.2f} dBm | skipping fit")
                else:
                    out = rr.output
                    delays, S, phases = out.extract("delays", "S", "Phases")
                    delays_us = delays * 1e-3
                    y = np.real(S)

                    try:
                        T2_fit_params = fitting.generalized_fit(
                            delays_us,
                            y,
                            cQED_models.T2_ramsey_model,
                            fit_p0,
                            plotting=False,
                            param_format="{:.6f}",
                        )
                        A, T2_us, n, fitted_det_MHz, phi, offset = T2_fit_params[0]
                        fit_ok = True

                        det_eff_map_MHz[i, j] = float(fitted_det_MHz)
                        T2_map_us[i, j]       = float(T2_us)

                        # inferred qubit shift (your convention)
                        dq_map_MHz[i, j] = -(float(fitted_det_MHz) - qb_detune_MHz)

                        fit_tuple = (float(A), float(T2_us), float(n),
                                     float(fitted_det_MHz), float(phi), float(offset))

                        # Update ONLY det guess
                        det_guess_new = float(fitted_det_MHz)
                        if USE_DET_CLAMP:
                            det_guess_new = float(np.clip(
                                det_guess_new,
                                det_guess0_MHz - DET_CLAMP_HALFSPAN_MHz,
                                det_guess0_MHz + DET_CLAMP_HALFSPAN_MHz,
                            ))
                        det_guess_MHz = det_guess_new
                        last_good_det_guess_MHz = det_guess_MHz

                        if j == 0:
                            row_anchor_det_MHz = float(det_guess_MHz)

                        if verbose_print:
                            pbar.write(
                                f"[FIT] ro_f={ro_f/1e6:.6f} MHz | P={power:+.2f} dBm | "
                                f"T2*={float(T2_us):.3f} µs | det_fit={float(fitted_det_MHz):+.6f} MHz | "
                                f"dq={float(dq_map_MHz[i,j]):+.6f} MHz | next_det_guess={det_guess_MHz:+.6f} MHz"
                            )

                        pbar.set_postfix(
                            ro_det_MHz=f"{ro_det/1e6:+.2f}",
                            P_dBm=f"{power:+.1f}",
                            det_fit_MHz=f"{float(fitted_det_MHz):+.4f}",
                            T2_us=f"{float(T2_us):.2f}",
                            refresh=False
                        )

                    except Exception as e:
                        det_guess_MHz = float(last_good_det_guess_MHz)
                        if print_on_fail and verbose_print:
                            pbar.write(
                                f"[FIT-FAIL] ro_f={ro_f/1e6:.6f} MHz | P={power:+.2f} dBm | "
                                f"det_guess_kept={det_guess_MHz:+.6f} MHz | err={type(e).__name__}: {e}"
                            )

                trace_store["points"].append(
                    dict(
                        ro_detuning_hz=float(ro_det),
                        power_dbm=float(power),
                        ro_frequency_hz=float(ro_f),
                        mode=getattr(rr, "mode", "run"),
                        delays=np.array(delays, copy=True) if delays is not None else None,
                        delays_us=np.array(delays_us, copy=True) if delays_us is not None else None,
                        S=np.array(S, copy=True) if S is not None else None,
                        phases=np.array(phases, copy=True) if phases is not None else None,
                        fit=fit_tuple,
                        fit_ok=bool(fit_ok),
                        fit_p0_used=list(fit_p0),
                        det_guess_used_MHz=float(fit_p0[3]),
                    )
                )

                pbar.update(1)

    finally:
        pbar.close()

    # ----------------------------
    # Save traces
    # ----------------------------
    out_path = "data/seq_1_device/ramsey_detunings_traces_ro/RamseyDetuning_traces_rodetuning_power.npz"
    np.savez_compressed(out_path, trace_store=trace_store)
    print(f"\nSaved raw traces to: {out_path}")
    print(f"Reload with: d=np.load('{out_path}', allow_pickle=True); trace_store=d['trace_store'].item()")

# ============================================================
# 3) PLOTS (single plot; curves for each detuning; x = linear power)
# ============================================================

# Convert dBm -> linear power for plotting
P_mW = 10 ** (np.asarray(powers_in_dbm, dtype=float) / 10.0)   # mW
# If you prefer Watts:
# P_W  = P_mW * 1e-3

# Choose what to plot:
#   det_eff_map_MHz : fitted Ramsey detuning (MHz)
#   dq_map_MHz      : inferred Stark shift (MHz)
Y = det_eff_map_MHz
y_label = "Fitted Ramsey detuning (MHz)"
title = "Fitted Ramsey detuning vs readout power (linear) — one curve per readout detuning"

# For inferred Stark shift instead, uncomment:
# Y = dq_map_MHz
# y_label = r"Inferred $\Delta f_q$ (MHz)"
# title = r"Inferred qubit Stark shift $\Delta f_q$ vs readout power (linear) — one curve per readout detuning"

fig, ax = plt.subplots(figsize=(11, 6))

for i, ro_det in enumerate(detunings_hz):
    y = np.asarray(Y[i, :], dtype=float)
    m = np.isfinite(P_mW) & np.isfinite(y)
    if not np.any(m):
        continue

    ax.plot(
        P_mW[m],
        y[m],
        marker="o",
        markersize=3,
        linewidth=1,
        label=f"{ro_det/1e6:+.1f} MHz",
    )

ax.set_xlabel("Readout power (mW)  [converted from dBm]")
ax.set_ylabel(y_label)
ax.set_title(title)

# Power spans decades when converted from dBm; log-x makes it readable.
# If you want strictly linear x-axis, comment this out:
ax.set_xscale("log")

ax.grid(True, which="both", alpha=0.3)
ax.legend(title="Readout-tone detuning", ncols=2, fontsize=9)
plt.show()


# ============================================================
# 4) Quick-look helper for saved traces
# ============================================================
def plot_saved_ramsey_trace(trace_store, ro_detuning_hz, power_dbm, tol_det_hz=1.0, tol_pow_db=1e-6):
    target = None
    for p in trace_store["points"]:
        if p.get("S", None) is None:
            continue
        if abs(p["ro_detuning_hz"] - ro_detuning_hz) <= tol_det_hz and abs(p["power_dbm"] - power_dbm) <= tol_pow_db:
            target = p
            break
    if target is None:
        raise ValueError("No matching trace found. Check tolerances / values.")

    x = target["delays_us"]
    y = np.real(target["S"])
    fit = target["fit"]

    plt.figure(figsize=(9, 5))
    plt.plot(x, y, "o-", markersize=3, linewidth=1, label="Re(S)")
    plt.xlabel(r"delay ($\mu s$)")
    plt.ylabel("Re(S)")
    plt.title(f"Ramsey: ro_det={target['ro_detuning_hz']/1e6:+.2f} MHz, P={target['power_dbm']:.2f} dBm")
    plt.grid(True, alpha=0.3)

    if fit is not None:
        A, T2_us, n, det_MHz, phi, offset = fit
        try:
            t = np.linspace(np.min(x), np.max(x), 600)
            yfit = cQED_models.T2_ramsey_model(t, A, T2_us, n, det_MHz, phi, offset)
            plt.plot(t, yfit, "-", linewidth=2, label=f"fit det={det_MHz:+.4f} MHz, T2*={T2_us:.2f} µs")
        except Exception:
            pass

    plt.legend()
    plt.show()


In [ ]:
frequency_correction = (qb_detune - T2_fit_params[0][3]*1e6)
experiment.attributes.qb_T2_ramsey = int(T2_fit_params[0][1] * u.us)
experiment.attributes.qb_fq += frequency_correction
experiment.save_attributes()

### T2/detunings as a function of second pump

In [ ]:
import logging, time
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from qubox.logging_config import temporarily_set_levels

# Assumes you already have these in your environment:
#   fitting, cQED_models, and units 'u' (e.g., from qubox import units as u)

qubox_logger = logging.getLogger("qubox")
qm_logger    = logging.getLogger("qm")


def run_reference_ramsey(
    experiment,
    sc,
    qb_detune,
    delay_end,
    dt,
    n_avg_ref=20000,
    fit_p0=None,
    plot=True,
    plot_title="Reference T2 Ramsey (SignalCore OFF)",
):
    """
    Runs a reference Ramsey with SignalCore output OFF.
    Fits using: T2_ramsey_model(t, A, T2, n, f_det, phi, offset)
    Returns dict with fitted params + raw trace.
    """
    sc.do_set_output_status(False)

    rr = experiment.T2_ramsey(qb_detune, delay_end, dt, n_avg=n_avg_ref)
    if getattr(rr, "mode", None) == "simulate":
        raise RuntimeError("Reference Ramsey returned simulate mode; no data to fit.")

    out = rr.output
    delays, S, phases = out.extract("delays", "S", "Phases")

    delays_us = delays * 1e-3   # ns -> µs
    y = np.real(S)

    if fit_p0 is None:
        qb_detune_MHz = float(qb_detune * 1e-6)
        fit_p0 = [0, 25, 1, qb_detune_MHz, 0, 0]  # [A, T2_us, n, det_MHz, phi, offset]

    T2_fit_params = fitting.generalized_fit(
        delays_us,
        y,
        cQED_models.T2_ramsey_model,
        fit_p0,
        plotting=bool(plot),
        plot_options={
            "figsize": (12, 6),
            "xlabel": r"delays ($\mu s$)",
            "ylabel": "amplitudes",
            "title": plot_title,
            "legend_fontsize": 16,
        } if plot else None,
        param_format="{:.6f}",
    )
    A, T2_us, n, fitted_det_MHz, phi, offset = T2_fit_params[0]

    print(f"[REF] Fitted detuning: {float(fitted_det_MHz):+.6f} MHz")
    print(f"[REF] Fitted T2*:      {float(T2_us):.3f} µs")
    print(f"[REF] Fitted n:        {float(n):.4f}")
    print(f"[REF] Fitted phi:      {float(phi):+.4f}")
    print(f"[REF] Fitted offset:   {float(offset):+.6f}")
    print(f"[REF] Fitted A:        {float(A):+.6f}")

    return dict(
        A=float(A),
        T2_us=float(T2_us),
        n=float(n),
        f_det_MHz=float(fitted_det_MHz),
        phi=float(phi),
        offset=float(offset),
        delays=np.array(delays, copy=True),
        delays_us=np.array(delays_us, copy=True),
        S=np.array(S, copy=True),
        phases=np.array(phases, copy=True) if phases is not None else None,
        n_avg_ref=int(n_avg_ref),
        fit_p0_used=list(fit_p0),
    )


# ============================================================
# 0) Setup
# ============================================================
sc = experiment.device_manager.get("signalcore_ro", connect=False)

base_f = float(experiment.attributes.ro_fq)   # Hz

# Ramsey settings
delay_end = 20 * u.us
dt        = 48 * u.ns
n_avg     = 1000

qb_detune = 0.2 * u.MHz
qb_detune_MHz = float(qb_detune * 1e-6)

# Sweep knobs (SC still set in dBm, but we will PLOT in linear power)
powers_in_dbm = np.linspace(-20, 5, 21)
detunings_hz  = np.linspace(-100e6, +100e6, 11)

settle_s = 0.02

# ============================================================
# 1) Reference Ramsey OUTSIDE temporarily_set_levels
# ============================================================
sc.do_set_frequency(base_f)
sc.do_set_power(float(powers_in_dbm[0]))
time.sleep(settle_s)

ref_fit_p0 = [0, 25, 1, qb_detune_MHz, 0, 0]
ref = run_reference_ramsey(
    experiment=experiment,
    sc=sc,
    qb_detune=qb_detune,
    delay_end=delay_end,
    dt=dt,
    n_avg_ref=5000,
    fit_p0=ref_fit_p0,
    plot=True,
    plot_title="Reference T2 Ramsey (SC OFF, high avg)",
)

# "Global" guess vector from reference (only det guess will be overwritten each iteration)
ref_guess = [
    ref["A"],
    ref["T2_us"],
    ref["n"],
    ref["f_det_MHz"],  # overwritten each iteration
    ref["phi"],
    ref["offset"],
]
det_guess0_MHz = float(ref["f_det_MHz"])

# ============================================================
# 2) Sweep INSIDE temporarily_set_levels
# ============================================================
with temporarily_set_levels([qubox_logger, qm_logger], logging.WARNING):

    sc.do_set_output_status(True)
    time.sleep(settle_s)

    verbose_print = True
    print_on_fail = True

    USE_DET_CLAMP = True
    DET_CLAMP_HALFSPAN_MHz = 10.0

    USE_ROW_ANCHOR = True
    row_anchor_det_MHz = det_guess0_MHz

    det_eff_map_MHz = np.full((len(detunings_hz), len(powers_in_dbm)), np.nan, dtype=float)
    dq_map_MHz      = np.full((len(detunings_hz), len(powers_in_dbm)), np.nan, dtype=float)
    T2_map_us       = np.full((len(detunings_hz), len(powers_in_dbm)), np.nan, dtype=float)

    trace_store = {
        "meta": {
            "base_f_Hz": base_f,
            "delay_end": delay_end,
            "dt": dt,
            "n_avg": n_avg,
            "qb_detune_MHz": qb_detune_MHz,
            "powers_in_dbm": powers_in_dbm,
            "detunings_hz": detunings_hz,
            "settle_s": settle_s,
            "reference_fit": ref,
            "ref_guess_used_for_all_runs_except_det": list(ref_guess),
            "USE_DET_CLAMP": USE_DET_CLAMP,
            "DET_CLAMP_HALFSPAN_MHz": DET_CLAMP_HALFSPAN_MHz,
            "USE_ROW_ANCHOR": USE_ROW_ANCHOR,
        },
        "points": []
    }

    total_pts = len(detunings_hz) * len(powers_in_dbm)
    pbar = tqdm(total=total_pts, desc="Ramsey detuning sweep (ro detuning × power)", unit="pt")

    try:
        for i, ro_det in enumerate(detunings_hz):
            ro_f = base_f + float(ro_det)
            sc.do_set_frequency(ro_f)
            time.sleep(settle_s)

            det_guess_MHz = float(row_anchor_det_MHz) if USE_ROW_ANCHOR else float(det_guess0_MHz)
            last_good_det_guess_MHz = det_guess_MHz

            for j, power in enumerate(powers_in_dbm):
                sc.do_set_power(float(power))
                time.sleep(settle_s)

                fit_p0 = list(ref_guess)
                fit_p0[3] = float(det_guess_MHz)

                pbar.set_postfix(
                    ro_det_MHz=f"{ro_det/1e6:+.2f}",
                    P_dBm=f"{power:+.1f}",
                    det_guess=f"{fit_p0[3]:+.3f}",
                    refresh=False
                )

                if verbose_print:
                    pbar.write(
                        f"[SET] ro_det={ro_det/1e6:+.2f} MHz | ro_f={ro_f/1e6:.6f} MHz | "
                        f"P={power:+.2f} dBm | det_guess={fit_p0[3]:+.6f} MHz"
                    )

                rr = experiment.T2_ramsey(qb_detune, delay_end, dt, n_avg=n_avg)

                # Default values for trace store
                delays = delays_us = S = phases = None
                fit_tuple = None
                fit_ok = False

                if getattr(rr, "mode", None) == "simulate":
                    if verbose_print:
                        pbar.write(f"[SIM] ro_f={ro_f/1e6:.6f} MHz | P={power:+.2f} dBm | skipping fit")
                else:
                    out = rr.output
                    delays, S, phases = out.extract("delays", "S", "Phases")
                    delays_us = delays * 1e-3
                    y = np.real(S)

                    try:
                        T2_fit_params = fitting.generalized_fit(
                            delays_us,
                            y,
                            cQED_models.T2_ramsey_model,
                            fit_p0,
                            plotting=False,
                            param_format="{:.6f}",
                        )
                        A, T2_us, n, fitted_det_MHz, phi, offset = T2_fit_params[0]
                        fit_ok = True

                        det_eff_map_MHz[i, j] = float(fitted_det_MHz)
                        T2_map_us[i, j]       = float(T2_us)

                        # inferred qubit shift (your convention)
                        dq_map_MHz[i, j] = -(float(fitted_det_MHz) - qb_detune_MHz)

                        fit_tuple = (float(A), float(T2_us), float(n),
                                     float(fitted_det_MHz), float(phi), float(offset))

                        # Update ONLY det guess
                        det_guess_new = float(fitted_det_MHz)
                        if USE_DET_CLAMP:
                            det_guess_new = float(np.clip(
                                det_guess_new,
                                det_guess0_MHz - DET_CLAMP_HALFSPAN_MHz,
                                det_guess0_MHz + DET_CLAMP_HALFSPAN_MHz,
                            ))
                        det_guess_MHz = det_guess_new
                        last_good_det_guess_MHz = det_guess_MHz

                        if j == 0:
                            row_anchor_det_MHz = float(det_guess_MHz)

                        if verbose_print:
                            pbar.write(
                                f"[FIT] ro_f={ro_f/1e6:.6f} MHz | P={power:+.2f} dBm | "
                                f"T2*={float(T2_us):.3f} µs | det_fit={float(fitted_det_MHz):+.6f} MHz | "
                                f"dq={float(dq_map_MHz[i,j]):+.6f} MHz | next_det_guess={det_guess_MHz:+.6f} MHz"
                            )

                        pbar.set_postfix(
                            ro_det_MHz=f"{ro_det/1e6:+.2f}",
                            P_dBm=f"{power:+.1f}",
                            det_fit_MHz=f"{float(fitted_det_MHz):+.4f}",
                            T2_us=f"{float(T2_us):.2f}",
                            refresh=False
                        )

                    except Exception as e:
                        det_guess_MHz = float(last_good_det_guess_MHz)
                        if print_on_fail and verbose_print:
                            pbar.write(
                                f"[FIT-FAIL] ro_f={ro_f/1e6:.6f} MHz | P={power:+.2f} dBm | "
                                f"det_guess_kept={det_guess_MHz:+.6f} MHz | err={type(e).__name__}: {e}"
                            )

                trace_store["points"].append(
                    dict(
                        ro_detuning_hz=float(ro_det),
                        power_dbm=float(power),
                        ro_frequency_hz=float(ro_f),
                        mode=getattr(rr, "mode", "run"),
                        delays=np.array(delays, copy=True) if delays is not None else None,
                        delays_us=np.array(delays_us, copy=True) if delays_us is not None else None,
                        S=np.array(S, copy=True) if S is not None else None,
                        phases=np.array(phases, copy=True) if phases is not None else None,
                        fit=fit_tuple,
                        fit_ok=bool(fit_ok),
                        fit_p0_used=list(fit_p0),
                        det_guess_used_MHz=float(fit_p0[3]),
                    )
                )

                pbar.update(1)

    finally:
        pbar.close()

    # ----------------------------
    # Save traces
    # ----------------------------
    out_path = "data/seq_1_device/ramsey_detunings_traces_ro/RamseyDetuning_traces_rodetuning_power.npz"
    np.savez_compressed(out_path, trace_store=trace_store)
    print(f"\nSaved raw traces to: {out_path}")
    print(f"Reload with: d=np.load('{out_path}', allow_pickle=True); trace_store=d['trace_store'].item()")

# ============================================================
# 3) PLOTS (single plot; curves for each detuning; x = linear power)
# ============================================================

# Convert dBm -> linear power for plotting
P_mW = 10 ** (np.asarray(powers_in_dbm, dtype=float) / 10.0)   # mW
# If you prefer Watts:
# P_W  = P_mW * 1e-3

# Choose what to plot:
#   det_eff_map_MHz : fitted Ramsey detuning (MHz)
#   dq_map_MHz      : inferred Stark shift (MHz)
Y = det_eff_map_MHz
y_label = "Fitted Ramsey detuning (MHz)"
title = "Fitted Ramsey detuning vs readout power (linear) — one curve per readout detuning"

# For inferred Stark shift instead, uncomment:
# Y = dq_map_MHz
# y_label = r"Inferred $\Delta f_q$ (MHz)"
# title = r"Inferred qubit Stark shift $\Delta f_q$ vs readout power (linear) — one curve per readout detuning"

fig, ax = plt.subplots(figsize=(11, 6))

for i, ro_det in enumerate(detunings_hz):
    y = np.asarray(Y[i, :], dtype=float)
    m = np.isfinite(P_mW) & np.isfinite(y)
    if not np.any(m):
        continue

    ax.plot(
        P_mW[m],
        y[m],
        marker="o",
        markersize=3,
        linewidth=1,
        label=f"{ro_det/1e6:+.1f} MHz",
    )

ax.set_xlabel("Readout power (mW)  [converted from dBm]")
ax.set_ylabel(y_label)
ax.set_title(title)

# Power spans decades when converted from dBm; log-x makes it readable.
# If you want strictly linear x-axis, comment this out:
ax.set_xscale("log")

ax.grid(True, which="both", alpha=0.3)
ax.legend(title="Readout-tone detuning", ncols=2, fontsize=9)
plt.show()


# ============================================================
# 4) Quick-look helper for saved traces
# ============================================================
def plot_saved_ramsey_trace(trace_store, ro_detuning_hz, power_dbm, tol_det_hz=1.0, tol_pow_db=1e-6):
    target = None
    for p in trace_store["points"]:
        if p.get("S", None) is None:
            continue
        if abs(p["ro_detuning_hz"] - ro_detuning_hz) <= tol_det_hz and abs(p["power_dbm"] - power_dbm) <= tol_pow_db:
            target = p
            break
    if target is None:
        raise ValueError("No matching trace found. Check tolerances / values.")

    x = target["delays_us"]
    y = np.real(target["S"])
    fit = target["fit"]

    plt.figure(figsize=(9, 5))
    plt.plot(x, y, "o-", markersize=3, linewidth=1, label="Re(S)")
    plt.xlabel(r"delay ($\mu s$)")
    plt.ylabel("Re(S)")
    plt.title(f"Ramsey: ro_det={target['ro_detuning_hz']/1e6:+.2f} MHz, P={target['power_dbm']:.2f} dBm")
    plt.grid(True, alpha=0.3)

    if fit is not None:
        A, T2_us, n, det_MHz, phi, offset = fit
        try:
            t = np.linspace(np.min(x), np.max(x), 600)
            yfit = cQED_models.T2_ramsey_model(t, A, T2_us, n, det_MHz, phi, offset)
            plt.plot(t, yfit, "-", linewidth=2, label=f"fit det={det_MHz:+.4f} MHz, T2*={T2_us:.2f} µs")
        except Exception:
            pass

    plt.legend()
    plt.show()


### T1 measurements from detunings

In [ ]:
import logging, time
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from qubox.logging_config import temporarily_set_levels

# Assumes you already have these in your environment:
#   fitting, cQED_models, and units 'u' (e.g., from qubox import units as u)

qubox_logger = logging.getLogger("qubox")
qm_logger    = logging.getLogger("qm")


# ============================================================
# Helper: fit T1 from a single T1_relaxation run result
# ============================================================
def fit_T1_from_rr(rr, fit_p0=(0, 10, 0), do_plot=False, plot_title="T1 relaxation"):
    if getattr(rr, "mode", None) == "simulate":
        return dict(fit_ok=False, fit=None, delays=None, delays_us=None, S=None, phases=None)

    out = rr.output
    delays, S, phases = out.extract("delays", "S", "Phases")

    delays_us = delays * 1e-3  # ns -> µs (your convention)
    y = np.real(S)

    T1_fit_parms = fitting.generalized_fit(
        delays_us,
        y,
        cQED_models.T1_relaxation_model,
        list(fit_p0),
        plotting=bool(do_plot),
        plot_options={
            "figsize": (10, 6),
            "xlabel": r"delays ($\mu s$)",
            "ylabel": "amplitudes",
            "title": plot_title,
            "legend_fontsize": 16,
        } if do_plot else None,
        param_format="{:.4f}",
    )
    A, T1_us, offset = T1_fit_parms[0]
    return dict(
        fit_ok=True,
        fit=(float(A), float(T1_us), float(offset)),
        T1_us=float(T1_us),
        delays=np.array(delays, copy=True),
        delays_us=np.array(delays_us, copy=True),
        S=np.array(S, copy=True),
        phases=np.array(phases, copy=True) if phases is not None else None,
    )


# ============================================================
# 0) Reuse your existing scan grids
#    (IMPORTANT: keep these the SAME as Ramsey scan)
# ============================================================
# If these already exist in your notebook from the Ramsey cell, you can delete this block.
powers_in_dbm = np.linspace(-20, 0, 21)
detunings_hz  = np.linspace(-25e6, +250e6, 11)

sc = experiment.device_manager.get("signalcore_ro", connect=False)
base_f = float(experiment.attributes.ro_fq)   # Hz
settle_s = 0.02

# ============================================================
# 1) T1 settings
# ============================================================
t1_delay_end = 80 * u.us
t1_dt        = 1000
t1_n_avg     = 2000
t1_fit_p0    = (0, 10, 0)  # [A, T1_us, offset]

# ============================================================
# 2) Sweep INSIDE temporarily_set_levels (same grid as Ramsey)
# ============================================================
with temporarily_set_levels([qubox_logger, qm_logger], logging.WARNING):

    sc.do_set_output_status(True)
    time.sleep(settle_s)

    T1_map_us = np.full((len(detunings_hz), len(powers_in_dbm)), np.nan, dtype=float)

    t1_trace_store = {
        "meta": {
            "base_f_Hz": base_f,
            "powers_in_dbm": np.array(powers_in_dbm, copy=True),
            "detunings_hz":  np.array(detunings_hz, copy=True),
            "delay_end": t1_delay_end,
            "dt": t1_dt,
            "n_avg": t1_n_avg,
            "fit_p0": list(t1_fit_p0),
            "settle_s": settle_s,
        },
        "points": []
    }

    total_pts = len(detunings_hz) * len(powers_in_dbm)
    pbar = tqdm(total=total_pts, desc="T1 sweep (ro detuning × power)", unit="pt")

    try:
        for i, ro_det in enumerate(detunings_hz):
            ro_f = base_f + float(ro_det)
            sc.do_set_frequency(ro_f)
            time.sleep(settle_s)

            for j, power in enumerate(powers_in_dbm):
                sc.do_set_power(float(power))
                time.sleep(settle_s)

                pbar.set_postfix(
                    ro_det_MHz=f"{ro_det/1e6:+.2f}",
                    P_dBm=f"{power:+.1f}",
                    refresh=False
                )

                rr = experiment.T1_relaxation(t1_delay_end, t1_dt, n_avg=t1_n_avg)

                # Fit + save trace
                if getattr(rr, "mode", None) == "simulate":
                    t1_trace_store["points"].append(
                        dict(
                            ro_detuning_hz=float(ro_det),
                            power_dbm=float(power),
                            ro_frequency_hz=float(ro_f),
                            mode="simulate",
                            delays=None, delays_us=None, S=None, phases=None,
                            fit=None, fit_ok=False,
                        )
                    )
                    pbar.update(1)
                    continue

                res = fit_T1_from_rr(
                    rr,
                    fit_p0=t1_fit_p0,
                    do_plot=False,  # keep False for sweeps
                    plot_title=f"T1 relaxation | ro_det={ro_det/1e6:+.2f} MHz | P={power:+.2f} dBm",
                )

                if res["fit_ok"]:
                    A, T1_us, offset = res["fit"]
                    T1_map_us[i, j] = float(T1_us)
                    pbar.set_postfix(
                        ro_det_MHz=f"{ro_det/1e6:+.2f}",
                        P_dBm=f"{power:+.1f}",
                        T1_us=f"{T1_us:.2f}",
                        refresh=False
                    )

                t1_trace_store["points"].append(
                    dict(
                        ro_detuning_hz=float(ro_det),
                        power_dbm=float(power),
                        ro_frequency_hz=float(ro_f),
                        mode=getattr(rr, "mode", "run"),
                        delays=res["delays"],
                        delays_us=res["delays_us"],
                        S=res["S"],
                        phases=res["phases"],
                        fit=res["fit"],
                        fit_ok=bool(res["fit_ok"]),
                    )
                )

                pbar.update(1)

    finally:
        pbar.close()

# ============================================================
# 3) Save T1 traces
# ============================================================
t1_out_path = "data/seq_1_device/t1_traces_ro/T1_traces_rodetuning_power.npz"
np.savez_compressed(t1_out_path, trace_store=t1_trace_store)



In [ ]:
print(f"\nSaved T1 raw traces to: {t1_out_path}")
print(f"Reload with: d=np.load('{t1_out_path}', allow_pickle=True); t1_trace_store=d['trace_store'].item()")

# ============================================================
# 4) Plot: single plot, one curve per readout detuning, x = linear power
# ============================================================
P_mW = 10 ** (np.asarray(powers_in_dbm, dtype=float) / 10.0)   # mW

fig, ax = plt.subplots(figsize=(11, 6))
for i, ro_det in enumerate(detunings_hz):
    y = np.asarray(T1_map_us[i, :], dtype=float)
    m = np.isfinite(P_mW) & np.isfinite(y)
    if not np.any(m):
        continue
    ax.plot(
        P_mW[m], y[m],
        marker="o", markersize=3, linewidth=1,
        label=f"{ro_det/1e6:+.1f} MHz",
    )

ax.set_xlabel("Readout power (mW)  [converted from dBm]")
ax.set_ylabel("Fitted T1 (µs)")
ax.set_title("T1 vs readout power (linear) — one curve per readout detuning")
ax.set_xscale("log")  # comment out if you truly want linear axis
ax.grid(True, which="both", alpha=0.3)
ax.legend(title="Readout-tone detuning", ncols=2, fontsize=9)
plt.show()


# ============================================================
# 5) Optional helper: quick-look a saved T1 trace later
# ============================================================
def plot_saved_T1_trace(t1_trace_store, ro_detuning_hz, power_dbm, tol_det_hz=1.0, tol_pow_db=1e-6):
    target = None
    for p in t1_trace_store["points"]:
        if p.get("S", None) is None:
            continue
        if abs(p["ro_detuning_hz"] - ro_detuning_hz) <= tol_det_hz and abs(p["power_dbm"] - power_dbm) <= tol_pow_db:
            target = p
            break
    if target is None:
        raise ValueError("No matching trace found. Check tolerances / values.")

    x = target["delays_us"]
    y = np.real(target["S"])
    fit = target["fit"]

    plt.figure(figsize=(9, 5))
    plt.plot(x, y, "o-", markersize=3, linewidth=1, label="Re(S)")
    plt.xlabel(r"delay ($\mu s$)")
    plt.ylabel("Re(S)")
    plt.title(f"T1: ro_det={target['ro_detuning_hz']/1e6:+.2f} MHz, P={target['power_dbm']:.2f} dBm")
    plt.grid(True, alpha=0.3)

    if fit is not None:
        A, T1_us, offset = fit
        try:
            t = np.linspace(np.min(x), np.max(x), 600)
            yfit = cQED_models.T1_relaxation_model(t, A, T1_us, offset)
            plt.plot(t, yfit, "-", linewidth=2, label=f"fit T1={T1_us:.2f} µs")
        except Exception:
            pass

    plt.legend()
    plt.show()


# ============================================================
# 6) (Optional) update attributes with ONE chosen T1 (e.g., max across sweep)
# ============================================================
# If you *really* want to update attributes, pick a single representative T1.
# Example: use the max finite T1 from the sweep:
finite = np.isfinite(T1_map_us)
if np.any(finite):
    T1_best_us = float(np.nanmax(T1_map_us))
    experiment.attributes.qb_T1_relax = T1_best_us * 1e3  # µs -> ns
    experiment.attributes.qb_therm_clks = int(experiment.attributes.qb_T1_relax)
    experiment.save_attributes()
    print(f"\nUpdated attributes using max T1 from sweep: {T1_best_us:.3f} µs")
else:
    print("\nNo valid T1 fits; attributes not updated.")


In [ ]:
detunings_hz

### T2 echo

In [ ]:
# ---------- T2 Echo ----------
delay_end = 30 * u.us
dt             = 0.5 * u.us

rr = experiment.T2_echo(delay_end, dt, n_avg=10000)

if rr.mode == "simulate":
    print("Simulation run — no stream data; skipping T2 Echo fit.")
else:
    out = rr.output
    delays, S, phases = out.extract("delays", "S", "Phases")

    fit_p0 = [-1, 40, 1, 0]
    T2_fit_params = fitting.generalized_fit(
        delays * 1e-3,   # ns -> µs
        S.real,
        cQED_models.T2_echo_model,
        fit_p0,
        plotting=True,
        plot_options={"figsize": (12, 6), r"xlabel": "delays ($\mu s$)", "ylabel": "amplitudes",
                      "title": "T2 Echo", "legend_fontsize": 16},
        param_format="{:.4f}",
    )
    experiment.attributes.qb_T2_echo = int(T2_fit_params[0][1] * u.us)
    experiment.save_attributes()

In [ ]:
experiment.attributes.qb_T2_echo = int(T2_fit_params[0][1] * u.us)
experiment.save_attributes()

## Advance Characterization

### Time Rabi Chevron

In [ ]:
# ---------- Time Rabi Chevron ----------
if_span = 12e6
df      = 0.2e6
max_pulse_duration = 1000  # ns
dt      = 4                # ns

rr = experiment.time_rabi_chevron(if_span, df, max_pulse_duration, dt,
                                  pulse_gain=1, n_avg=100)

if rr.mode == "simulate": 
    print("Simulation run — skipping Time Rabi Chevron plot.")
else:
    out = rr.output
    durations, detunings, amplitudes = out.extract("durations", "detunings", "amplitudes")
    plot_hm(amplitudes, detunings * 1e-6, durations,
            r"qubit detunings $\Delta$ (MHz)", "pulse duration (ns)", "Time Rabi Chevron")


### Power Rabi Chevron

In [ ]:
# ---------- Power Rabi Chevron ----------
if_span = 12e6
df      = 0.2e6
max_gain = 1.0
dg       = 0.01
# r_len defined elsewhere (ns)
r_len = 16
rr = experiment.power_rabi_chevron(if_span, df, max_gain, dg, pulse_duration=r_len, n_avg=100)

if rr.mode == "simulate":
    print("Simulation run — skipping Power Rabi Chevron plot.")
else:
    out = rr.output
    gains, detunings, amplitudes = out.extract("gains", "detunings", "amplitudes")
    plot_hm(amplitudes, detunings * 1e-6, gains,
            r"qubit detunings $\Delta$ (MHz)", "pulse gains", "Power Rabi Chevron")


### Ramsey Chevron

In [ ]:
# ---------- Power Rabi Chevron ----------
if_span = 12e6
df      = 0.2e6
max_gain = 1.0
dg       = 0.01
# r_len defined elsewhere (ns)

rr = experiment.power_rabi_chevron(if_span, df, max_gain, dg, pulse_duration=r_len, n_avg=100)

if rr.mode == "simulate":
    print("Simulation run — skipping Power Rabi Chevron plot.")
else:
    out = rr.output
    gains, detunings, amplitudes = out.extract("gains", "detunings", "amplitudes")
    plot_hm(amplitudes, detunings * 1e-6, gains,
            r"qubit detunings $\Delta$ (MHz)", "pulse gains", "Power Rabi Chevron")


### Resonator Spectroscopy x180

In [ ]:
# ---------- Resonator Spectroscopy with x180 (single run) ----------
n_avg      = 20_000
freq_start = 8592 * u.MHz
freq_end   = 8598.5 * u.MHz
df         = 50 * u.kHz

# Run the experiment (wrapper returns absolute probe freqs via proc_attach)
rr_g = experiment.resonator_spectroscopy_x180(freq_start, freq_end, df, r180="zero", n_avg=n_avg)
out_g = rr_g.output
frequencies, S_g = out_g.extract("frequencies", "S")

rr_e = experiment.resonator_spectroscopy_x180(freq_start, freq_end, df, n_avg=n_avg)
out_e = rr_e.output
frequencies, S_e = out_e.extract("frequencies", "S")

freqs_mhz = frequencies * 1e-6

def _guess_center(x_mhz, y_complex):
    mag = np.abs(y_complex)
    return float(x_mhz[np.argmax(mag)])

p0_g = [_guess_center(freqs_mhz, S_g), 1.0, -1e-4, 0.0]
custom_plot_opts = {"figsize": (10, 6), "xlabel": "frequency (MHz)", "ylabel": "R", "title": "Resonator Spectroscopy Fit", "legend_fontsize": 16}
fit_g = generalized_fit(freqs_mhz, np.abs(S_g), cQED_models.resonator_spec_model, p0_g, plotting=True, plot_options=custom_plot_opts, param_format="{:.4f}")

p0_e = [_guess_center(freqs_mhz, S_e), 1.0, -1e-4, 0.0]
custom_plot_opts = {"figsize": (10, 6), "xlabel": "frequency (MHz)", "ylabel": "R", "title": "Resonator Spectroscopy Fit", "legend_fontsize": 16}
fit_e = generalized_fit(freqs_mhz, np.abs(S_e), cQED_models.resonator_spec_model, p0_e, plotting=True, plot_options=custom_plot_opts, param_format="{:.4f}")

fit_g_f0 = fit_g[0][0]
fit_e_f0 = fit_e[0][0]
chi_mhz = (fit_g_f0 - fit_e_f0)
print(f"Estimated χ ≈ {chi_mhz:.6f} MHz (from single run)")

# Qubit Pulse Calibration and Benchmarking

### Drag calibration YALE

In [ ]:
from qubox.analysis.algorithms import find_roots
# --- sweep over the multiplier 'alpha' (coeff_eff = base_drag_amp * alpha) ---
amps = np.linspace(-0.5, 0.5, 20)
base_alpha  = 1
n_avg          = 50000

rr = experiment.drag_calibration_YALE(amps, base_alpha, n_avg)

if getattr(rr, "mode", None) == "simulate":
    print("Simulation run — no stream data; skipping DRAG-YALE analysis/plot.")
else:
    # Extract complex IQ for each of the two sequences + their state probs
    # Assumes dp.qubit_proc turned (I,Q) into complex streams "S_1" and "S_2"
    # and that "state1", "state2" are averaged excited probs.
    S_1, S_2 = rr.output.extract("S_1", "S_2")
    plt.figure(figsize=(10, 6))
    plt.plot(amps, S_1.real, 'o-', linewidth=2, markersize=4, label=r'$X_{180} - Y_{90}$ sequence')
    plt.plot(amps, S_2.real, 's-', linewidth=2, markersize=4, label=r'$Y_{180} - X_{90}$ sequence')
    plt.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5, label=r'$\langle\sigma_z\rangle = 0$')
    plt.xlabel("DRAG amplitude scaling factor (α)", fontsize=12)
    plt.ylabel(r"$\langle\sigma_z\rangle$", fontsize=12)
    plt.title(f"DRAG Calibration (Yale Method) - base α = {base_alpha}", fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)

    # Find and mark the roots
    roots = find_roots(amps, S_1.real - S_2.real)
    roots = np.array(roots)
    possible_alpha_candidates = roots * base_alpha

    # Mark the crossing points
    for root in roots:
        plt.axvline(x=root, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
        
    plt.tight_layout()
    plt.show()

    print("Possible alpha candidates (Yale method):", possible_alpha_candidates)
    print(f"Optimal DRAG coefficient: α = {possible_alpha_candidates[0]:.4f}" if len(possible_alpha_candidates) > 0 else "No roots found")

### Qubit pulse train r180 calibration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Use combined (signal+reference) qubit_pulse_train output
# rr = experiment.qubit_pulse_train(..., run_reference=True, ...)
# and extract N, Pe, Pe_ref, S, S_ref from the SAME run.
#
# Then fit ratio R = Pe/Pe_ref to get δ and (optionally) T2.
# ============================================================

# ----------------------------
# Config
# ----------------------------
K = 2
N_values = np.arange(0, 20, K)   # these are the N used by the program (number of rotation pulses played)
n_avg = 200000

reference_pulse = "x90"
rotation_pulse  = "sel_x180"

FIT_T2 = True                 # set False if you only want δ
T2_IN_MICROSECONDS = False    # only affects initial guess if using attr.T2_ramsey

EPS_REF = 1e-6

# Time model: x90 then N rotation pulses, each of length tau_ns
TIME_MODEL = "x90_plus_N_tau"

# ----------------------------
# Helpers
# ----------------------------
def _ensure_1d(x, dtype=None):
    x = np.asarray(x)
    if x.ndim != 1:
        x = np.ravel(x)
    if dtype is not None:
        x = x.astype(dtype, copy=False)
    return x

def _extract_params(fit_result):
    if isinstance(fit_result, dict):
        for k in ("fit_params", "params", "popt"):
            if k in fit_result:
                return np.asarray(fit_result[k], dtype=float)
    if isinstance(fit_result, (list, tuple)):
        first = fit_result[0]
        if np.ndim(first) == 1:
            return np.asarray(first, dtype=float)
    return np.asarray(fit_result, dtype=float)

def _scalar(x):
    return float(np.asarray(x).ravel()[0])


# ----------------------------
# 1) Run combined program (signal + reference inside same QUA job)
# ----------------------------
rr = experiment.qubit_pulse_train(
    N_values,
    reference_pulse=reference_pulse,
    rotation_pulse=rotation_pulse,
    run_reference=True,
    n_avg=n_avg,
)

if rr.mode == "simulate":
    raise RuntimeError("Program ran in simulate mode; no real data to analyze.")

out = rr.output

# ----------------------------
# 2) Extract directly (already aligned)
# ----------------------------
# NOTE: your example had a small typo: you used a comma, which makes a 1-tuple.
# Do this instead:
N_vals, Pe, Pe_ref, S, S_ref = out.extract("N_values", "Pe", "Pe_ref", "S", "S_ref")

N_vals = _ensure_1d(N_vals, int)
Pe     = _ensure_1d(Pe, float)
Pe_ref = _ensure_1d(Pe_ref, float)
S      = _ensure_1d(S, complex)
S_ref  = _ensure_1d(S_ref, complex)

# Optional sanity: if you don't trust Pe/Pe_ref coming from firmware/analysis,
# you can recompute from S/S_ref using your calibrated mapping:
# Pe     = _ensure_1d(measureMacro.compute_Pe_from_S(S), float)
# Pe_ref = _ensure_1d(measureMacro.compute_Pe_from_S(S_ref), float)

# ----------------------------
# 3) Sanity plots: Pe and Pe_ref vs N
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(N_vals, Pe, "o-", label="Signal Pe (x90 + N*x180)")
ax.plot(N_vals, Pe_ref, "s--", label="Reference Pe_ref (x90 + N*r0)")
ax.set_xlabel(r"$N$ (number of rotation pulses played)")
ax.set_ylabel(r"$P_e$")
ax.set_title(r"$P_e$ and $P_e^{\rm ref}$ vs $N$ (same run)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

# ----------------------------
# 4) Ratio data R = Pe / Pe_ref
# ----------------------------
mask = Pe_ref > EPS_REF
N_fit = N_vals[mask]
R_data = Pe[mask] / Pe_ref[mask]

# ----------------------------
# 5) Time axis for T2 envelope
# ----------------------------
# If your output contains a pulse length, use it. Otherwise set tau_ns manually.
try:
    tau_ns = _scalar(out.extract("rotation_time"))
except Exception:
    # fallback: set this to your known x180 length in ns
    tau_ns = 16.0
    print(f"[WARN] out.extract('rotation_time') failed; using tau_ns={tau_ns} ns manually.")

# assume x90 duration ~ tau_ns unless you have separate key
t_x90_ns = tau_ns

def total_time_s(N):
    N = np.asarray(N, dtype=float)
    # x90 then N rotation pulses
    t_ns = t_x90_ns + tau_ns * N
    return t_ns * 1e-9

# ----------------------------
# 6) Fit model: R(N) = B + A * exp(-t/T2) * sin(N*δ)
#    (ignore explicit T1; using ratio from same run reduces drift sensitivity)
# ----------------------------
attr = experiment.attributes
T2_guess = getattr(attr, "T2_ramsey", None)
if (T2_guess is None) or (float(T2_guess) <= 0):
    T2_guess_s = 10e-6
else:
    T2_guess_s = float(T2_guess) * (1e-6 if T2_IN_MICROSECONDS else 1.0)
if FIT_T2:
    def ratio_model(N, B, A, delta, logT2):
        N = np.asarray(N, dtype=float)
        t_s = total_time_s(N)
        T2_s = np.exp(logT2)
        env = np.exp(-t_s / T2_s)
        return B + A * env * np.sin(N * delta)

    fit_p0 = [1.0, 0.5, 0.02, np.log(T2_guess_s)]
else:
    def ratio_model(N, B, A, delta):
        N = np.asarray(N, dtype=float)
        t_s = total_time_s(N)
        # if not fitting T2, set envelope to 1
        return B + A * np.sin(N * delta)

    fit_p0 = [1.0, 0.5, 0.02]

fit_result = fitting.generalized_fit(
    N_fit,
    R_data,
    ratio_model,
    fit_p0,
    plotting=False,
    param_format="{:.4e}",
)
p = _extract_params(fit_result)

if FIT_T2:
    B_fit, A_fit, delta_fit, logT2_fit = p[:4]
    T2_fit_s = float(np.exp(logT2_fit))
else:
    B_fit, A_fit, delta_fit = p[:3]
    T2_fit_s = None

pi_amp_scale = np.pi / (np.pi + delta_fit)

print("\n=== Pulse-train π calibration (single-run signal+reference) ===")
print(f"tau_ns: {tau_ns:.3f}  (assumed x90={t_x90_ns:.3f} ns)")
print(f"Fit B: {B_fit:.4e}")
print(f"Fit A: {A_fit:.4e}")
print(f"Fit δ (rad): {delta_fit:.6e}")
if FIT_T2:
    print(f"Fit T2_eff: {T2_fit_s:.6e} s  ({T2_fit_s*1e6:.3f} µs)")
print(f"Suggested π amplitude scale factor: {pi_amp_scale:.8f}  (~1 - δ/π = {1 - delta_fit/np.pi:.8f})")

# ----------------------------
# 7) Plot ratio + fit
# ----------------------------
R_fit = ratio_model(N_fit, *([B_fit, A_fit, delta_fit, logT2_fit] if FIT_T2 else [B_fit, A_fit, delta_fit]))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(N_fit, R_data, "o", label=r"Data: $P_e/P_e^{\rm ref}$")
ax.plot(N_fit, R_fit, "-", label="Fit", linewidth=1.5)
ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel(r"$N$ (number of rotation pulses played)")
ax.set_ylabel(r"$P_e/P_e^{\rm ref}$")
ax.set_title("Normalized Pulse-Train Calibration (same run)")
ax.grid(True, alpha=0.3)
ax.legend()

# Top axis: total time (µs)
def N_to_t_us(N):
    return total_time_s(N) * 1e6
def t_us_to_N(t_us):
    t_s = np.asarray(t_us, dtype=float) * 1e-6
    return (t_s - t_x90_ns * 1e-9) / (tau_ns * 1e-9)

secax = ax.secondary_xaxis("top", functions=(N_to_t_us, t_us_to_N))
secax.set_xlabel("Total sequence time (µs)")
fig.tight_layout()
plt.show()

# ----------------------------
# 8) Optional: N=0 sanity check
# ----------------------------
if np.any(N_vals == 0):
    i0 = np.where(N_vals == 0)[0][0]
    r0 = Pe[i0] / Pe_ref[i0] if Pe_ref[i0] > 0 else np.nan
    print(f"N=0 sanity: Pe={Pe[i0]:.4f}, Pe_ref={Pe_ref[i0]:.4f}, ratio={r0:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Config
# ----------------------------
K = 2
N_pi = np.arange(0, 20, K)      # N_pi = number of rotation pulses played (even only)
n_avg = 200000

reference_pulse = "x90"
rotation_pulse  = "x180"
idle_pulse      = "r0"

# If your attr.T2_ramsey is stored in microseconds, set this True
T2_IN_MICROSECONDS = False

# Choose how you want to model total time:
# - If your program is literally: x90 then N_pi rotation pulses, use:
TIME_MODEL = "x90_plus_Npi_tau"   # (recommended for your described experiment)
# - If you truly want old "(2N+1)*tau" using N_pairs, set: TIME_MODEL = "twoNplus1_tau"


# ----------------------------
# Small helpers (cell-friendly)
# ----------------------------
def _ensure_1d(x, dtype=None):
    x = np.asarray(x)
    if x.ndim != 1:
        x = np.ravel(x)
    if dtype is not None:
        x = x.astype(dtype, copy=False)
    return x

def _align_ref_by_N(N_ref, S_ref, N_target):
    """
    Align reference S_ref onto the N grid of the signal.
    Raises KeyError if any N_target is missing in N_ref.
    """
    m = {int(n): s for n, s in zip(N_ref, S_ref)}
    return np.array([m[int(n)] for n in N_target], dtype=complex)

def _extract_params(fit_result):
    """
    Tries to pull parameter array from different generalized_fit return styles.
    """
    if isinstance(fit_result, dict):
        for k in ("fit_params", "params", "popt"):
            if k in fit_result:
                return np.asarray(fit_result[k], dtype=float)
    if isinstance(fit_result, (list, tuple)):
        first = fit_result[0]
        if np.ndim(first) == 1:
            return np.asarray(first, dtype=float)
    return np.asarray(fit_result, dtype=float)


# ----------------------------
# 1) Run experiment + reference
# ----------------------------
rr_ref = experiment.qubit_pulse_train(
    N_pi, reference_pulse=reference_pulse, rotation_pulse=idle_pulse, n_avg=n_avg
)
rr = experiment.qubit_pulse_train(
    N_pi, reference_pulse=reference_pulse, rotation_pulse=rotation_pulse, n_avg=n_avg
)

if (rr.mode != "simulate") and (rr_ref.mode != "simulate"):
    out, out_ref = rr.output, rr_ref.output

    # ----------------------------
    # 2) Extract + align
    # ----------------------------
    N_sig, S_sig = out.extract("N_values", "S")
    N_ref, S_ref = out_ref.extract("N_values", "S")

    N_sig = _ensure_1d(N_sig, int)
    N_ref = _ensure_1d(N_ref, int)
    S_sig = _ensure_1d(S_sig, complex)
    S_ref = _ensure_1d(S_ref, complex)

    # Sanity: these should match the N_pi you requested, but don’t assume ordering
    # (we align reference to signal by key)
    S_ref_aligned = _align_ref_by_N(N_ref, S_ref, N_sig)

    # Convert complex readout -> Pe
    Pe     = _ensure_1d(measureMacro.compute_Pe_from_S(S_sig), float)
    Pe_ref = _ensure_1d(measureMacro.compute_Pe_from_S(S_ref_aligned), float)

    # ----------------------------
    # 3) Basic sanity plot: Pe vs number of rotation pulses
    # ----------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(N_sig, Pe, "o-", label=r"With $\pi$ pulses")
    ax.plot(N_sig, Pe_ref, "s--", label=r"Reference (idle)")
    ax.set_xlabel(r"$N_\pi$ (number of rotation pulses played)")
    ax.set_ylabel(r"$P_e$")
    ax.set_title(r"$P_e$ vs $N_\pi$")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    # ----------------------------
    # 4) Time axis + T2 envelope
    # ----------------------------
    # Your code used out_ref.extract("rotation_time"). Keep it, but ensure it's a scalar.
    tau_ns = out_ref.extract("rotation_time")
    tau_ns = float(np.asarray(tau_ns).ravel()[0])

    # Optional: define N_pairs if you like the "pairs" x-axis (since you're using even N_pi)
    # N_pairs = N_pi/2; and sin(2*N_pairs*delta) == sin(N_pi*delta)
    N_pairs_sig = (N_sig // 2).astype(int)

    attr = experiment.attributes
    T2_eff = getattr(attr, "T2_ramsey", None)  # may be None
    if (T2_eff is not None) and T2_IN_MICROSECONDS:
        T2_eff = float(T2_eff) * 1e-6
    elif T2_eff is not None:
        T2_eff = float(T2_eff)

    def E2(t_s):
        if (T2_eff is None) or (T2_eff <= 0):
            return np.ones_like(t_s, dtype=float)
        return np.exp(-t_s / T2_eff)

    # Total-time model (only affects the exp(-t/T2) envelope and top axis)
    if TIME_MODEL == "x90_plus_Npi_tau":
        # experiment: x90 then N_pi rotation pulses; assumes x90 duration ~ tau_ns
        t_ns = tau_ns * (1.0 + N_sig.astype(float))
    elif TIME_MODEL == "twoNplus1_tau":
        # legacy form: define N_pairs and use (2*N_pairs + 1)*tau
        t_ns = tau_ns * (2.0 * N_pairs_sig.astype(float) + 1.0)
    else:
        raise ValueError(f"Unknown TIME_MODEL: {TIME_MODEL}")

    t_s = t_ns * 1e-9

    # ----------------------------
    # 5) Ratio to suppress T1 + fit δ
    # ----------------------------
    eps = 1e-6
    mask = Pe_ref > eps

    N_sig_fit = N_sig[mask]
    t_s_fit   = t_s[mask]
    R_data    = (Pe[mask] / Pe_ref[mask])

    # Model choice (clear + robust):
    # Use N_sig_fit directly as "number of π pulses", so the sinusoid is sin(N_pi * δ).
    # Add a baseline B so the fit doesn't have to force R→1 exactly.
    def ratio_model(N_pi_vals, B, A, delta):
        N_pi_vals = np.asarray(N_pi_vals, dtype=float)
        # recompute time consistently inside model (no indexing tricks)
        if TIME_MODEL == "x90_plus_Npi_tau":
            t_s_loc = (tau_ns * (1.0 + N_pi_vals)) * 1e-9
        else:
            # if using pairs model, map N_pi -> N_pairs -> time
            N_pairs = N_pi_vals / 2.0
            t_s_loc = (tau_ns * (2.0 * N_pairs + 1.0)) * 1e-9

        return B + A * E2(t_s_loc) * np.sin(N_pi_vals * delta)

 

In [ ]:
fit_p0 = [1.9786e-02, 1, 5.6453e-01]  # B~1, A moderate, delta small (rad)

fit_result = fitting.generalized_fit(
    N_sig_fit,
    R_data,
    ratio_model,
    fit_p0,
    plotting=False,
    param_format="{:.4e}",
)
B_fit, A_fit, delta_fit = _extract_params(fit_result)[:3]

# π amplitude correction: scale amplitude so (π+δ)*s = π
pi_amp_scale = np.pi / (np.pi + delta_fit)

print("=== Pulse-train π calibration ===")
print(f"T2_eff (s): {T2_eff}")
print(f"Fit B: {B_fit:.4e}")
print(f"Fit A: {A_fit:.4e}")
print(f"Fit δ (rad): {delta_fit:.4e}")
print(f"Suggested π amplitude scale factor: {pi_amp_scale:.8f}")
print(f"(~1 - δ/π = {1 - delta_fit/np.pi:.8f})")

# ----------------------------
# 6) Plot ratio + fit
# ----------------------------
R_fit = ratio_model(N_sig_fit, B_fit, A_fit, delta_fit)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(N_sig_fit, R_data, "o", label=r"Data: $P_e/P_e^{\rm ref}$")
ax.plot(N_sig_fit, R_fit, "-", label="Fit", linewidth=1.5)
ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.5)

ax.set_xlabel(r"$N_\pi$ (number of rotation pulses played)")
ax.set_ylabel(r"$P_e/P_e^{\rm ref}$")
ax.set_title(r"Normalized Pulse-Train Calibration")
ax.grid(True, alpha=0.3)
ax.legend()

# Secondary top axis: time in µs (affine invert assumed)
def Npi_to_t_us(Npi):
    Npi = np.asarray(Npi, dtype=float)
    if TIME_MODEL == "x90_plus_Npi_tau":
        return (tau_ns * (1.0 + Npi)) * 1e-3
    else:
        Npairs = Npi / 2.0
        return (tau_ns * (2.0 * Npairs + 1.0)) * 1e-3

def t_us_to_Npi(t_us):
    t_ns_ = np.asarray(t_us, dtype=float) * 1e3
    # invert affine time model
    if TIME_MODEL == "x90_plus_Npi_tau":
        return (t_ns_ / tau_ns) - 1.0
    else:
        # t_ns = tau*(Npi + 1) in this simplified mapping too, so same inverse
        return (t_ns_ / tau_ns) - 1.0

secax = ax.secondary_xaxis("top", functions=(Npi_to_t_us, t_us_to_Npi))
secax.set_xlabel("Total sequence time (µs)")

fig.tight_layout()
plt.show()


### Qubit pulse train r90 calibration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qm.qua import amp
# ------------------------ Config ------------------------
N_values   = np.arange(0, 10, 1)   # N: we will have 4N x90 pulses
n_avg      = 100000
rot_len_ns = 16.0                  # length of each rotation pulse
K          = 4                     # <-- for x90 calibration: 4N pulses

# Pulses
r90_pulse  = "x90"*amp(1.0225312929852983)    # nominal π/2 pulse we want to calibrate
r_idle     = "r0"     # idle / null pulse for reference

# ------------------ Run QUA experiments -----------------
# Reference: same timing, but x90 pulses replaced by idle "r0"
rr_ref = experiment.qubit_pulse_train(
    N_values,
    n_avg=n_avg,
    K=K,
    r90_pulse=r90_pulse,   # used for the initial π/2 prep
    rotation_pulse=r_idle  # no rotations in the train
)

# Actual run: with x90 pulses in the train (4N of them)
rr = experiment.qubit_pulse_train(
    N_values,
    n_avg=n_avg,
    K=K,
    r90_pulse=r90_pulse,   # initial π/2 prep
    rotation_pulse=r90_pulse   # the repeated rotations you want to calibrate
)

if (rr.mode != "simulate") and (rr_ref.mode != "simulate"):
    out     = rr.output
    out_ref = rr_ref.output

    # Extract
    N, S             = out.extract("N_values", "S")
    N_ref, S_ref, _  = out_ref.extract("N_values", "S", "ro_len")

    N     = np.asarray(N, dtype=int)
    N_ref = np.asarray(N_ref, dtype=int)
    S     = np.asarray(S, dtype=complex)
    S_ref = np.asarray(S_ref, dtype=complex)

    # Align reference by N
    ref_map = {int(n): s for n, s in zip(N_ref, S_ref)}
    S_ref_aligned = np.array([ref_map[int(n)] for n in N])

    # sigma_z = -1 (g), +1 (e)
    Pe     = (1.0 + S.real) / 2.0
    Pe_ref = (1.0 + S_ref_aligned.real) / 2.0

    # --- Basic Pe vs N plot (for sanity) ---
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(N, Pe, "o-", label=r"With $x_{90}$ pulses")
    ax.plot(N, Pe_ref, "s--", label=r"Reference (idle)")
    ax.set_xlabel(r"$N$ (blocks; $4N$ $x_{90}$ pulses)")
    ax.set_ylabel(r"$P_e$")
    ax.set_title(r"$P_e$ vs $N$ (x90 pulse train)")
    ax.grid(True, alpha=0.3)
    ax.set_xticks(N)
    ax.legend()

    # total time: t(N) = (K N + 1) * τ, with τ = rot_len_ns
    def N_to_t_us(N_val):
        return rot_len_ns * (K * np.asarray(N_val) + 1) * 1e-3  # µs

    def t_us_to_N(t_us):
        t_ns_ = np.asarray(t_us) * 1e3
        return (t_ns_ / rot_len_ns - 1.0) / K

    secax = ax.secondary_xaxis("top", functions=(N_to_t_us, t_us_to_N))
    secax.set_xlabel("Total sequence time (µs)")
    fig.tight_layout()
    plt.show()

    # ---------- Normalize to cancel T1: R(N) = Pe / Pe_ref ----------
    eps = 1e-6
    mask = Pe_ref > eps

    N_fit       = N[mask]
    Pe_ref_fit  = Pe_ref[mask]
    Pe_ratio    = Pe[mask] / Pe_ref_fit

    # Time + T2 envelope
    rot_len_ns = float(rot_len_ns)
    t_ns = rot_len_ns * (K * N_fit + 1)   # (K N + 1) pulses, each of length τ
    t_s  = t_ns * 1e-9

    attr   = experiment.attributes
    T2_eff = getattr(attr, "T2_ramsey", None)  # assume seconds; rescale if needed

    if (T2_eff is None) or (T2_eff <= 0):
        def E2_func(t):
            return np.ones_like(t)
    else:
        def E2_func(t):
            return np.exp(-t / T2_eff)

    E2 = E2_func(t_s)

    # ---------- Fit model: R(N) = 1 + A * E2(N) * sin(K N δ) ----------
    # here K=4, so total extra angle = 4N δ_{x90}
    def ratio_model(N_vals, A, delta):
        N_vals = np.asarray(N_vals, dtype=float)
        idx = N_vals.astype(int)
        # map idx into local index [0..len(N_fit)-1]
        pos = np.searchsorted(N_fit, idx)
        return 1.0 + A * E2[pos] * np.sin(K * idx * delta)

    fit_p0 = [10.0, 1e-4]

    fit_result = fitting.generalized_fit(
        N_fit,
        Pe_ratio,
        ratio_model,
        fit_p0,
        plotting=False,
        param_format="{:.4e}",
    )

    # figure out how generalized_fit returns the parameters
    try:
        A_fit, delta_fit = fit_result[0]
    except (TypeError, KeyError):
        A_fit, delta_fit = fit_result

    # x90 amplitude correction:
    # θ_{x90} = π/2 + δ  ⇒  scale = (π/2)/(π/2 + δ)
    x90_correction = (np.pi / 2.0) / (np.pi / 2.0 + delta_fit)

    print(f"[x90 calibration] Fit A: {A_fit:.4e}")
    print(f"[x90 calibration] Fit δ (rad): {delta_fit:.4e}")
    print(f"Suggested x90 amplitude scale factor: {x90_correction:.8f}")
    print(f"(~1 - 2δ/π = {1 - 2*delta_fit/np.pi:.8f})")

    # ---------- Plot normalized data + fit ----------
    Pe_ratio_fit = ratio_model(N_fit, A_fit, delta_fit)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(N_fit, Pe_ratio, "o", label=r"Data: $P_e/P_e^{\rm ref}$")
    ax.plot(N_fit, Pe_ratio_fit, "-", label="Fit", linewidth=1.5)
    ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.5)

    ax.set_xlabel(r"$N$ (blocks; $4N$ $x_{90}$ pulses)")
    ax.set_ylabel(r"$P_e/P_e^{\rm ref}$")
    ax.set_title(r"Normalized x90 Pulse-Train Calibration (T$_1$ canceled)")
    ax.grid(True, alpha=0.3)
    ax.set_xticks(N_fit)
    ax.legend()

    secax = ax.secondary_xaxis("top", functions=(N_to_t_us, t_us_to_N))
    secax.set_xlabel("Total sequence time (µs)")

    fig.tight_layout()
    plt.show()


### All XY

In [ ]:
rr = experiment.all_XY(gate_indices=None, prefix="", qb_detuning=0*u.MHz, n_avg=20000)

if rr.mode == "simulate":
    print("Simulation run — no stream data; skipping All-XY plot.")
else:
    out = rr.output
    raw_sz, sz, ops = out.extract("raw_sz", "sz", "ops")

    attr = experiment.attributes
    ro_duration = measureMacro.active_length()
    print(raw_sz)

    plt.figure(figsize=(10, 6))
    plt.title("All XY")
    x_positions = np.arange(1, len(ops) + 1)
    plt.bar(x_positions, sz, label=r"$\sigma_z$")
    plt.axhline(y=1, color='red', linestyle='--', linewidth=1.5, label=r'$\sigma_z = +1$')
    plt.axhline(y=-1, color='blue', linestyle='--', linewidth=1.5, label=r'$\sigma_z = -1$')
    labels = [f"{op[0]}-{op[1]}" for op in ops]
    plt.xticks(x_positions, labels, rotation="vertical")
    plt.ylabel(r"$<\sigma_z>$")
    plt.xlabel("Gate pair")
    plt.legend()
    plt.tight_layout()
    plt.show()


### Randomized Benchmarking

In [ ]:
m_list = np.arange(0, 120, 16)
num_sequence = 20

runres = experiment.randomized_benchmarking(
    m_list=m_list,
    num_sequence=num_sequence,
    n_avg=10000,
    max_sequences_per_compile=5,
    primitive_prefix="",          # or primitives_by_id=...
    interleave_op=None,           # <-- explicit (optional)
)

Pe = runres.output.extract("Pe_corr")      # shape (len(m_list), num_sequence)
surv_list = 1.0 - Pe.mean(axis=1)

popt, pcov = fitting.generalized_fit(
    m_list,
    surv_list,
    cQED_models.rb_survival_model,
    p0=[0.99, 0.5, 0.5],
    plotting=True,
    param_format="{:.4f}",
    plot_options={
        "figsize": (9, 4),
        "xlabel": "m (number of Cliffords)",
        "ylabel": "Survival probability",
        "title": "Randomized Benchmarking",
    },
)

fit_p, fit_A, fit_B = popt

r_cl = (1.0 - fit_p) / 2.0          # avg error per Clifford (1 qubit)
F_cl = (1.0 + fit_p) / 2.0          # avg Clifford fidelity

print(f"p      = {fit_p:.6f}")
print(f"Average error per Clifford   = {r_cl:.3e}")
print(f"Average Clifford fidelity    = {F_cl*100:.6f}%")


In [ ]:
qpm.serialize_program()

### Interleaved Randomized Benchmarking (iRB) for unselective pulse

In [ ]:
# ----------------------------
# Multi-Gate Interleaved Randomized Benchmarking
# Quantify fidelity of x90, x180, y90, y180
# ----------------------------

m_list = np.array([0, 2, 5, 8, 12, 20, 30, 45, 60, 80, 120, 140, 160])
num_sequence = 40
n_avg = 10000

# Define gates to test
gates_to_test = ["x90", "x180", "y90", "y180"]

# 1) Reference RB (baseline) - run once
print("=" * 60)
print("Running Reference RB (no interleaving)...")
print("=" * 60)

run_ref = experiment.randomized_benchmarking(
    m_list=m_list,
    num_sequence=num_sequence,
    n_avg=n_avg,
    max_sequences_per_compile=10,
    primitive_prefix="",
    interleave_op=None,
)

Pe_ref = run_ref.output.extract("Pe_corr")
surv_ref = 1.0 - Pe_ref.mean(axis=1)

# Fit reference RB (without plotting)
popt_ref, pcov_ref = fitting.generalized_fit(
    m_list,
    surv_ref,
    cQED_models.rb_survival_model,
    p0=[0.99, 0.5, 0.5],
    plotting=False,  # Disable individual plotting
    param_format="{:.4f}",
)
p_ref, A_ref, B_ref = popt_ref

print(f"\nReference RB: p_ref = {p_ref:.6f}")
print(f"EPC (avg error per Clifford) = {(1.0 - p_ref) / 2.0:.3e}")
print(f"Avg Clifford fidelity = {(1.0 + p_ref) / 2.0 * 100:.6f}%\n")

# 2) Run interleaved RB for each gate
results = {}

for gate in gates_to_test:
    print("=" * 60)
    print(f"Running Interleaved RB for gate: {gate}")
    print("=" * 60)
    
    run_int = experiment.randomized_benchmarking(
        m_list=m_list,
        num_sequence=num_sequence,
        n_avg=n_avg,
        max_sequences_per_compile=10,
        primitive_prefix="",
        interleave_op=gate,
    )
    
    Pe_int = run_int.output.extract("Pe_corr")
    surv_int = 1.0 - Pe_int.mean(axis=1)
    
    # Fit interleaved RB (without plotting)
    popt_int, pcov_int = fitting.generalized_fit(
        m_list,
        surv_int,
        cQED_models.rb_survival_model,
        p0=[0.99, 0.5, 0.5],
        plotting=False,  # Disable individual plotting
        param_format="{:.4f}",
    )
    p_int, A_int, B_int = popt_int
    
    # Calculate gate fidelity
    p_G = p_int / p_ref
    r_G = (1.0 - p_G) / 2.0
    F_G = 1.0 - r_G
    
    # Store results
    results[gate] = {
        'surv_data': surv_int,
        'p_int': p_int,
        'A': A_int,
        'B': B_int,
        'p_G': p_G,
        'r_G': r_G,
        'F_G': F_G,
    }
    
    print(f"  p_int = {p_int:.6f}, p_G = {p_G:.6f}, F_G = {F_G*100:.6f}%\n")

# 3) Plot all results together
fig, ax = plt.subplots(figsize=(12, 7))

# Define colors and markers for each gate
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
markers = ['o', 's', '^', 'D', 'v']

# Plot reference RB
m_fine = np.linspace(min(m_list), max(m_list), 200)
surv_ref_fit = cQED_models.rb_survival_model(m_fine, p_ref, A_ref, B_ref)

ax.plot(m_fine, surv_ref_fit, '-', color='black', linewidth=2.5, 
        label=f'Reference (p={p_ref:.4f})', zorder=10)
ax.plot(m_list, surv_ref, 'o', color='black', markersize=8, 
        markerfacecolor='white', markeredgewidth=2, zorder=11)

# Plot each interleaved RB
for i, gate in enumerate(gates_to_test):
    res = results[gate]
    surv_int_fit = cQED_models.rb_survival_model(m_fine, res['p_int'], res['A'], res['B'])
    
    ax.plot(m_fine, surv_int_fit, '-', color=colors[i], linewidth=2, 
            label=f'{gate} (p={res["p_int"]:.4f}, F={res["F_G"]*100:.2f}%)', 
            alpha=0.85)
    ax.plot(m_list, res['surv_data'], markers[i], color=colors[i], 
            markersize=7, alpha=0.85)

ax.set_xlabel('Number of Cliffords (m)', fontsize=13)
ax.set_ylabel('Survival Probability', fontsize=13)
ax.set_title('Interleaved Randomized Benchmarking: Multi-Gate Fidelity', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='best', framealpha=0.95)
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

# 4) Print summary table
print("\n" + "=" * 80)
print("GATE FIDELITY SUMMARY")
print("=" * 80)
print(f"{'Gate':<10} {'p_int':<12} {'p_G':<12} {'Error Rate (r_G)':<20} {'Fidelity (F_G)':<15}")
print("-" * 80)
for gate in gates_to_test:
    res = results[gate]
    print(f"{gate:<10} {res['p_int']:<12.6f} {res['p_G']:<12.6f} {res['r_G']:<20.3e} {res['F_G']*100:<15.6f}%")
print("=" * 80)

In [ ]:
qpm.serialize_program()

### Interleaved Randomized Benchmarking (iRB) for selective pulse

In [ ]:
# ----------------------------
# Multi-Gate Interleaved Randomized Benchmarking - SELECTIVE PULSES
# Quantify fidelity of sel_x90, sel_x180, sel_y90, sel_y180
# ----------------------------

m_list = [1, 4, 8, 16, 32, 40, 48, 64]
num_sequence = 30
n_avg = 10000

# Define selective gates to test
sel_gates_to_test = ["sel_x90", "sel_x180", "sel_y90", "sel_y180"]

# 1) Reference RB (baseline) - run once
print("=" * 60)
print("Running Reference RB for Selective Pulses (no interleaving)...")
print("=" * 60)

run_ref_sel = experiment.randomized_benchmarking(
    m_list=m_list,
    num_sequence=num_sequence,
    n_avg=n_avg,
    max_sequences_per_compile=10,
    primitive_prefix="",
    interleave_op=None,
)

Pe_ref_sel = run_ref_sel.output.extract("Pe_corr")
surv_ref_sel = 1.0 - Pe_ref_sel.mean(axis=1)

# Fit reference RB (without plotting)
popt_ref_sel, pcov_ref_sel = fitting.generalized_fit(
    m_list,
    surv_ref_sel,
    cQED_models.rb_survival_model,
    p0=[0.99, 0.5, 0.5],
    plotting=False,  # Disable individual plotting
    param_format="{:.4f}",
)
p_ref_sel, A_ref_sel, B_ref_sel = popt_ref_sel

print(f"\nReference RB: p_ref = {p_ref_sel:.6f}")
print(f"EPC (avg error per Clifford) = {(1.0 - p_ref_sel) / 2.0:.3e}")
print(f"Avg Clifford fidelity = {(1.0 + p_ref_sel) / 2.0 * 100:.6f}%\n")

# 2) Run interleaved RB for each selective gate
results_sel = {}

for gate in sel_gates_to_test:
    print("=" * 60)
    print(f"Running Interleaved RB for selective gate: {gate}")
    print("=" * 60)
    
    run_int_sel = experiment.randomized_benchmarking(
        m_list=m_list,
        num_sequence=num_sequence,
        n_avg=n_avg,
        max_sequences_per_compile=10,
        primitive_prefix="",
        interleave_op=gate,
    )
    
    Pe_int_sel = run_int_sel.output.extract("Pe_corr")
    surv_int_sel = 1.0 - Pe_int_sel.mean(axis=1)
    
    # Fit interleaved RB (without plotting)
    popt_int_sel, pcov_int_sel = fitting.generalized_fit(
        m_list,
        surv_int_sel,
        cQED_models.rb_survival_model,
        p0=[0.99, 0.5, 0.5],
        plotting=False,  # Disable individual plotting
        param_format="{:.4f}",
    )
    p_int_sel, A_int_sel, B_int_sel = popt_int_sel
    
    # Calculate gate fidelity
    p_G_sel = p_int_sel / p_ref_sel
    r_G_sel = (1.0 - p_G_sel) / 2.0
    F_G_sel = 1.0 - r_G_sel
    
    # Store results
    results_sel[gate] = {
        'surv_data': surv_int_sel,
        'p_int': p_int_sel,
        'A': A_int_sel,
        'B': B_int_sel,
        'p_G': p_G_sel,
        'r_G': r_G_sel,
        'F_G': F_G_sel,
    }
    
    print(f"  p_int = {p_int_sel:.6f}, p_G = {p_G_sel:.6f}, F_G = {F_G_sel*100:.6f}%\n")

# 3) Plot all results together
fig, ax = plt.subplots(figsize=(12, 7))

# Define colors and markers for each selective gate
colors_sel = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
markers_sel = ['o', 's', '^', 'D', 'v']

# Plot reference RB
m_fine = np.linspace(min(m_list), max(m_list), 200)
surv_ref_sel_fit = cQED_models.rb_survival_model(m_fine, p_ref_sel, A_ref_sel, B_ref_sel)

ax.plot(m_fine, surv_ref_sel_fit, '-', color='black', linewidth=2.5, 
        label=f'Reference (p={p_ref_sel:.4f})', zorder=10)
ax.plot(m_list, surv_ref_sel, 'o', color='black', markersize=8, 
        markerfacecolor='white', markeredgewidth=2, zorder=11)

# Plot each interleaved RB
for i, gate in enumerate(sel_gates_to_test):
    res = results_sel[gate]
    surv_int_sel_fit = cQED_models.rb_survival_model(m_fine, res['p_int'], res['A'], res['B'])
    
    ax.plot(m_fine, surv_int_sel_fit, '-', color=colors_sel[i], linewidth=2, 
            label=f'{gate} (p={res["p_int"]:.4f}, F={res["F_G"]*100:.2f}%)', 
            alpha=0.85)
    ax.plot(m_list, res['surv_data'], markers_sel[i], color=colors_sel[i], 
            markersize=7, alpha=0.85)

ax.set_xlabel('Number of Cliffords (m)', fontsize=13)
ax.set_ylabel('Survival Probability', fontsize=13)
ax.set_title('Interleaved RB: Selective Pulse Fidelity', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='best', framealpha=0.95)
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

# 4) Print summary table
print("\n" + "=" * 80)
print("SELECTIVE GATE FIDELITY SUMMARY")
print("=" * 80)
print(f"{'Gate':<15} {'p_int':<12} {'p_G':<12} {'Error Rate (r_G)':<20} {'Fidelity (F_G)':<15}")
print("-" * 80)
for gate in sel_gates_to_test:
    res = results_sel[gate]
    print(f"{gate:<15} {res['p_int']:<12.6f} {res['p_G']:<12.6f} {res['r_G']:<20.3e} {res['F_G']*100:<15.6f}%")
print("=" * 80)

### Qubit State Tomography

In [ ]:
from scipy.spatial.transform import Rotation as R
from qm.qua import *
def _fmt_row(label: str, v: np.ndarray) -> str:
    return (
        f"  {label:<10}"
        f"X = {v[0]:>+8.4f}  "
        f"Y = {v[1]:>+8.4f}  "
        f"Z = {v[2]:>+8.4f}  "
        f"|s| = {np.linalg.norm(v):>8.4f}"
    )

theta = np.pi
phi = 0
axis = np.array([np.cos(phi), np.sin(phi), 0.0], dtype=float)   # XY-plane axis at phase phi
ideal_vec = R.from_rotvec(theta * axis).apply([0.0, 0.0, 1.0])
#ideal_vec = R.from_euler('zy', [phi, theta]).apply([0, 0, 1])
arb_rot = QubitRotation(theta, phi, build=True)

experiment.burn_pulses()

attr = experiment.attributes
qb_fock_0_if = qpm.calculate_el_if_fq("qubit", attr.fock_fqs[0])
qb_bare_if = qpm.calculate_el_if_fq("qubit", attr.qb_fq)
n_avg = 100000
def state_prep():
    arb_rot.play()

    align()

runres = experiment.qubit_state_tomography(state_prep, n_avg)
raw_sx, raw_sy, raw_sz, sx, sy, sz = runres.output.extract(
    "raw_sx", "raw_sy", "raw_sz", "sx", "sy", "sz"
)

raw_vec  = np.array([raw_sx, raw_sy, raw_sz], dtype=float)
corr_vec = np.array([sx,    sy,    sz   ], dtype=float)
corr_vec_norm = corr_vec / np.linalg.norm(corr_vec) * np.linalg.norm(ideal_vec)
print(_fmt_row("ideal", ideal_vec))
print(_fmt_row("raw", raw_vec))
print(_fmt_row("corrected", corr_vec))
print(_fmt_row("corr (normed)", corr_vec_norm))
ideal_norm  = float(np.linalg.norm(ideal_vec))
corr_norm = float(np.linalg.norm(corr_vec))
costheta = np.dot(ideal_vec, corr_vec)/(ideal_norm*corr_norm)
print(f"cos(θ) = {costheta:.5f}")
print(f"θ beween ideal and corrected vector {(np.arccos(costheta)*180/np.pi):.3f} degrees")
ideal_bloch_vecs = cQED_models.qubit_pulse_train_model(1, theta, phi, amp_err=0)
print(_fmt_row("ideal (model)", ideal_bloch_vecs))

### Convention calibration

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R
from qm.qua import play, align

# ----------------------------
# Pretty printing + utilities
# ----------------------------
def _fmt_row(label: str, v: np.ndarray) -> str:
    v = np.asarray(v, float).reshape(3)
    return (
        f"  {label:<16}"
        f"X = {v[0]:>+9.5f}  "
        f"Y = {v[1]:>+9.5f}  "
        f"Z = {v[2]:>+9.5f}  "
        f"|s| = {np.linalg.norm(v):>9.5f}"
    )

def _angle_deg(a: np.ndarray, b: np.ndarray, eps: float = 1e-15) -> float:
    a = np.asarray(a, float).reshape(3)
    b = np.asarray(b, float).reshape(3)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < eps or nb < eps:
        return np.nan
    c = float(np.dot(a, b) / (na * nb))
    c = np.clip(c, -1.0, 1.0)
    return float(np.degrees(np.arccos(c)))

def _norm_to(v: np.ndarray, target_norm: float, eps: float = 1e-15) -> np.ndarray:
    v = np.asarray(v, float).reshape(3)
    n = np.linalg.norm(v)
    if n < eps:
        return v.copy()
    return v / n * target_norm

def ideal_bloch_so3(theta: float, phi: float, r0=(0.0,0.0,1.0)) -> np.ndarray:
    axis = np.array([np.cos(phi), np.sin(phi), 0.0], dtype=float)
    return R.from_rotvec(theta * axis).apply(np.asarray(r0, float))

# ----------------------------
# Printing helpers
# ----------------------------
def print_results_block(res: dict, *, use_model: bool = True):
    theta = res["theta"]
    phi = res["phi"]
    name = res["named_pulse"]

    print("\n" + "="*92)
    print(f"Named pulse: '{name}'   vs   Param: QubitRotation(theta,phi)")
    print(f"theta={theta/np.pi:.4f}π, phi={phi/np.pi:.4f}π")
    print("-"*92)

    print(_fmt_row("ideal (SO3)", res["v_so3"]))
    if use_model and (res["v_model"] is not None):
        print(_fmt_row("ideal (model)", res["v_model"]))

    print("\n-- Tomography results --")
    print(_fmt_row("named raw", res["raw_named"]))
    print(_fmt_row("named corr", res["corr_named"]))
    print(_fmt_row("named corrN", res["corr_named_normed"]))

    print(_fmt_row("param raw", res["raw_param"]))
    print(_fmt_row("param corr", res["corr_param"]))
    print(_fmt_row("param corrN", res["corr_param_normed"]))

    print("\n-- Angle comparisons (deg) --")
    ang = res["angles_deg"]
    print(f"  angle(named corr, param corr)  = {ang['named_vs_param']:8.3f}")
    print(f"  angle(named corr, ideal SO3)   = {ang['named_vs_so3']:8.3f}")
    print(f"  angle(param corr, ideal SO3)   = {ang['param_vs_so3']:8.3f}")
    if use_model and ("named_vs_model" in ang):
        print(f"  angle(named corr, model)       = {ang['named_vs_model']:8.3f}")
        print(f"  angle(param corr, model)       = {ang['param_vs_model']:8.3f}")
        print(f"  angle(ideal SO3, model)        = {ang['so3_vs_model']:8.6f}")

def print_summary_table(results: list[dict], *, use_model: bool = True):
    cols = ["pulse", "theta/pi", "phi/pi", "∠(named,param)", "∠(named,SO3)", "∠(param,SO3)"]
    if use_model:
        cols += ["∠(named,model)", "∠(param,model)"]

    header = " | ".join(f"{c:>14}" for c in cols)
    print("\n" + header)
    print("-" * len(header))

    for r in results:
        ang = r["angles_deg"]
        row = [
            r["named_pulse"],
            f"{r['theta']/np.pi:+.3f}",
            f"{r['phi']/np.pi:+.3f}",
            f"{ang['named_vs_param']:+7.2f}",
            f"{ang['named_vs_so3']:+7.2f}",
            f"{ang['param_vs_so3']:+7.2f}",
        ]
        if use_model:
            row += [
                f"{ang.get('named_vs_model', np.nan):+7.2f}",
                f"{ang.get('param_vs_model', np.nan):+7.2f}",
            ]
        print(" | ".join(f"{x:>14}" for x in row))

# ----------------------------
# Run the full suite (BUNDLED into one tomography call)
# ----------------------------
def run_named_param_suite(n_avg: int = 25000, *, use_model: bool = True, print_blocks: bool = True):
    suite = [
        ("x90",   +np.pi/2, 0.0),
        ("xn90",  -np.pi/2, 0.0),
        ("y90",   +np.pi/2, np.pi/2),
        ("yn90",  -np.pi/2, np.pi/2),
        ("x180",  +np.pi,   0.0),
        ("y180",  +np.pi,   np.pi/2),
    ]

    # ---- 1) Build all state-prep callables & parametric rotations ----
    state_preps = []   # flat list of callables, length = 2 * len(suite)
    labels = []        # tracking: (suite_idx, "named"/"param")

    arb_rots = {}
    for idx, (name, theta, phi) in enumerate(suite):
        # Build the parametric QubitRotation for this entry
        arb_rot = QubitRotation(theta, phi, build=True)
        arb_rots[idx] = arb_rot

        # Named pulse state-prep
        def _make_named_prep(pulse_name):
            def prep():
                play(pulse_name, "qubit")
                align()
            return prep

        # Parametric pulse state-prep
        def _make_param_prep(rot):
            def prep():
                rot.play()
                align()
            return prep

        state_preps.append(_make_named_prep(name))
        labels.append((idx, "named"))

        state_preps.append(_make_param_prep(arb_rot))
        labels.append((idx, "param"))

    # Burn all pulses once (all QubitRotations are already built)
    experiment.burn_pulses()

    n_preps = len(state_preps)
    print(f"Running bundled tomography: {n_preps} state preps × {n_avg} avg  (1 program call)")

    # ---- 2) Single bundled tomography call ----
    runres = experiment.qubit_state_tomography(state_preps, n_avg)

    # Extract arrays — shape (P,) for multi-prep
    raw_sx, raw_sy, raw_sz, sx, sy, sz = runres.output.extract(
        "raw_sx", "raw_sy", "raw_sz", "sx", "sy", "sz"
    )
    raw_sx = np.asarray(raw_sx, float).ravel()
    raw_sy = np.asarray(raw_sy, float).ravel()
    raw_sz = np.asarray(raw_sz, float).ravel()
    sx = np.asarray(sx, float).ravel()
    sy = np.asarray(sy, float).ravel()
    sz = np.asarray(sz, float).ravel()

    # ---- 3) Unpack into per-suite-entry results ----
    # We have pairs: (named_0, param_0, named_1, param_1, ...)
    tomo = {}  # (suite_idx, kind) -> (raw_vec, corr_vec)
    for i, (suite_idx, kind) in enumerate(labels):
        raw_vec  = np.array([raw_sx[i], raw_sy[i], raw_sz[i]], dtype=float)
        corr_vec = np.array([sx[i],     sy[i],     sz[i]],     dtype=float)
        tomo[(suite_idx, kind)] = (raw_vec, corr_vec)

    results = []
    for idx, (name, theta, phi) in enumerate(suite):
        v_so3 = ideal_bloch_so3(theta, phi)
        v_model = None
        if use_model:
            v_model = cQED_models.qubit_pulse_train_model(
                1, theta, phi, r0=(0.0, 0.0, 1.0),
                delta=0.0, amp_err=0.0, phase_err=0.0
            )

        rawA,  corrA  = tomo[(idx, "named")]
        rawB,  corrB  = tomo[(idx, "param")]
        corrA_n = _norm_to(corrA, np.linalg.norm(v_so3))
        corrB_n = _norm_to(corrB, np.linalg.norm(v_so3))

        angles = {
            "named_vs_param": _angle_deg(corrA, corrB),
            "named_vs_so3":   _angle_deg(corrA, v_so3),
            "param_vs_so3":   _angle_deg(corrB, v_so3),
        }
        if use_model and v_model is not None:
            angles |= {
                "named_vs_model": _angle_deg(corrA, v_model),
                "param_vs_model": _angle_deg(corrB, v_model),
                "so3_vs_model":   _angle_deg(v_so3, v_model),
            }

        results.append({
            "named_pulse": name,
            "theta": float(theta),
            "phi": float(phi),
            "v_so3": v_so3,
            "v_model": v_model,
            "raw_named": rawA,
            "corr_named": corrA,
            "corr_named_normed": corrA_n,
            "raw_param": rawB,
            "corr_param": corrB,
            "corr_param_normed": corrB_n,
            "angles_deg": angles,
        })

    # ---- 4) Print summary ----
    print_summary_table(results, use_model=use_model)
    if print_blocks:
        for res in results:
            print_results_block(res, use_model=use_model)

    return results

# ---- run everything ----
results = run_named_param_suite(n_avg=25000, use_model=True, print_blocks=True)

### tomography pulse train

In [ ]:
# Run arbitrary rotation N times and measure corrected Bloch vector components
# BUNDLED: all N values run in a single tomography program call
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
from qm.qua import play, align

# Define the rotation parameters
theta = np.pi
phi = 0
axis = np.array([np.cos(phi), np.sin(phi), 0.0], dtype=float)
ideal_vec = R.from_rotvec(theta * axis).apply([0.0, 0.0, 1.0])

pulse_type = "unselective"
# Create rotation gate
arb_rot = QubitRotation(theta, phi, build=True)
experiment.burn_pulses()

# Define N values (number of times to apply the rotation)
N_values = np.arange(0, 50, 4)  # 0 to 20 rotations
n_avg = 25000

# ---- Build one state-prep callable per N value ----
def _make_prep(rot, n_reps):
    """Return a QUA callable that plays `rot` exactly `n_reps` times."""
    def prep():
        for _ in range(n_reps):
            rot.play()
            align()
    return prep

state_preps = [_make_prep(arb_rot, int(N)) for N in N_values]

print(f"Running bundled tomography: {len(state_preps)} state preps × {n_avg} avg  (1 program call)")

# ---- Single bundled tomography call ----
runres = experiment.qubit_state_tomography(state_preps, n_avg)

# Extract arrays — shape (P,) for multi-prep
raw_sx, raw_sy, raw_sz, sx, sy, sz = runres.output.extract(
    "raw_sx", "raw_sy", "raw_sz", "sx", "sy", "sz"
)
X_corr = np.asarray(sx, float).ravel()
Y_corr = np.asarray(sy, float).ravel()
Z_corr = np.asarray(sz, float).ravel()

# Normalize corrected vectors to ideal norm
corr_vecs = np.column_stack([X_corr, Y_corr, Z_corr])  # (P, 3)
norms = np.linalg.norm(corr_vecs, axis=1, keepdims=True)
norms[norms < 1e-15] = 1.0  # avoid division by zero
corr_vecs_norm = corr_vecs / norms * np.linalg.norm(ideal_vec)
X_corr_norm = corr_vecs_norm[:, 0]
Y_corr_norm = corr_vecs_norm[:, 1]
Z_corr_norm = corr_vecs_norm[:, 2]

# ---- Print summary ----
print(f"\nCompleted all {len(N_values)} measurements!")
for i, N in enumerate(N_values):
    print(f"  N={N:3d}  Corr: X={X_corr[i]:+.4f}, Y={Y_corr[i]:+.4f}, Z={Z_corr[i]:+.4f}"
          f"  Norm: X={X_corr_norm[i]:+.4f}, Y={Y_corr_norm[i]:+.4f}, Z={Z_corr_norm[i]:+.4f}")

# ---- Plots ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Corrected (not normalized)
ax1.plot(N_values, X_corr, 'o-', label='X (corrected)', color='red', markersize=6)
ax1.plot(N_values, Y_corr, 's-', label='Y (corrected)', color='green', markersize=6)
ax1.plot(N_values, Z_corr, '^-', label='Z (corrected)', color='blue', markersize=6)
ax1.set_xlabel('N (Number of rotations)', fontsize=12)
ax1.set_ylabel('Bloch vector component', fontsize=12)
ax1.set_title(f'Corrected Bloch components vs N rotations ({pulse_type} pulse)\n(θ={theta/np.pi:.2f}π, φ={phi/np.pi:.2f}π)', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)

# Plot 2: Corrected (normalized)
ax2.plot(N_values, X_corr_norm, 'o-', label='X (corrected, normalized)', color='red', markersize=6)
ax2.plot(N_values, Y_corr_norm, 's-', label='Y (corrected, normalized)', color='green', markersize=6)
ax2.plot(N_values, Z_corr_norm, '^-', label='Z (corrected, normalized)', color='blue', markersize=6)
ax2.set_xlabel('N (Number of rotations)', fontsize=12)
ax2.set_ylabel('Bloch vector component', fontsize=12)
ax2.set_title(f'Corrected (normalized) Bloch components vs N rotations ({pulse_type} pulse)\n(θ={theta/np.pi:.2f}π, φ={phi/np.pi:.2f}π)', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\nIdeal vector after 1 rotation: X={ideal_vec[0]:.4f}, Y={ideal_vec[1]:.4f}, Z={ideal_vec[2]:.4f}")

### Bayes pulse optimization using TOMO

In [ ]:
# ============================================================
# BAYESIAN OPTIMIZATION (GP + Expected Improvement)
# Closed-loop calibration for QubitRotation using BO over:
#   x = [delta_hat, amp_err_hat, phase_err_hat, zeta]
#
# Mapping:
#   (delta_hat, amp_err_hat, phase_err_hat) -> (d_omega, d_lambda, d_alpha)
#      via fit_params_to_qubitrotation_knobs(dt_s, n_samp)
#   zeta is optimized directly and used as a forced global Z-frame in the cost
#
# Features:
#  1) clear_output(wait=True) at each evaluation (optional)
#  2) prints cost + fit_params + knobs every evaluation
#  3) END: plots cost vs eval + running best (and DOES NOT get cleared)
#
# Requirements:
#   numpy, scipy, matplotlib, scikit-learn
#   You already have:
#     - make_state_prep(prep_rot, arb_rot, N)
#     - extract_bloch_from_tomo(runres)
#     - fit_params_to_qubitrotation_knobs(...)
#   And objects:
#     - experiment (has qubit_state_tomography, burn_pulses, maybe attributes.dt_s)
#     - QubitRotation
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from scipy.spatial.transform import Rotation as R

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel


# ----------------------------
# Small helpers
# ----------------------------

def safe_normalize(v, eps=1e-12):
    v = np.asarray(v, float)
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.maximum(n, eps)

def wrap_pm_pi(x: float) -> float:
    return float((x + np.pi) % (2 * np.pi) - np.pi)

def apply_rz_xy(vec, zeta):
    c = np.cos(zeta); s = np.sin(zeta)
    x, y, z = float(vec[0]), float(vec[1]), float(vec[2])
    return np.array([c*x - s*y, s*x + c*y, z], float)

def build_ideal_dirs_for_target(theta, phi, r0_dict):
    axis = np.array([np.cos(phi), np.sin(phi), 0.0], float)
    rot = R.from_rotvec(float(theta) * axis)
    out = {}
    for k, r0 in r0_dict.items():
        out[k] = safe_normalize(rot.apply(np.asarray(r0, float)))
    return out

def direction_cost(meas_dirs, ideal_dirs, *, weights=None, zeta_force=None):
    """
    Cost = sum_k w_k * (1 - dot( meas_hat[k], Rz(zeta_force)*ideal_hat[k] ))
    where all vectors are normalized.
    """
    keys = list(meas_dirs.keys())
    if weights is None:
        weights = {k: 1.0 for k in keys}

    meas_hat  = {k: safe_normalize(meas_dirs[k])  for k in keys}
    ideal_hat = {k: safe_normalize(ideal_dirs[k]) for k in keys}

    zeta_hat = 0.0 if zeta_force is None else float(wrap_pm_pi(zeta_force))

    dots = {}
    cost = 0.0
    for k in keys:
        w = float(weights.get(k, 1.0))
        rm = np.asarray(meas_hat[k], float)
        ri = np.asarray(ideal_hat[k], float)
        ri = apply_rz_xy(ri, zeta_hat)

        d = float(np.clip(np.dot(rm, ri), -1.0, 1.0))
        dots[k] = d
        cost += w * (1.0 - d)

    return float(cost), float(zeta_hat), dots


# ----------------------------
# BO core: scaling + EI
# ----------------------------

def _bounds_to_arrays(bounds):
    names = list(bounds.keys())
    lo = np.array([bounds[n][0] for n in names], float)
    hi = np.array([bounds[n][1] for n in names], float)
    return names, lo, hi

def _scale01(X, lo, hi):
    X = np.asarray(X, float)
    return (X - lo) / (hi - lo)

def _unscale01(U, lo, hi):
    U = np.asarray(U, float)
    return lo + U * (hi - lo)

def expected_improvement(mu, sigma, y_best, xi=0.01):
    # EI for MINIMIZATION
    mu = np.asarray(mu, float)
    sigma = np.asarray(sigma, float)
    sigma = np.maximum(sigma, 1e-12)
    from scipy.stats import norm
    imp = (y_best - mu - xi)
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return np.maximum(ei, 0.0)

def suggest_next_by_ei(gp, lo, hi, *, n_cand=8000, rng=None, xi=0.01):
    if rng is None:
        rng = np.random.default_rng()
    d = len(lo)

    U = rng.random((int(n_cand), d))
    Xcand = _unscale01(U, lo, hi)

    mu, std = gp.predict(_scale01(Xcand, lo, hi), return_std=True)
    y_best = float(np.min(gp.y_train_))
    ei = expected_improvement(mu, std, y_best, xi=xi)
    j = int(np.argmax(ei))
    return Xcand[j], float(ei[j])


# ----------------------------
# Main BO (4D: delta_hat, amp_err_hat, phase_err_hat, zeta)
# ----------------------------

def bayes_optimize_qubitrotation_fitparams_with_zeta(
    *,
    experiment,
    QubitRotation,
    theta,
    phi,
    prep_defs,
    n_avg: int,

    # bounds in FIT-PARAM space
    bounds_fit=None,  # dict(delta_hat=..., amp_err_hat=..., phase_err_hat=..., zeta=...)

    # conversion needs pulse length
    dt_s: float | None = None,
    n_samp: int | None = None,
    waveform_source_rot=None,          # if n_samp not provided, must have .waveforms()
    d_omega_sign: float = +1.0,

    # gate extras
    ref_r180_pulse=None,
    build_kwargs=None,

    # cost settings
    weights=None,

    # BO settings
    n_init=8,
    n_iter=25,
    repeats_per_eval=1,
    candidate_pool=8000,
    xi=0.02,
    random_seed=3,

    # UX
    verbose=True,
    plot_progress=True,
    clear_each_eval=True,
):
    theta = float(theta); phi = float(phi)
    if build_kwargs is None:
        build_kwargs = {}

    # dt_s inference
    if dt_s is None:
        dt_s = float(getattr(getattr(experiment, "attributes", object()), "dt_s", 1e-9))

    # n_samp inference
    if n_samp is None:
        if waveform_source_rot is None:
            raise ValueError("Provide n_samp=... or waveform_source_rot=... (must have .waveforms()).")
        _Iw, _Qw, wf_len, _marker = waveform_source_rot.waveforms()
        n_samp = int(wf_len)

    dt_s = float(dt_s)
    n_samp = int(n_samp)
    T = dt_s * n_samp

    # bounds (4D)
    if bounds_fit is None:
        bounds_fit = dict(
            delta_hat=(-0.20, +0.20),        # rad/pulse
            amp_err_hat=(-0.03, +0.03),      # fraction
            phase_err_hat=(-0.50, +0.50),    # rad
            zeta=(-np.pi, +np.pi),           # rad
        )

    names, lo, hi = _bounds_to_arrays(bounds_fit)
    rng = np.random.default_rng(int(random_seed))
    cache = {}

    # Ideal directions
    r0_dict = {
        "g":  np.array([0.0, 0.0, 1.0]),
        "+x": np.array([1.0, 0.0, 0.0]),
        "+y": np.array([0.0, 1.0, 0.0]),
        "e":  np.array([0.0, 0.0, -1.0]),
        "-x": np.array([-1.0, 0.0, 0.0]),
        "-y": np.array([0.0, -1.0, 0.0]),
    }
    ideal_all = build_ideal_dirs_for_target(theta, phi, r0_dict)
    ideal_dirs = {k: ideal_all[k] for k in prep_defs.keys()}

    if weights is None:
        weights = {k: 1.0 for k in prep_defs.keys()}

    def x_to_dict(x):
        x = np.asarray(x, float).ravel()
        return {names[i]: float(x[i]) for i in range(len(names))}

    # storage
    X, y, hist = [], [], []
    eval_counter = 0

    # GP
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(length_scale=np.ones(len(names)), length_scale_bounds=(1e-3, 1e3), nu=2.5)
        + WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-8, 1e-1))
    )
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=0.0,
        normalize_y=True,
        n_restarts_optimizer=3,
        random_state=int(random_seed),
    )

    def _print_eval(tag, out, extra=""):
        nonlocal eval_counter
        eval_counter += 1

        dots = out.get("dots", {})
        dot_str = " ".join([f"{k}:{dots.get(k, np.nan):+.3f}" for k in prep_defs.keys()])

        if clear_each_eval:
            clear_output(wait=True)

        fp = out["fit_params"]
        kb = out["knobs"]

        print(f"[{tag:<4} #{eval_counter:03d}] cost={out['cost']:.6e}  zeta={out['zeta']:+.5f}  | {dot_str}")
        print(
            "             fit_params="
            f"(delta_hat={fp['delta_hat']:+.5f}, amp_err_hat={fp['amp_err_hat']:+.3%}, "
            f"phase_err_hat={fp['phase_err_hat']:+.5f}, zeta={fp['zeta']:+.5f}){extra}"
        )
        print(
            "             knobs="
            f"(d_lambda={kb['d_lambda']:+.6e}, d_alpha={kb['d_alpha']:+.6f}, d_omega={kb['d_omega']:+.6e})  "
            f"T={kb['T']*1e9:.2f} ns"
        )

        if len(y) > 0:
            jbest = int(np.argmin(np.asarray(y, float)))
            best = hist[jbest]
            print(f"             BEST cost={best['cost']:.6e}  fit_params={best['fit_params']}")

    def eval_cost(x, *, tag):
        # cache
        key = tuple(np.round(np.asarray(x, float), 12))
        if key in cache:
            return cache[key]

        fp = x_to_dict(x)

        # convert (delta_hat, amp_err_hat, phase_err_hat) -> knobs
        knobs = fit_params_to_qubitrotation_knobs(
            amp_err_hat=fp["amp_err_hat"],
            phase_err_hat=fp["phase_err_hat"],
            delta_hat=fp["delta_hat"],
            dt_s=dt_s,
            n_samp=n_samp,
            d_omega_sign=float(d_omega_sign),
        )

        # Build pulse
        arb_rot = QubitRotation(
            theta, phi,
            d_lambda=knobs["d_lambda"],
            d_alpha=knobs["d_alpha"],
            d_omega=knobs["d_omega"],
            ref_r180_pulse=ref_r180_pulse,
            build=True,
            **build_kwargs,
        )
        experiment.burn_pulses()

        # Evaluate
        costs, zetas = [], []
        dots_last, meas_last = None, None

        for _ in range(int(repeats_per_eval)):
            meas_dirs = {}
            keys = list(prep_defs.keys())
            rng.shuffle(keys)
            for k in keys:
                prep_rot = prep_defs[k]
                rr = experiment.qubit_state_tomography(
                    make_state_prep(prep_rot, arb_rot, N=1),
                    int(n_avg),
                )
                meas_dirs[k] = extract_bloch_from_tomo(rr)

            c, zeta_hat, dots = direction_cost(
                meas_dirs, ideal_dirs,
                weights=weights,
                zeta_force=fp["zeta"],
            )
            costs.append(c)
            zetas.append(zeta_hat)
            dots_last = dots
            meas_last = meas_dirs

        out = dict(
            cost=float(np.mean(costs)),
            zeta=float(np.mean(zetas)),
            fit_params=dict(fp),
            knobs=dict(knobs),
            dots=dots_last,
            meas=meas_last,
        )
        cache[key] = out
        _print_eval(tag, out)
        return out

    def record(x, out, tag):
        X.append(np.asarray(x, float))
        y.append(float(out["cost"]))
        hist.append(dict(
            tag=str(tag),
            x=np.asarray(x, float).copy(),
            cost=float(out["cost"]),
            zeta=float(out["zeta"]),
            fit_params=out["fit_params"],
            knobs=out["knobs"],
            dots=out["dots"],
        ))

    # banner
    if verbose and not clear_each_eval:
        print("[BO] optimizing FIT params (including zeta)")
        for n in names:
            print(f"  {n:>12} : ({bounds_fit[n][0]}, {bounds_fit[n][1]})")
        print(f"  dt_s={dt_s:.3e} s, n_samp={n_samp}, T={T:.3e} s")

    # init
    for _ in range(int(n_init)):
        x0 = lo + (hi - lo) * rng.random(len(names))
        out = eval_cost(x0, tag="init")
        record(x0, out, tag="init")

    # BO iterations
    for _ in range(int(n_iter)):
        X_arr = np.vstack(X)
        y_arr = np.asarray(y, float)
        gp.fit(_scale01(X_arr, lo, hi), y_arr)

        x_next, ei_val = suggest_next_by_ei(
            gp, lo, hi,
            n_cand=candidate_pool,
            rng=rng,
            xi=xi,
        )

        out = eval_cost(x_next, tag="bo")
        record(x_next, out, tag="bo")

    # best
    costs_all = np.asarray(y, float)
    jbest = int(np.argmin(costs_all))
    best = hist[jbest]

    # IMPORTANT: do NOT clear after plotting (or you erase the figure)
    # We may clear ONCE before final plot if you used clear_each_eval
    if clear_each_eval:
        clear_output(wait=True)

    # plot history
    if plot_progress:
        costs = np.asarray([h["cost"] for h in hist], float)
        best_running = np.minimum.accumulate(costs)

        plt.figure(figsize=(9, 4.5))
        plt.plot(costs, marker="o", lw=1.5, label="cost")
        plt.plot(best_running, lw=2.0, label="running best")
        plt.xlabel("evaluation #")
        plt.ylabel("cost (sum_k 1 - dot)")
        plt.title("BO cost vs evaluation")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

    # final summary (NO clear_output here)
    print("\n[BO DONE]")
    print(f"  best cost       = {best['cost']:.6e}")
    print(f"  best zeta       = {best['zeta']:+.6f}")
    print(f"  best fit_params = {best['fit_params']}")
    print(
        "  best knobs      = "
        f"(d_lambda={best['knobs']['d_lambda']:+.6e}, "
        f"d_alpha={best['knobs']['d_alpha']:+.6f}, "
        f"d_omega={best['knobs']['d_omega']:+.6e})"
    )
    print(f"  best dots       = {best['dots']}")

    return dict(
        best_fit_params=best["fit_params"],
        best_knobs=dict(
            d_lambda=float(best["knobs"]["d_lambda"]),
            d_alpha=float(best["knobs"]["d_alpha"]),
            d_omega=float(best["knobs"]["d_omega"]),
        ),
        best_cost=float(best["cost"]),
        best_zeta=float(best["zeta"]),
        history=hist,
        names_fit=names,
        bounds_fit=bounds_fit,
        dt_s=dt_s,
        n_samp=n_samp,
        X=np.vstack(X),
        y=np.asarray(y, float),
    )


# ============================================================
# EXAMPLE USAGE
# ============================================================
theta = np.pi
phi = 0.0
n_avg = 30000

prep_defs = {
    "g":  None,
    "+x": QubitRotation(+np.pi/2, np.pi/2, build=True),
    "+y": QubitRotation(-np.pi/2, 0.0,    build=True),
}

bounds_fit = dict(
    delta_hat=(-0.20, +0.20),
    amp_err_hat=(-0.03, +0.03),
    phase_err_hat=(-0.50, +0.50),
    zeta=(-np.pi, +np.pi),
)

# If you don't know n_samp, supply a rotation instance that has .waveforms()
waveform_source_rot = QubitRotation(theta, phi, build=True)

result = bayes_optimize_qubitrotation_fitparams_with_zeta(
    experiment=experiment,
    QubitRotation=QubitRotation,
    theta=theta,
    phi=phi,
    prep_defs=prep_defs,
    n_avg=n_avg,
    bounds_fit=bounds_fit,
    waveform_source_rot=waveform_source_rot,   # OR just pass n_samp=...
    d_omega_sign=+1.0,
    ref_r180_pulse="sel_ref_r180_pulse",
    n_init=8,
    n_iter=200,
    repeats_per_eval=1,
    candidate_pool=8000,
    xi=0.02,
    random_seed=3,
    verbose=True,
    plot_progress=True,
    clear_each_eval=True,
)

print("BEST fit_params:", result["best_fit_params"])
print("BEST knobs:", result["best_knobs"], "cost=", result["best_cost"], "zeta=", result["best_zeta"])


# Readout Calibration and Benchmarking

### pulse registeration

In [ ]:
from qubox.tools.waveforms import build_CLEAR_waveform_from_physics

attr = experiment.attributes
kappa = 2 * np.pi * attr.ro_kappa   # example 5 MHz resonator linewidth
chi   = 2 * np.pi * attr.ro_chi     # example 1 MHz dispersive shift

A_steady   = 0.03
t_duration = 300
t_kick     = 40

clear_wf = build_CLEAR_waveform_from_physics(
    t_duration=t_duration,
    t_kick=t_kick,
    A_steady=A_steady,
    kappa_rad_s=kappa,
    chi_rad_s=chi,
)

ringdown_len = 100
total_len    = ringdown_len + len(clear_wf)

combined_clear_wf = np.zeros(total_len)
combined_clear_wf[0:len(clear_wf)] = clear_wf

clear_ro_op   = "readout"
clear_pulseOp = PulseOp(
    "resonator",
    clear_ro_op,
    clear_ro_op + "_pulse",
    type="measurement",
    length=total_len,
    I_wf=combined_clear_wf,
)
experiment.register_pulse(clear_pulseOp, override=True, persist=True, burn=True, save=False)
#run_gui(experiment.resonator_spectroscopy)
experiment.pulseOpMngr.display_op("resonator", clear_ro_op, freq_range=(-200, 200))

n_avg      = 25_000
freq_start = 8595 * u.MHz
freq_end   = 8597.5 * u.MHz
df         = 50 * u.kHz

result = experiment.resonator_spectroscopy(clear_ro_op, freq_start, freq_end, df, n_avg=n_avg)
if result.mode == "simulate":
    print("Simulation run, no data to analyze.")
else:
    frequencies, magnitude = result.output.extract("frequencies", "magnitude")
    p0 = [np.average([freq_start, freq_end])*1e-6, 1, -1e-4, 0]
    custom_plot_opts = {"figsize": (10, 6), "xlabel": "frequency (MHz)", "ylabel": "R", "title": "Resonator Spectroscopy Fit", "legend_fontsize": 16}
    fit_result = generalized_fit(frequencies*1e-6, magnitude, cQED_models.resonator_spec_model, p0, plotting=True, plot_options=custom_plot_opts, param_format="{:.4f}")


In [ ]:
experiment.pulseOpMngr.print_state()

### square wf readout

In [ ]:
n_avg      = 1_000
freq_start = 8590 * u.MHz
freq_end   = 8600 * u.MHz
df         = 100 * u.kHz

ro_gain    = 1
readout_op = "readout"
ro_pulse   = readout_op + "_pulse"   # "readout_pulse"

pulse_len    = 256
ringdown_len = 144
total_len    = pulse_len + ringdown_len

# Build desired I/Q shapes (drive on for pulse_len, then off)
I_wf = np.zeros(total_len)
amp  = 0.030
I_wf[:pulse_len] = amp

pom = experiment.pulseOpMngr

# 1) Update I waveform (arbitrary), keep Q as constant 0
pom.modify_waveform("readout_I_wf", I_wf, persist=True, allow_type_change=True)
pom.modify_pulse_op(
    PulseOp(
        element="resonator",
        op="readout",
        pulse="readout_pulse",
        type="measurement",
        length=total_len,
        I_wf_name="readout_I_wf",
        Q_wf_name="readout_Q_wf",
        I_wf=None,
        Q_wf=None,
        int_weights_mapping=None,
        int_weights_defs=None,
    ),
    persist=True,
)

experiment.save_pulses()
experiment.burn_pulses()
# --- run the spectroscopy as before --------------------------------
result = experiment.resonator_spectroscopy(
    readout_op, freq_start, freq_end, df, n_avg=n_avg
)

if result.mode == "simulate":
    print("Simulation run, no data to analyze.")
else:
    frequencies, magnitude = result.output.extract("frequencies", "magnitude")
    p0 = [
        np.average([freq_start, freq_end]) * 1e-6,
        1,
        -1e-4,
        0,
    ]
    custom_plot_opts = {
        "figsize": (10, 6),
        "xlabel": "frequency (MHz)",
        "ylabel": "R",
        "title": "Resonator Spectroscopy Fit",
        "legend_fontsize": 16,
    }
    fit_result = fitting.generalized_fit(
        frequencies * 1e-6,
        magnitude,
        cQED_models.resonator_spec_model,
        p0,
        plotting=True,
        plot_options=custom_plot_opts,
        param_format="{:.4f}",
    )


In [ ]:
experiment.pulseOpMngr.get_pulseOp_by_element_op("resonator", readout_op)

### clear wf readout

In [ ]:
from qubox.tools.waveforms import build_CLEAR_waveform_from_physics, CLEAR_waveform

attr   = experiment.attributes
kappa  = 2 * np.pi * attr.ro_kappa   # resonator linewidth [rad/s]
chi    = 2 * np.pi * attr.ro_chi     # dispersive shift [rad/s]

A_steady   = 0.14
t_duration = 228+20
t_kick     = 22+20
A_rise_hi, A_rise_lo = 0.0493, -0.0496
A_fall_lo, A_fall_hi = 0.0294, 0.2
clear_wf = CLEAR_waveform(t_duration, t_kick, A_steady, A_rise_hi, A_rise_lo, A_fall_lo, A_fall_hi)

ringdown_len = 0
total_len    = ringdown_len + len(clear_wf)

combined_clear_wf = np.zeros(total_len)
combined_clear_wf[0:len(clear_wf)] = clear_wf

clear_ro_op = "readout"      # still use canonical readout op
pom         = experiment.pulseOpMngr

# 1) Update contents of the reserved I waveform.
#    Q can remain a constant 0.0 waveform.
pom.modify_waveform("readout_I_wf", combined_clear_wf, persist=True)
# If you really want to be explicit about Q staying zero:
# pom.modify_waveform("readout_Q_wf", 0.0, persist=True)

# 2) Patch the canonical readout pulse to have the new total length.
#    IMPORTANT: we supply only names, NOT samples, so we don't
#    trigger add_waveform(...) on reserved IDs.
clear_pulse_patch = PulseOp(
    element="resonator",
    op=clear_ro_op,
    pulse=clear_ro_op + "_pulse",   # "readout_pulse"
    type="measurement",
    length=total_len,
    digital_marker=None,            # keep existing marker
    I_wf_name="readout_I_wf",
    Q_wf_name="readout_Q_wf",
    I_wf=None,
    Q_wf=None,
    int_weights_mapping=None,       # keep existing mapping
    int_weights_defs=None,
)

pom.modify_pulse_op(clear_pulse_patch, persist=True)

# Optional visualization of the new CLEAR readout pulse
experiment.pulseOpMngr.display_op("resonator", clear_ro_op, freq_range=(-200, 200))

# ─── run spectroscopy as before ─────────────────────────────────────
n_avg      = 25_000
freq_start = 8595 * u.MHz
freq_end   = 8597.5 * u.MHz
df         = 50 * u.kHz

result = experiment.resonator_spectroscopy(clear_ro_op, freq_start, freq_end, df, n_avg=n_avg)
if result.mode == "simulate":
    print("Simulation run, no data to analyze.")
else:
    frequencies, magnitude = result.output.extract("frequencies", "magnitude")
    p0 = [np.average([freq_start, freq_end]) * 1e-6, 1, -1e-4, 0]
    custom_plot_opts = {
        "figsize": (10, 6),
        "xlabel": "frequency (MHz)",
        "ylabel": "R",
        "title": "Resonator Spectroscopy Fit",
        "legend_fontsize": 16,
    }
    fit_result = fitting.generalized_fit(
        frequencies * 1e-6,
        magnitude,
        cQED_models.resonator_spec_model,
        p0,
        plotting=True,
        plot_options=custom_plot_opts,
        param_format="{:.4f}",
    )


## Readout benchmarking

### readout ge raw trace 

In [ ]:
ro_gain = 1
ro_depl_clks = 10000
ro_fq = experiment.attributes.ro_fq
ro_detuning = 0
output = experiment.readout_ge_raw_trace(ro_fq+ro_detuning, r180="x180", ro_depl_clks=ro_depl_clks, n_avg=10000)
adc1_g, adc2_g, adc1_e, adc2_e = output.output.extract("adc1_g", "adc2_g", "adc1_e", "adc2_e")
# Create figure with 2 subplots
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# --- Ground state traces ---
axes[0].plot(adc1_g, label="adc1_g (I)")
axes[0].plot(adc2_g, label="adc2_g (Q)")
axes[0].set_title(
    f"Ground | ro_fq={ro_fq*1e-6:.3f} MHz, ro_detuning={ro_detuning*1e-6:.3f} MHz,\n"
    f"ro_gain={ro_gain}, ro_depl_clks={ro_depl_clks}"
)
axes[0].legend()
axes[0].grid(True)

# --- Excited state traces ---
axes[1].plot(adc1_e, label="adc1_e (I)")
axes[1].plot(adc2_e, label="adc2_e (Q)")
axes[1].set_title("Excited")
axes[1].legend()
axes[1].grid(True)

# Shared x-axis label
axes[1].set_xlabel("Sample index")

plt.tight_layout()

plt.show()
plt.plot(adc1_e-adc1_g, label="adc diff (I)")
plt.plot(adc2_e-adc2_g, label="adc diff (Q)")
plt.legend()

### readout ge discrimination

In [ ]:
experiment.attributes.ro_chi

In [ ]:
experiment.set_readout_SPA_pump_power(11)

In [ ]:
r180       = "x180"
n_samples  = 25_000
readout_op = "readout"
detuning = experiment.attributes.ro_chi
drive_frequency = experiment.attributes.ro_fq + detuning
res = experiment.readout_ge_discrimination(
    readout_op, drive_frequency, gain=1, update_measureMacro=True, base_weight_keys=("cos", "sin", "minus_sin"), n_samples=n_samples, persist=True, 
    b_plot=True, plots=("raw_blob", "rot_blob", "hist","info")
)
experiment.save_pulses()
# Handle RunResult vs Output

out = res.output if hasattr(res, "mode") else res
if hasattr(res, "mode") and res.mode == "simulate":
    print("Simulation run — no hardware streams; skipping discriminator fit summary.")
else:
    pass

In [ ]:
measureMacro._ro_disc_params

In [ ]:
res.output.keys()

In [ ]:
experiment.pulseOpMngr.get_pulseOp_by_element_op("resonator", "readout")

### readout ge integrated trace

In [ ]:
experiment.attributes.ro_fq += experiment.attributes.ro_chi

In [ ]:
ro_op = "readout"

pulseOp = experiment.pulseOpMngr.get_pulseOp_by_element_op("resonator", ro_op)

num_div = 75

cos_w_key, sin_w_key, m_sin_w_key = "cos", "sin", "minus_sin"
weights = [cos_w_key, sin_w_key, m_sin_w_key, cos_w_key]

runres = experiment.readout_ge_integrated_trace(ro_op, weights, num_div, n_avg=20000)
time_list = runres.output.extract("time_list")

g_trace, e_trace = runres.output.extract("g_trace", "e_trace")

g = np.asarray(g_trace)
e = np.asarray(e_trace)

n_arrows = 10

# --- Make a single figure with 2 subplots ------------------------------------
fig, (ax_time, ax_phase) = plt.subplots(1, 2, figsize=(10, 4))

# ===== Subplot 1: |S|(t) ======================================================
ax_time.plot(time_list, np.abs(g), 'o-', label='Ground')
ax_time.plot(time_list, np.abs(e), 's-', label='Excited')

ax_time.set_xlabel('Time (ns)')
ax_time.set_ylabel(r'$|S|$ (arb)')
ax_time.set_title('Integrated readout GE trace')
ax_time.legend()

# ===== Subplot 2: phase-space trajectory =====================================
ax_phase.plot(g.real, g.imag, linestyle="--", linewidth=2, color="tab:red",
              label=r"$|g\rangle$")
ax_phase.plot(e.real, e.imag, linestyle="-",  linewidth=2, color="darkred",
              label=r"$|e\rangle$")

# starting points
ax_phase.plot(g.real[0], g.imag[0], "o", color="tab:red")
ax_phase.plot(e.real[0], e.imag[0], "o", color="darkred")

def add_arrows(traj, color, ax):
    idxs = np.linspace(0, len(traj) - 2, n_arrows, dtype=int)
    for i in idxs:
        x0, y0 = traj.real[i],   traj.imag[i]
        x1, y1 = traj.real[i+1], traj.imag[i+1]
        ax.annotate(
            "", xy=(x1, y1), xytext=(x0, y0),
            arrowprops=dict(arrowstyle="->", color=color, lw=1.2)
        )

add_arrows(g, "tab:red", ax_phase)
add_arrows(e, "darkred", ax_phase)

ax_phase.set_xlabel(r"Re$(\alpha)$")
ax_phase.set_ylabel(r"Im$(\alpha)$")
ax_phase.set_aspect("equal", "box")
ax_phase.grid(True, alpha=0.3)
ax_phase.legend(loc="best")
ax_phase.set_title('Phase-space trajectory')

plt.tight_layout()
plt.show()



### readout loss (kappa)

In [ ]:
ro_op = "readout_long"
ro_pulse = ro_op + "_pulse"
I_wf_name = ro_op + "_I_wf"
Q_wf_name = ro_op + "_Q_wf"
pulse_len = 300           # ns
ringdown_len = 600        # ns
readout_len = pulse_len + ringdown_len

I_wf = np.zeros(readout_len)
Q_wf = np.zeros(readout_len)
amp = 0.05
I_wf[0:pulse_len] = amp

pulseOp = PulseOp(
    experiment.attributes.ro_el,
    ro_op,
    ro_pulse,
    type="measurement",
    length=readout_len,
    I_wf_name=I_wf_name,
    Q_wf_name=Q_wf_name,
    I_wf=I_wf.tolist(),
    Q_wf=Q_wf.tolist(),
)
experiment.register_pulse(pulseOp, override=True, persist=False, burn=True, save=False)

n_avg      = 5_000
freq_start = 8594 * u.MHz
freq_end   = 8600 * u.MHz
df         = 50 * u.kHz

result = experiment.resonator_spectroscopy(ro_op, freq_start, freq_end, df, n_avg=n_avg)
if result.mode == "simulate":
    print("Simulation run, no data to analyze.")
else:
    frequencies, magnitude = result.output.extract("frequencies", "magnitude")
    p0 = [np.average([freq_start, freq_end])*1e-6, 1, -1e-4, 0]
    custom_plot_opts = {"figsize": (10, 6), "xlabel": "frequency (MHz)", "ylabel": "R", "title": "Resonator Spectroscopy Fit", "legend_fontsize": 16}
    fit_result = fitting.generalized_fit(frequencies*1e-6, magnitude, cQED_models.resonator_spec_model, p0, plotting=True, plot_options=custom_plot_opts, param_format="{:.4f}")



In [ ]:

pulseOp = experiment.pulseOpMngr.get_pulseOp_by_element_op("resonator", ro_op)

num_div = 225

cos_w_key, sin_w_key, m_sin_w_key = "cos", "sin", "minus_sin"
weights = [cos_w_key, sin_w_key, m_sin_w_key, cos_w_key]

runres = experiment.readout_ge_integrated_trace(ro_op, weights, num_div, n_avg=100000)
time_list = runres.output.extract("time_list")

g_trace, e_trace = runres.output.extract("g_trace", "e_trace")




In [ ]:
# Treat time_list as ns for this analysis
t_ns = time_list

# Magnitudes (field amplitude)
g_abs = np.abs(g_trace)
e_abs = np.abs(e_trace)

# --- Ring-down window: from end of drive to end of readout --------------------

mask = (t_ns >= pulse_len+16) & (t_ns <= readout_len)
t_ring = t_ns[mask]

if t_ring.size < 3:
    raise ValueError("Not enough points in ring-down region for a reliable fit.")

# Fix t0 to the first ring-down sample
t0_fixed = float(t_ring[0])

def exp_model_fixed_t0(t, A, tau, offset):
    """
    Exponential decay with fixed t0:
        y(t) = offset + A * exp(-(t - t0_fixed)/tau)
    t, tau in ns.
    """
    return models.exponential_model(t, A, tau, t0_fixed, offset)

def fit_mag_trace(y_mag, label):
    """
    Fit magnitude |trace| in ring-down region.

    Model:
        |α(t)| = offset + A * exp(-(t - t0)/tau)

    For field amplitude decay:
        α ~ exp(-κ t / 2)
        ⇒ κ = 2 / tau[s]
        ⇒ κ/2π [MHz] = 1000 / (π * tau_ns)
    """
    y_ring = np.asarray(y_mag[mask], float)

    # Initial guesses
    offset0 = float(y_ring[-1])
    A0 = float(max(y_ring[0] - offset0, 1e-9))
    tau0_ns = max((t_ring[-1] - t_ring[0]) / 2.0, 1.0)

    p0 = (A0, tau0_ns, offset0)

    try:
        popt, pcov = generalized_fit(
            t_ring,
            y_ring,
            exp_model_fixed_t0,
            p0,
            plotting=False,
        )
        if popt is None:
            raise RuntimeError("Fit failed in generalized_fit.")

        A, tau_ns, offset = popt

        # κ/2π in MHz
        kappa_over_2pi_MHz = 1000.0 / (np.pi * tau_ns)

        print(
            f"{label:10s}: "
            f"A = {A:+.4e}, "
            f"tau = {tau_ns:.3f} ns, "
            f"offset = {offset:+.4e}, "
            f"κ/2π = {kappa_over_2pi_MHz:.3f} MHz"
        )

        return A, tau_ns, offset, kappa_over_2pi_MHz

    except Exception as err:
        print(f"{label:10s}: FIT FAILED → {err}")
        return None, None, None, None

# --- Fit |g_trace| and |e_trace| ---------------------------------------------

A_g, tau_g_ns, off_g, kappa_g_MHz = fit_mag_trace(g_abs, "g_abs")
A_e, tau_e_ns, off_e, kappa_e_MHz = fit_mag_trace(e_abs, "e_abs")

# --- Plot magnitudes + fits ---------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 6))

# Data
ax.scatter(t_ring, g_abs[mask], s=14, marker="o", label="|g_trace| data")
ax.scatter(t_ring, e_abs[mask], s=14, marker="s", label="|e_trace| data")

# Fits
if A_g is not None:
    g_fit = exp_model_fixed_t0(t_ring, A_g, tau_g_ns, off_g)
    ax.plot(
        t_ring,
        g_fit,
        label=f"|g_trace| fit (κ/2π={kappa_g_MHz:.3f} MHz)",
    )

if A_e is not None:
    e_fit = exp_model_fixed_t0(t_ring, A_e, tau_e_ns, off_e)
    ax.plot(
        t_ring,
        e_fit,
        label=f"|e_trace| fit (κ/2π={kappa_e_MHz:.3f} MHz)",
    )

ax.set_title("Readout Ring-down: |g| and |e| Magnitudes with Exponential Fits")
ax.set_xlabel("Time [ns]")
ax.set_ylabel("Magnitude [a.u.]")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### residual photon ramsey

In [ ]:
test_ro_op = "readout_test"
ro_pulse = test_ro_op + "_pulse"
I_wf_name = test_ro_op + "_I_wf"
Q_wf_name = test_ro_op + "_Q_wf"
amp = 0.02
pulseOp = PulseOp( "resonator", ro_pulse, test_ro_op, op_type="measurement", length=400, I_wf_name=I_wf_name, Q_wf_name=Q_wf_name,I_wf=amp,Q_wf=0)
experiment.register_pulse(pulseOp, override=True, persist=False, burn=False, save=False)

measure_ro_op = "readout_long"
ro_pulse = measure_ro_op + "_pulse"
I_wf_name = measure_ro_op + "_I_wf"
Q_wf_name = measure_ro_op + "_Q_wf"
amp = 0.025
pulseOp = PulseOp( "resonator", ro_pulse, measure_ro_op, op_type="measurement", length=1000, I_wf_name=I_wf_name, Q_wf_name=Q_wf_name,I_wf=amp,Q_wf=0)
experiment.register_pulse(pulseOp, override=True, persist=False, burn=True, save=False)

n_avg      = 5_000
freq_start = 8596 * u.MHz
freq_end   = 8598.5 * u.MHz
df         = 50 * u.kHz


result = experiment.resonator_spectroscopy(test_ro_op, freq_start, freq_end, df, n_avg=n_avg)
frequencies, magnitude = result.output.extract("frequencies", "magnitude")
p0 = [np.average([freq_start, freq_end])*1e-6, 1, -1e-4, 0]
custom_plot_opts = {"figsize": (10, 6), "xlabel": "frequency (MHz)", "ylabel": "R", "title": "Resonator Spectroscopy Fit", "legend_fontsize": 16}
fit_result = generalized_fit(frequencies*1e-6, magnitude, cQED_models.resonator_spec_model, p0, plotting=True, plot_options=custom_plot_opts, param_format="{:.4f}")

result = experiment.resonator_spectroscopy(measure_ro_op, freq_start, freq_end, df, n_avg=n_avg)
frequencies, magnitude = result.output.extract("frequencies", "magnitude")
p0 = [np.average([freq_start, freq_end])*1e-6, 1, -1e-4, 0]
custom_plot_opts = {"figsize": (10, 6), "xlabel": "frequency (MHz)", "ylabel": "R", "title": "Resonator Spectroscopy Fit", "legend_fontsize": 16}
fit_result = generalized_fit(frequencies*1e-6, magnitude, cQED_models.resonator_spec_model, p0, plotting=True, plot_options=custom_plot_opts, param_format="{:.4f}")



In [ ]:

ro_detuning = experiment.attributes.ro_chi
r180 = "x180"
n_samples = 100000
disc_res = experiment.readout_ge_discrimination(
    measure_ro_op, r180, set_measureMacro=True, base_weight_keys=("cos", "sin", "minus_sin"), ro_detuning=ro_detuning, n_samples=n_samples, persist=False, 
    b_plot=True, plots=("hist","rot_blob", "info")
)

In [ ]:

#pulseOp = experiment.pulseOpMngr.get_pulseOp_by_element_op(attr.ro_el, measure_ro_op)
#measureMacro.set_pulse_op(pulseOp, active_op=measure_ro_op)

t_R_begin = 4 * u.ns
t_R_end = 800 * u.ns
dt = 4 * u.ns
t_relax = 16 * u.ns
t_buffer = 480 * u.ns
n_avg = 250_000
ro_detuning = experiment.attributes.ro_chi
qb_detuning = 10e6

experiment.save_unique_identifier = "zero_photon_reference"
rr_ref = experiment.residual_photon_ramsey(t_R_begin, t_R_end, dt, test_ro_op, 
                                    ro_detuning=ro_detuning, qb_detuning=qb_detuning, 
                                    t_relax=t_relax, t_buffer=t_buffer, 
                                    test_ro_amp=0,
                                        measure_ro_op=measure_ro_op, n_avg=n_avg)



In [ ]:
S, states, t_R = rr_ref.output.extract("S", "S", "t_R")

def S_model(t_R_s, Delta, phi0, n0, Gamma2, chi, kappa):
    """
    t_R_s in seconds.
    S(t_R) = 1/2 [ 1 - Im{ exp[ -(Gamma2 + i*Delta) t_R
                            + i (phi0 - 2 n0 chi tau) ] } ]

    tau(t_R) = (1 - exp(-(kappa + 2 i chi) t_R)) / (kappa + 2 i chi)
    """
    t_R_s = np.asarray(t_R_s, dtype=float)
    i = 1j

    tau = (1 - np.exp(-(kappa + 2*i*chi) * t_R_s)) / (kappa + 2*i*chi)
    exponent = -(Gamma2 + i*Delta) * t_R_s + 1j * (phi0 - 2*n0*chi*tau)
    return 0.5 * (1 - np.imag(np.exp(exponent)))

Gamma2_0 = 1 / (experiment.attributes.qb_T2_echo*1e-9)    
kappa_0  = 2 * np.pi * experiment.attributes.ro_kappa     
Delta    = 2 * np.pi * qb_detuning                 
chi_0    = 2 * np.pi * experiment.attributes.ro_chi  
def S_model_wrapper(
    t_R,
    n0,
    phi0,
    A,
    B 
):
    n0 = 0
    S_th = S_model(t_R, Delta, phi0, n0, Gamma2_0, chi_0, kappa_0)
    return A * S_th + B

Pe = sigma_z_to_probs(states.real)


y_data = Pe
p0 = [0, 0, y_data.real.ptp(), y_data.min()]
bounds = ([0, -2*np.pi, -2*y_data.real.ptp(), -5], [10, 2*np.pi, 2*y_data.real.ptp(), 5])
custom_plot_opts = {"figsize": (8, 4), "xlabel": "ramsey delay (s)", "ylabel": "amplitude", "title": f"residual photon ground state (g) t_relax={t_relax} ns", "legend_fontsize": 16}
g_fit = generalized_fit(t_R*1e-9, y_data, S_model_wrapper, p0=p0, bounds=bounds, plotting=True, plot_options=custom_plot_opts)


fit_params  = g_fit[0]
_, _, A_ref, B_ref = fit_params


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qubox.analysis.analysis_tools import sigma_z_to_probs

t_relax_list = np.arange(16, 400, 16)

n0_list = []

for t_relax in t_relax_list:
    n_avg = 40000
    rr = experiment.residual_photon_ramsey(t_R_begin, t_R_end, dt, test_ro_op, 
                                        ro_detuning=ro_detuning, qb_detuning=qb_detuning, 
                                        t_relax=t_relax, t_buffer=t_buffer, 
                                            measure_ro_op=measure_ro_op, n_avg=n_avg)


    S, states, t_R = rr.output.extract("S", "S", "t_R")

    Gamma2_0 = 1 / (experiment.attributes.qb_T2_echo*1e-9)    
    kappa_0  = 2 * np.pi * experiment.attributes.ro_kappa     
    Delta    = 2 * np.pi * qb_detuning                 
    chi_0    = 2 * np.pi * experiment.attributes.ro_chi  
    def S_model_wrapper(
        t_R,
        n0,
        phi0,
        A,
        B 
    ):
        A = A_ref
        B = B_ref
        S_th = S_model(t_R, Delta, phi0, n0, Gamma2_0, chi_0, kappa_0)
        return A * S_th + B

    Pe = sigma_z_to_probs(states.real)


    y_data = Pe
    p0 = [0, 0, y_data.real.ptp(), y_data.min()]
    bounds = ([0, -2*np.pi, -2*y_data.real.ptp(), -5], [10, 2*np.pi, 2*y_data.real.ptp(), 5])
    custom_plot_opts = {"figsize": (8, 4), "xlabel": "ramsey delay (s)", "ylabel": "amplitude", "title": f"residual photon ground state (g) t_relax={t_relax} ns", "legend_fontsize": 16}
    g_fit = generalized_fit(t_R*1e-9, y_data, S_model_wrapper, p0=p0, bounds=bounds, plotting=True, plot_options=custom_plot_opts)


    fit_params  = g_fit[0]
    n0_fit, _, A_ref, B_ref = fit_params
    n0_list.append(n0_fit)


In [ ]:
generalized_fit(
    t_relax_list,
    n0_list,
    models.exponential_model,
    p0=[max(n0_list), 100, 0, 0],
    plotting=True,
    plot_options={"figsize": (8, 4), "xlabel": "t_relax (ns)", "ylabel": "n0", "title": "residual photon n0 vs t_relax", "legend_fontsize": 16}
)

In [ ]:

t_relax_list = np.arange(16, 200, 4)

n0_list = []

for t_relax in t_relax_list:
    n_avg = 40000
    rr = experiment.residual_photon_ramsey(t_R_begin, t_R_end, dt, test_ro_op, 
                                        ro_detuning=ro_detuning, qb_detuning=qb_detuning, 
                                        t_relax=t_relax, t_buffer=t_buffer, 
                                            measure_ro_op=measure_ro_op, n_avg=n_avg)


    S, states, t_R = rr.output.extract("S", "S", "t_R")

    Gamma2_0 = 1 / (experiment.attributes.qb_T2_echo*1e-9)    
    kappa_0  = 2 * np.pi * experiment.attributes.ro_kappa     
    Delta    = 2 * np.pi * qb_detuning                 
    chi_0    = 2 * np.pi * experiment.attributes.ro_chi  
    def S_model_wrapper(
        t_R,
        n0,
        phi0,
        A,
        B 
    ):
        A = A_ref
        B = B_ref
        S_th = S_model(t_R, Delta, phi0, n0, Gamma2_0, chi_0, kappa_0)
        return A * S_th + B

    Pe = sigma_z_to_probs(states.real)


    y_data = Pe
    p0 = [0, 0, y_data.real.ptp(), y_data.min()]
    bounds = ([0, -2*np.pi, -2*y_data.real.ptp(), -5], [10, 2*np.pi, 2*y_data.real.ptp(), 5])
    custom_plot_opts = {"figsize": (8, 4), "xlabel": "ramsey delay (s)", "ylabel": "amplitude", "title": f"residual photon ground state (g) t_relax={t_relax} ns", "legend_fontsize": 16}
    g_fit = generalized_fit(t_R*1e-9, y_data, S_model_wrapper, p0=p0, bounds=bounds, plotting=True, plot_options=custom_plot_opts)


    fit_params  = g_fit[0]
    n0_fit, _, A_ref, B_ref = fit_params
    n0_list.append(n0_fit)


In [ ]:
generalized_fit(
    t_relax_list,
    n0_list,
    models.exponential_model,
    p0=[max(n0_list), 100, 0, 0],
    plotting=True,
    plot_options={"figsize": (8, 4), "xlabel": "t_relax (ns)", "ylabel": "n0", "title": "residual photon n0 vs t_relax", "legend_fontsize": 16}
)

### butterfly measurement

In [ ]:
experiment.set_readout_SPA_pump_power(9.5)
#experiment.set_readout_SPA_pump_frequency(17172097706.157593)

In [ ]:

r180       = "x180"
n_samples  = 20_000
readout_op = "readout"
detuning = experiment.attributes.ro_chi
drive_frequency = experiment.attributes.ro_fq + detuning
res = experiment.readout_ge_discrimination(
    readout_op, drive_frequency, gain=1, update_measureMacro=True, base_weight_keys=("cos", "sin", "minus_sin"), n_samples=n_samples, persist=True, 
    b_plot=True, plots=("raw_blob", "rot_blob", "hist","info"), blob_k_g=3.5
)

""" 

rot_mu_g, rot_mu_e, threshold = res.o
utput.extract("rot_mu_g", "rot_mu_e", "threshold")
sigma_g, sigma_e = res.output.extract("sigma_g", "sigma_e")
k_g = k_e = 3
n_samples = 20000
prep_kwargs = {
    "Ig": rot_mu_g.real,
    "Qg": rot_mu_g.imag,
    "rg2": (k_g*sigma_g)**2,
    "Ie": rot_mu_e.real,
    "Qe": rot_mu_e.imag,
    "re2": (k_e*sigma_e)**2,
    "threshold": threshold
}
"""

runres = experiment.readout_butterfly_measurement(
    prep_policy=None,
    prep_kwargs=None,
    show_analysis=True,
    update_measureMacro=True,
    n_samples=n_samples,
    M0_MAX_TRIALS=500
)
F, Q = runres.output.extract("F", "Q")

In [ ]:
accept_rate, average_tries = runres.output.extract("acceptance_rate", "average_tries")
print("accept rate:", accept_rate)
print("average tries:", average_tries)

### active qubit reset benchmarking

In [ ]:
prep_policy = "BLOBS"
rot_mu_g = measureMacro._ro_disc_params["rot_mu_g"]
rot_mu_e = measureMacro._ro_disc_params["rot_mu_e"]
threshold = measureMacro._ro_disc_params["threshold"]
sigma_g = measureMacro._ro_disc_params["sigma_g"]
sigma_e = measureMacro._ro_disc_params["sigma_e"]
k_g = k_e = 3
n_samples = 20000


prep_kwargs = {
    "Ig": rot_mu_g.real,
    "Qg": rot_mu_g.imag,
    "rg2": (k_g*sigma_g)**2,
    "Ie": rot_mu_e.real,
    "Qe": rot_mu_e.imag,
    "re2": (k_e*sigma_e)**2,
    "threshold": threshold
}


MAX_PREP_TRIALS = 256
n_shots = 50000
show_analysis = True
runres = experiment.active_qubit_reset_benchmark(prep_policy, prep_kwargs, show_analysis, MAX_PREP_TRIALS, n_shots)
import numpy as np
import matplotlib.pyplot as plt

# You already have:
S_0, S_1, S_2 = runres.output.extract("S_0", "S_1", "S_2")

S_0 = np.asarray(S_0)
S_1 = np.asarray(S_1)
S_2 = np.asarray(S_2)

S_0g, S_0e = S_0[:, 0], S_0[:, 1]
S_1g, S_1e = S_1[:, 0], S_1[:, 1]
S_2g, S_2e = S_2[:, 0], S_2[:, 1]

def scatter_iq(ax, Sg, Se, title):
    ax.scatter(np.real(Sg), np.imag(Sg), s=1, alpha=0.25, label="prep g branch")
    ax.scatter(np.real(Se), np.imag(Se), s=1, alpha=0.25, label="prep e branch")
    ax.set_title(title)
    ax.set_xlabel("I")
    ax.set_ylabel("Q")
    ax.axis("equal")
    ax.grid(True, alpha=0.2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=200, constrained_layout=True)

scatter_iq(axes[0], S_0g, S_0e, "S₀ (M0): prep check")
scatter_iq(axes[1], S_1g, S_1e, "S₁ (M1): before conditional π")
scatter_iq(axes[2], S_2g, S_2e, "S₂ (M2): verification")

axes[2].legend(loc="best", markerscale=3, frameon=True)
plt.show()
bench = runres.output.extract("reset_benchmark")
R_reset = bench["matrices"]["R_sim"]["matrix"]
print("R_sim (trigger-based) =\n", R_reset)





In [ ]:
m2, m2c = runres.output.extract("m2","m2_corrected")
m2_avg = (m2*1).mean(axis=0)
m2c_avg = (m2c*1).mean(axis=0)
print("M2 average (uncorrected):", m2_avg)
print("M2 average (corrected):", m2c_avg)

In [ ]:
pp.ro_state_correct_proc()

In [ ]:
thr = measureMacro._ro_disc_params["threshold"]
S1 = runres.output.extract("S_1")  # or real(S_1)
I1 = S1.real
m1 = runres.output.extract("m1")

print(np.mean(m1[:,0] == (I1[:,0] > thr)))
print(np.mean(m1[:,1] == (I1[:,1] > thr)))


In [ ]:
experiment.quaProgMngr.serialize_program()

In [ ]:
bench = runres.output.extract("reset_benchmark")
R_reset_sim = bench["matrices"]["R_sim"]["matrix"]   # columns: rep0, rep1
R_reset_sim

In [ ]:
experiment.quaProgMngr.serialize_program()

## Readout optimization

### ampltiude, length optimization

In [ ]:
experiment.set_readout_SPA_pump_power(9.5)

In [ ]:
r180 = "x180"
base_voltage = 0.05
min_g, max_g, dg = 0.5, 1.8, 0.1
min_len, max_len, dlen = 100, 400, 16

drive_frequency = experiment.attributes.ro_fq
out = experiment.readout_amp_len_opt(drive_frequency,
    min_len, max_len, dlen, min_g, max_g, dg, base_voltage=base_voltage,
    ringdown_len=200,
    ge_disc_kwargs={"n_samples": 10000, "k_g": 3.0, "k_e": 3.0, "b_plot": False},
    butterfly_kwargs={"n_shots": 25000, "M0_MAX_TRIALS": 400, "show_analysis": False},
)


In [ ]:
fidelity_matrix, QND_matrix = out.extract("fidelity_matrix", "QND_matrix")

# Clean NaNs
fidelity_matrix = np.nan_to_num(fidelity_matrix, nan=0.0)
QND_matrix     = np.nan_to_num(QND_matrix,     nan=0.0)

# Axes
ro_lens, amplitudes = out.extract("ro_lens", "amplitudes")   # ro_lens: body length (ns)
try:
    total_lens_ns = out["ro_total_lens_ns"]                  # drive + ringdown (ns)
except KeyError:
    total_lens_ns = ro_lens                                  # fallback: just use body length

# --- Plots for F and QND (unchanged) ---
plotting.plot_hm(
    fidelity_matrix, ro_lens, amplitudes * 1e3,
    ylabel="amp (mV)", xlabel="ro_len (ns)",
    title="Fidelity", barlabel="F"
)
plotting.plot_hm(
    QND_matrix, ro_lens, amplitudes * 1e3,
    ylabel="amp (mV)", xlabel="ro_len (ns)",
    title="QND", barlabel="QND"
)

# --- Overall scalar metric: emphasize QND over F ---

# Weights for the scalar score: score = w_Q * Q + w_F * F
# (w_Q > w_F gives more weight to QND.)
w_Q = 0.8
w_F = 0.2

score_matrix = w_Q * QND_matrix + w_F * fidelity_matrix

plotting.plot_hm(
    score_matrix, ro_lens, amplitudes * 1e3,
    ylabel="amp (mV)", xlabel="ro_len (ns)",
    title=fr"$w_Q Q + w_F F$  (w_Q={w_Q}, w_F={w_F})",
    barlabel="score"
)

gains = out["ro_gains"]  # 1D array, shape (n_gain,)

# 1) Global max of the scalar metric
S_max = np.nanmax(score_matrix)

# 2) Allow a small tolerance around the max (to avoid picking a noisy outlier)
dS = 0.01  # absolute tolerance on score; tune as you like
mask_S = score_matrix >= (S_max - dS)

idx_g, idx_L = np.where(mask_S)

if len(idx_g) == 0:
    # Very defensive fallback: use global argmax directly
    flat_idx = np.nanargmax(score_matrix)
    amp_index, ro_index = np.unravel_index(flat_idx, score_matrix.shape)
else:
    # Among near-max points, choose:
    #   1) larger QND
    #   2) if tie, larger F
    #   3) if tie, shorter total length
    def tie_break(ij):
        i, j = ij
        return (
            -QND_matrix[i, j],               # max QND
            -fidelity_matrix[i, j],          # then max F
            total_lens_ns[j],                # then min total length
        )

    amp_index, ro_index = min(zip(idx_g, idx_L), key=tie_break)

# Extract optimal point
opt_len_body  = ro_lens[ro_index]
opt_len_total = total_lens_ns[ro_index]
opt_amp       = amplitudes[amp_index]
opt_F         = fidelity_matrix[amp_index, ro_index]
opt_Q         = QND_matrix[amp_index, ro_index]
opt_gain      = gains[amp_index]
opt_S         = score_matrix[amp_index, ro_index]

print(
    "Optimal point (QND-prioritized metric with F as secondary):\n"
    f"  gain         = {opt_gain:.3f}\n"
    f"  body length  = {opt_len_body} ns\n"
    f"  total length = {opt_len_total} ns (incl. ringdown)\n"
    f"  amplitude    = {opt_amp:.4f} V\n"
    f"  F            = {opt_F:.4f}\n"
    f"  QND          = {opt_Q:.4f}\n"
    f"  score        = w_Q*Q + w_F*F = {opt_S:.4f}"
)



### frequency optimization

In [ ]:
attr = experiment.attributes
rf_begin = attr.ro_fq + attr.ro_chi - 1e6
rf_end = attr.ro_fq + 1e6
df = 100 * u.kHz
result = experiment.readout_frequency_optimization(rf_begin, rf_end, df, n_runs=50_000)

frequencies, fidelities = result.output.extract("frequencies", "fidelities")
plt.plot(frequencies*1e-6, fidelities, marker='o')
plt.xlabel("Readout Frequency (MHz)")
plt.ylabel("Fidelity")
plt.title("Readout Frequency Optimization")
argmax_index, max_fidelity = algorithms.argmax_general(fidelities)
frequencies, fidelities = result.output.extract("frequencies", "fidelities")
plt.plot(frequencies*1e-6, fidelities, marker='o')
plt.xlabel("Readout Frequency (MHz)")
plt.ylabel("Fidelity")
plt.title("Readout Frequency Optimization")
argmax_index, max_fidelity = algorithms.argmax_general(fidelities)
opt_freq = frequencies[argmax_index]
print(f"Optimal readout frequency: {opt_freq*1e-6} MHz with fidelity {max_fidelity:.4f}")
experiment.attributes.ro_fq = opt_freq
experiment.save_attributes()

### CLEAR waveform

In [ ]:
A_steady   = 0.03
ringdown_len = 800
total_len    = ringdown_len + len(clear_wf)

square_len = len(clear_wf)  # to keep the same active duration as CLEAR
square_wf  = np.full(square_len, A_steady, dtype=float)

combined_square_wf = np.zeros(total_len)
combined_square_wf[0:square_len] = square_wf

square_ro_op   = "readout_square"
square_pulseOp = PulseOp(
    "resonator",
    square_ro_op,
    type="measurement",
    length=total_len,
    I_wf=combined_square_wf,
)

experiment.register_pulse(square_pulseOp, override=True, persist=False, burn=True, save=False)

n_avg = 25000
cos_w_key, sin_w_key, m_sin_w_key = "cos", "sin", "minus_sin"
weights = [cos_w_key, sin_w_key, m_sin_w_key, cos_w_key]
drive_frequency = experiment.attributes.ro_fq + experiment.attributes.ro_chi
runres_square = experiment.readout_ge_integrated_trace(square_ro_op, drive_frequency, weights, n_avg=n_avg)
time_list_c, g_square, e_square = runres_square.output.extract("time_list", "g_trace", "e_trace")


fig, ax1 = plt.subplots(figsize=(8, 5))
l1, = ax1.plot(time_list_c, g_square.real, 'o-', label='|g> (SQUARE)')
l2, = ax1.plot(time_list_c, e_square.real, 's-', label='|e> (SQUARE)')
ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('|S| (arb)')
ax1.set_title('Integrated readout GE trace with SQUARE pulse')

ax2 = ax1.twinx()
l3, = ax2.plot(range(len(square_wf)), 1e3*square_wf, 'k--', label='SQUARE pulse')
ax2.set_ylabel('Amplitude (mV)')

lines  = [l1, l2, l3]
labels = [ln.get_label() for ln in lines]
ax1.legend(lines, labels, loc='best')
plt.tight_layout()
plt.show()

In [ ]:
from qubox.tools.waveforms import build_CLEAR_waveform_from_physics

attr = experiment.attributes
kappa = 2 * np.pi * attr.ro_kappa   # example 5 MHz resonator linewidth
chi   = 2 * np.pi * attr.ro_chi     # example 1 MHz dispersive shift

# -------------------------
# 1) CLEAR pulse PulseOp
# -------------------------
t_steady_duration = 200
t_kick     = 10

A_steady = 0.028
A_steady_default = A_steady
ringdown_len = 800

clear_wf = build_CLEAR_waveform_from_physics(
    t_duration=t_steady_duration+4*t_kick,
    t_kick=t_kick,
    A_steady=A_steady,
    kappa_rad_s=kappa,
    chi_rad_s=chi,
)
total_len    = ringdown_len + len(clear_wf)



combined_clear_wf = np.zeros(total_len)
combined_clear_wf[0:len(clear_wf)] = clear_wf


clear_op         = "readout_clear"
clear_pulse_name = f"{clear_op}_pulse"
clear_I_wf_name  = f"{clear_op}_I_wf"
clear_Q_wf_name  = f"{clear_op}_Q_wf"

pom = experiment.pulseOpMngr
pom.create_measurement_pulse(
    element=attr.ro_el,
    op=clear_op,
    length=total_len,
    pulse_name=clear_pulse_name,
    I_wf_name=clear_I_wf_name,
    Q_wf_name=clear_Q_wf_name,
    I_samples=A_steady_default,  # small square as initial guess
    Q_samples=0.0,
    digital_marker="ON",
    int_weights_mapping=None,
    int_weights_defs=None,
    persist=False,   # keep it volatile; optimizer will just keep modifying it
    override=True,
)
experiment.burn_pulses()

n_avg = 25000
drive_frequency = experiment.attributes.ro_fq + experiment.attributes.ro_chi
cos_w_key, sin_w_key, m_sin_w_key = "cos", "sin", "minus_sin"
weights = [cos_w_key, sin_w_key, m_sin_w_key, cos_w_key]
runres_clear = experiment.readout_ge_integrated_trace(clear_op, drive_frequency, weights, n_avg=n_avg)
time_list_c, g_clear, e_clear = runres_clear.output.extract("time_list", "g_trace", "e_trace")


fig, ax1 = plt.subplots(figsize=(8, 5))
l1, = ax1.plot(time_list_c, g_clear.real, 'o-', label='|g> (CLEAR)')
l2, = ax1.plot(time_list_c, e_clear.real, 's-', label='|e> (CLEAR)')
ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('|S| (arb)')
ax1.set_title('Integrated readout GE trace with CLEAR pulse')

ax2 = ax1.twinx()
l3, = ax2.plot(range(len(clear_wf)), 1e3*clear_wf, 'k--', label='CLEAR pulse')
ax2.set_ylabel('Amplitude (mV)')

lines  = [l1, l2, l3]
labels = [ln.get_label() for ln in lines]
ax1.legend(lines, labels, loc='best')
plt.tight_layout()
plt.show()



In [ ]:
experiment.set_readout_SPA_pump_power(9)

In [ ]:
from qubox.tools.waveforms import build_CLEAR_waveform_from_physics

attr = experiment.attributes
kappa = 2 * np.pi * attr.ro_kappa   # example 5 MHz resonator linewidth
chi   = 2 * np.pi * attr.ro_chi     # example 1 MHz dispersive shift

# -------------------------
# 1) CLEAR pulse PulseOp
# -------------------------
t_steady_duration = 260
t_kick     = 50

A_steady = 0.03
A_steady_default = A_steady
ringdown_len = 800

clear_wf = build_CLEAR_waveform_from_physics(
    t_duration=t_steady_duration+4*t_kick,
    t_kick=t_kick,
    A_steady=A_steady,
    kappa_rad_s=kappa,
    chi_rad_s=chi,
)

total_len    = ringdown_len + len(clear_wf)

combined_clear_wf = np.zeros(total_len)
combined_clear_wf[0:len(clear_wf)] = clear_wf


clear_op         = "readout_clear_"
clear_pulse_name = f"{clear_op}_pulse"
clear_I_wf_name  = f"{clear_op}_I_wf"
clear_Q_wf_name  = f"{clear_op}_Q_wf"

pom = experiment.pulseOpMngr
pom.create_measurement_pulse(
    element=attr.ro_el,
    op=clear_op,
    length=total_len,
    pulse_name=clear_pulse_name,
    I_wf_name=clear_I_wf_name,
    Q_wf_name=clear_Q_wf_name,
    I_samples=combined_clear_wf,  # small square as initial guess
    Q_samples=0.0,
    digital_marker="ON",
    int_weights_mapping=None,
    int_weights_defs=None,
    persist=False,   # keep it volatile; optimizer will just keep modifying it
    override=True,
)
experiment.burn_pulses()
pulseInfo_before = pom.get_pulseOp_by_element_op(attr.ro_el, clear_op)
pom.display_op(attr.ro_el, clear_op)
pom.modify_waveform(clear_I_wf_name, combined_clear_wf, persist=False, allow_type_change=True)
pulseInfo_after = pom.get_pulseOp_by_element_op(attr.ro_el, clear_op)
pom.display_op(attr.ro_el, clear_op)
experiment.burn_pulses()

print("Before modification:", pulseInfo_before)
print("After modification: ", pulseInfo_after)


drive_frequency = experiment.attributes.ro_fq + experiment.attributes.ro_chi
runres = experiment.readout_butterfly_measurement(
    clear_op,
    drive_frequency,
    show_analysis=True,
    set_measureMacro=True,
    n_shots=50_000,
    M0_MAX_TRIALS=200,
    run_ge_discrimination=True,
    ge_disc_kwargs={
        "n_samples": 25_000,                     # was your n_samples before
        "base_weight_keys": ("cos", "sin", "minus_sin"),
        "persist": False,
        "b_plot": True,
        "plots": ("rot_blob", "hist", "info"),
        "fig_title": "control analysis",
        "gain": 1,
        "k_g": 3,
        "k_e": 3,
    },
)
F, Q = runres.output.extract("F", "Q")



In [ ]:
I_wf, Q_wf= pom.get_pulse_waveforms("readout_clear__pulse")

### Readout Bayes optimization

In [ ]:
import numpy as np
from typing import List, Dict, Any, Tuple, Optional

import matplotlib.pyplot as plt
import logging
import json
import os
from datetime import datetime
from pathlib import Path

from skopt.space import Real, Integer
from skopt import gp_minimize
from qubox.tools.waveforms import CLEAR_waveform
from qubox.pulse_manager import MAX_AMPLITUDE, PulseOp
from qubox.logging_config import temporarily_set_levels

from IPython.display import display
import ipywidgets as widgets


def optimize_clear_pulse_bayes(
    experiment: "cQED_Experiment",
    *,
    t_duration: int,
    A_steady_default: float = 0.03,
    t_kick_default: int = 50,
    optimize_A_steady: bool = False,
    optimize_t_kick: bool = False,
    optimize_SPA_pump_power: bool = False,
    optimize_drive_frequency: bool = False,
    optimize_t_duration: bool = False,
    optimize_t_ringdown: bool = False,  # NEW: optimize ringdown time
    use_clear_pulse: bool = True,  # NEW: if False, use simple square pulse
    t_ringdown_default: int = 0,  # NEW: default ringdown time in clock cycles
    ringdown_bounds: Tuple[int, int] = (0, 200),  # NEW: ringdown time bounds
    spa_pump_bounds: Tuple[float, float] = (5.0, 13.0),
    readout_op: str = "readout",
    n_calls: int = 25,
    n_random_starts: int = 5,
    random_state: Optional[int] = None,
    ge_disc_kwargs: Optional[Dict[str, Any]] = None,
    MAX_AMPLITUDE: float = MAX_AMPLITUDE,
    persist = True,
    checkpoint_file: Optional[str] = None,  # NEW: checkpoint file path
    resume: bool = False,  # NEW: whether to resume from checkpoint
) -> Tuple[Any, Dict[str, Any]]:
    """
    Bayesian-optimizes readout pulse parameters using readout_butterfly_measurement as the figure of merit.
    cost = 1 - F * Q, with F and Q coming from readout_butterfly_measurement.
    
    Parameters:
    - use_clear_pulse: If True, optimize CLEAR pulse params. If False, use simple square pulse.
    - optimize_t_ringdown: If True, optimize the zero-amplitude ringdown period after the pulse.
    - t_ringdown_default: Default ringdown time (added after main pulse with zero amplitude).
    - ringdown_bounds: (min, max) ringdown time in clock cycles (must be multiple of 4).
    - checkpoint_file: Path to checkpoint file (default: auto-generated in experiment dir)
    - resume: If True, load and resume from existing checkpoint file
    """

    # ----------------------------
    # Setup checkpoint file
    # ----------------------------
    if checkpoint_file is None:
        # Auto-generate checkpoint filename with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        checkpoint_file = os.path.join(
            experiment.base_path,
            f"readout_opt_checkpoint_{timestamp}.json"
        )
    
    checkpoint_path = Path(checkpoint_file)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Load existing checkpoint if resuming
    x0_list = []
    y0_list = []
    if resume and checkpoint_path.exists():
        print(f"[RO-OPT] Loading checkpoint from: {checkpoint_path}")
        with open(checkpoint_path, "r") as f:
            checkpoint_data = json.load(f)
        
        # Extract previous evaluations
        for eval_record in checkpoint_data.get("evaluations", []):
            x0_list.append(eval_record["x"])
            y0_list.append(eval_record["cost"])
        
        print(f"[RO-OPT] Loaded {len(x0_list)} previous evaluations")
        
        # Adjust n_random_starts if we have enough warm-start points
        if len(x0_list) >= n_random_starts:
            n_random_starts = 0
            print(f"[RO-OPT] Using all {len(x0_list)} points as initial dataset (no random starts)")
    else:
        if resume:
            print(f"[RO-OPT] No checkpoint found at {checkpoint_path}, starting fresh")
        else:
            print(f"[RO-OPT] Starting fresh optimization (resume=False)")
    
    def save_checkpoint(x, cost, iteration):
        """Save current evaluation to checkpoint file"""
        # Convert numpy types to Python native types for JSON serialization
        def convert_to_native(val):
            if isinstance(val, (np.integer, np.int64, np.int32)):
                return int(val)
            elif isinstance(val, (np.floating, np.float64, np.float32)):
                return float(val)
            elif isinstance(val, np.ndarray):
                return val.tolist()
            else:
                return val
        
        # Load existing data
        if checkpoint_path.exists():
            with open(checkpoint_path, "r") as f:
                data = json.load(f)
        else:
            data = {
                "metadata": {
                    "readout_op": readout_op,
                    "use_clear_pulse": use_clear_pulse,
                    "optimize_A_steady": optimize_A_steady,
                    "optimize_t_kick": optimize_t_kick,
                    "optimize_SPA_pump_power": optimize_SPA_pump_power,
                    "optimize_drive_frequency": optimize_drive_frequency,
                    "optimize_t_duration": optimize_t_duration,
                    "optimize_t_ringdown": optimize_t_ringdown,
                    "started_at": datetime.now().isoformat(),
                },
                "evaluations": []
            }
        
        # Append new evaluation (convert all values to JSON-serializable types)
        data["evaluations"].append({
            "iteration": int(iteration),
            "x": [convert_to_native(v) for v in x],
            "cost": float(cost),
            "timestamp": datetime.now().isoformat(),
        })
        data["metadata"]["last_updated"] = datetime.now().isoformat()
        
        # Write atomically (write to temp, then rename)
        temp_path = checkpoint_path.with_suffix(".tmp")
        with open(temp_path, "w") as f:
            json.dump(data, f, indent=2)
        temp_path.replace(checkpoint_path)

    qubox_logger = logging.getLogger("qubox")
    qm_logger = logging.getLogger("qm")

    with temporarily_set_levels([qubox_logger, qm_logger], logging.WARNING):

        attr = experiment.attributes
        pom = experiment.pulseOpMngr

        # bounds for t_duration optimization
        min_t_duration = 100
        max_t_duration = 400
        max_t_duration = 4 * (max_t_duration // 4)

        t_duration_clipped = max(min_t_duration, min(max_t_duration, int(t_duration)))
        t_duration_base = 4 * int(round(t_duration_clipped / 4))
        if t_duration_base < min_t_duration:
            t_duration_base = min_t_duration
        if t_duration_base > max_t_duration:
            t_duration_base = max_t_duration

        initial_length = max_t_duration if optimize_t_duration else t_duration_base

        # names tied to this CLEAR readout op
        clear_op = readout_op
        clear_pulse_name = f"{clear_op}_pulse"
        clear_I_wf_name = f"{clear_op}_I_wf"
        clear_Q_wf_name = f"{clear_op}_Q_wf"

        # ensure the measurement pulse + waveforms exist
        pom.create_measurement_pulse(
            element=attr.ro_el,
            op=clear_op,
            length=initial_length,
            pulse_name=clear_pulse_name,
            I_wf_name=clear_I_wf_name,
            Q_wf_name=clear_Q_wf_name,
            I_samples=A_steady_default,
            Q_samples=0.0,
            digital_marker="ON",
            int_weights_mapping=None,
            int_weights_defs=None,
            persist=persist,
            override=True,
        )

        # base drive frequency for resonator
        base_drive_frequency = attr.ro_fq + attr.ro_chi

        # defaults for ge_disc_kwargs (if not provided)
        if ge_disc_kwargs is None:
            ge_disc_kwargs = {
                "n_samples": 10_000,
                "base_weight_keys": ("cos", "sin", "minus_sin"),
                "persist": persist,
                "gain": 1,
                "auto_update_postsel": True,  # Auto-update measureMacro post-selection config
                "blob_k_g": 3.0,
                "blob_k_e": 3.0,
                "b_plot": False,  # Disable plotting during optimization to prevent figure overflow
            }

        # Global max_t_kick bound
        if optimize_t_duration:
            max_t_kick_global = max(1, max_t_duration // 4)
        else:
            max_t_kick_global = max(1, t_duration_base // 4)

        # histories for plotting
        cost_hist: List[float] = []
        F_hist: List[float] = []
        Q_hist: List[float] = []

        best_record: Dict[str, Any] = {
            "cost": np.inf,
            "F": np.nan,
            "Q": np.nan,
            "x": None,
            "params": None,
        }

        # Stop button for controlled termination
        stop_button = widgets.Button(
            description='Stop Optimization',
            button_style='danger',
            tooltip='Click to stop optimization gracefully after current iteration',
            icon='stop'
        )
        stop_flag = {"stop": False}
        
        def on_stop_click(b):
            stop_flag["stop"] = True
            b.description = 'Stopping...'
            b.disabled = True
            print("\n[RO-OPT] Stop requested. Will finish current iteration and terminate gracefully.")
        
        stop_button.on_click(on_stop_click)
        display(stop_button)
        print("[RO-OPT] Click 'Stop Optimization' button to terminate gracefully.\n")

        # parameterization helper: x -> pulse parameters
        def decode_pulse_params(x_pulse: List[float]) -> Dict[str, Any]:
            if use_clear_pulse:
                # CLEAR pulse parameters
                alpha_rise_hi, alpha_rise_lo, u_fall_hi, v_fall_lo = x_pulse[0:4]
                idx = 4
            else:
                # Simple square pulse - no CLEAR parameters
                idx = 0

            if optimize_A_steady:
                alpha_steady = x_pulse[idx]
                idx += 1
                A_steady = MAX_AMPLITUDE * alpha_steady
            else:
                A_steady = A_steady_default

            if use_clear_pulse and optimize_t_kick:
                t_kick = int(x_pulse[idx])
                idx += 1
            else:
                t_kick = t_kick_default

            if optimize_t_duration:
                t_duration_4 = int(x_pulse[idx])
                t_duration_eff = 4 * t_duration_4
                idx += 1
            else:
                t_duration_eff = t_duration_base

            if optimize_t_ringdown:
                t_ringdown_4 = int(x_pulse[idx])
                t_ringdown = 4 * t_ringdown_4
                idx += 1
            else:
                t_ringdown = t_ringdown_default

            def clip(a: float) -> float:
                return max(-MAX_AMPLITUDE, min(MAX_AMPLITUDE, a))

            params = {
                "A_steady": A_steady,
                "t_duration": t_duration_eff,
                "t_ringdown": t_ringdown,
            }

            if use_clear_pulse:
                A_rise_hi = MAX_AMPLITUDE * alpha_rise_hi
                A_rise_lo = -MAX_AMPLITUDE * alpha_rise_lo
                A_fall_hi = MAX_AMPLITUDE * u_fall_hi
                A_fall_lo = MAX_AMPLITUDE * (2.0 * v_fall_lo - 1.0)
                
                params.update({
                    "t_kick": t_kick,
                    "A_rise_hi": clip(A_rise_hi),
                    "A_rise_lo": clip(A_rise_lo),
                    "A_fall_lo": clip(A_fall_lo),
                    "A_fall_hi": clip(A_fall_hi),
                })

            return params

        # patch pulse helper
        def apply_pulse(params: Dict[str, Any], persist: bool) -> None:
            """Build pulse envelope and patch the readout pulse."""
            A_steady = params["A_steady"]
            t_duration_eff = params["t_duration"]
            t_ringdown = params["t_ringdown"]

            if not (min_t_duration <= t_duration_eff <= max_t_duration):
                raise ValueError(f"t_duration={t_duration_eff} out of range")

            if use_clear_pulse:
                # Build CLEAR waveform
                t_kick = params["t_kick"]
                A_rise_hi = params["A_rise_hi"]
                A_rise_lo = params["A_rise_lo"]
                A_fall_lo = params["A_fall_lo"]
                A_fall_hi = params["A_fall_hi"]
                
                max_t_kick_local = max(1, t_duration_eff // 4)
                if not (1 <= t_kick <= max_t_kick_local):
                    raise ValueError(f"t_kick={t_kick} out of range")

                pulse_env = CLEAR_waveform(
                    t_duration=t_duration_eff,
                    t_kick=t_kick,
                    A_steady=A_steady,
                    A_rise_hi=A_rise_hi,
                    A_rise_lo=A_rise_lo,
                    A_fall_lo=A_fall_lo,
                    A_fall_hi=A_fall_hi,
                )
            else:
                # Simple square pulse
                pulse_env = np.full(t_duration_eff, A_steady)

            # Add ringdown period (zero amplitude)
            if t_ringdown > 0:
                ringdown_env = np.zeros(t_ringdown)
                pulse_env = np.concatenate([pulse_env, ringdown_env])

            # Pad to multiple of 4
            pad = (-len(pulse_env)) % 4
            if pad:
                pulse_env = np.pad(pulse_env, (0, pad))

            total_len = len(pulse_env)
            combined_wf = np.zeros(total_len)
            combined_wf[:len(pulse_env)] = pulse_env

            pom.modify_waveform(clear_I_wf_name, combined_wf, persist=persist, allow_type_change=True)

            pulse_patch = PulseOp(
                element=attr.ro_el, op=clear_op, pulse=clear_pulse_name, type="measurement",
                length=total_len, digital_marker=None,
                I_wf_name=clear_I_wf_name, Q_wf_name=clear_Q_wf_name,
                I_wf=None, Q_wf=None, int_weights_mapping=None, int_weights_defs=None,
            )

            pom.modify_pulse_op(pulse_patch, persist=persist)
            experiment.burn_pulses()

        # define skopt parameter space
        pulse_param_space = []
        
        if use_clear_pulse:
            # CLEAR pulse parameters
            pulse_param_space.extend([
                Real(0.0, 1.0, name="alpha_rise_hi"),
                Real(0.0, 1.0, name="alpha_rise_lo"),
                Real(-1.0, 1.0, name="u_fall_hi"),
                Real(0.0, 1.0, name="v_fall_lo"),
            ])

        if optimize_A_steady:
            pulse_param_space.append(Real(1e-3, 1.0, name="alpha_steady"))
        
        if use_clear_pulse and optimize_t_kick:
            pulse_param_space.append(Integer(1, max_t_kick_global, name="t_kick"))
        
        if optimize_t_duration:
            pulse_param_space.append(Integer(min_t_duration // 4, max_t_duration // 4, name="t_duration_4"))
        
        if optimize_t_ringdown:
            ringdown_min, ringdown_max = ringdown_bounds
            ringdown_min_4 = max(0, ringdown_min // 4)
            ringdown_max_4 = ringdown_max // 4
            pulse_param_space.append(Integer(ringdown_min_4, ringdown_max_4, name="t_ringdown_4"))

        n_pulse_dims = len(pulse_param_space)
        param_space = list(pulse_param_space)
        spa_index: Optional[int] = None
        drive_index: Optional[int] = None

        if optimize_SPA_pump_power:
            spa_index = len(param_space)
            spa_min, spa_max = spa_pump_bounds
            param_space.append(Real(spa_min, spa_max, name="spa_pump_power"))

        if optimize_drive_frequency:
            drive_index = len(param_space)
            fq_min = float(attr.ro_fq - 5e6)
            fq_max = float(attr.ro_fq + 5e6)
            param_space.append(Real(fq_min, fq_max, name="drive_frequency"))

        # Modified objective function with checkpointing
        iteration_counter = [len(x0_list)]  # Start from loaded checkpoint count
        
        # objective function
        def objective(x: List[float]) -> float:
            """Run ge_disc + butterfly, return cost, and checkpoint"""
            x_pulse = x[:n_pulse_dims]
            params = decode_pulse_params(x_pulse)

            # Enforce constraints
            if use_clear_pulse:
                if not (params["A_rise_hi"] > 0 and params["A_rise_lo"] < 0):
                    return 1.0
                if not (params["A_fall_lo"] < params["A_fall_hi"]):
                    return 1.0
                t_kick = params["t_kick"]
                max_t_kick_local = max(1, params["t_duration"] // 4)
                if not (1 <= t_kick <= max_t_kick_local):
                    return 1.0
            
            t_duration_eff = params["t_duration"]
            if not (min_t_duration <= t_duration_eff <= max_t_duration):
                return 1.0
            if not (params["A_steady"] > 0):
                return 1.0

            # Set SPA pump power if optimizing
            if optimize_SPA_pump_power and spa_index is not None:
                spa_power = float(x[spa_index])
                experiment.set_readout_SPA_pump_power(spa_power)
                params["spa_pump_power"] = spa_power
            else:
                params["spa_pump_power"] = None

            # Choose drive frequency
            if optimize_drive_frequency and drive_index is not None:
                local_drive_frequency = float(x[drive_index])
                params["drive_frequency"] = local_drive_frequency
            else:
                local_drive_frequency = base_drive_frequency
                params["drive_frequency"] = local_drive_frequency

            # Patch pulse
            apply_pulse(params, persist=persist)

            # STEP 1: Run GE discrimination to set up measureMacro and post-selection config
            # CRITICAL: burn_rot_weights=True regenerates integration weights for the new pulse length
            try:
                _ = experiment.readout_ge_discrimination(
                    measure_op=clear_op,
                    drive_frequency=local_drive_frequency,
                    update_measureMacro=True,
                    burn_rot_weights=True,  # Must be True to regenerate integration weights
                    **ge_disc_kwargs
                )
            except Exception as e:
                print(f"[RO-OPT] GE discrimination failed: {e}")
                return 1.0

            # STEP 2: Run butterfly measurement (uses measureMacro config from ge_disc)
            try:
                runres = experiment.readout_butterfly_measurement(
                    show_analysis=False,
                    update_measureMacro=False,
                    n_samples=10_000,
                    M0_MAX_TRIALS=500,
                    use_stored_config=True,
                )
            except Exception as e:
                print(f"[RO-OPT] Butterfly measurement failed: {e}")
                return 1.0

            F, Q = runres.output.extract("F", "Q")
            F = float(F)
            Q = float(Q)

            if not (np.isfinite(F) and np.isfinite(Q)):
                score = 1.0
            else:
                F_clipped = min(max(F, 0.0), 1.0)
                Q_clipped = min(max(Q, 0.0), 1.0)
                score = 1.0 - (F_clipped * Q_clipped)

            F_hist.append(F)
            Q_hist.append(Q)
            cost_hist.append(score)

            if score < best_record["cost"]:
                best_record["cost"] = score
                best_record["F"] = F
                best_record["Q"] = Q
                best_record["x"] = list(x)
                best_record["params"] = params.copy()

            # Save checkpoint after each evaluation
            current_iter = iteration_counter[0]
            save_checkpoint(x, score, current_iter)
            iteration_counter[0] += 1

            # Print ALL parameters including optional ones
            print_parts = [
                f"[RO-OPT] iter={current_iter}",
                f"A_steady={params['A_steady']:.4f}",
                f"t_dur={params['t_duration']}",
            ]
            
            # Always print t_ringdown if it exists (even if 0)
            if "t_ringdown" in params:
                print_parts.append(f"t_ringdown={params['t_ringdown']}")
            
            if use_clear_pulse:
                print_parts.extend([
                    f"t_kick={params['t_kick']}",
                    f"A_rise_hi={params['A_rise_hi']:.4f}",
                    f"A_rise_lo={params['A_rise_lo']:.4f}",
                    f"A_fall_hi={params['A_fall_hi']:.4f}",
                    f"A_fall_lo={params['A_fall_lo']:.4f}",
                ])
            
            if params.get("spa_pump_power") is not None:
                print_parts.append(f"SPA_pump={params['spa_pump_power']:.2f}dBm")
            
            if optimize_drive_frequency:
                fq_mhz = params["drive_frequency"] / 1e6
                print_parts.append(f"drive_fq={fq_mhz:.2f}MHz")
            
            print_parts.extend([
                f"F={F:.4f}",
                f"Q={Q:.4f}",
                f"cost={score:.6f}",
            ])
            
            print(", ".join(print_parts))

            return score

        # Callback function to check stop flag after each iteration
        def check_stop_callback(res):
            """Callback that stops optimization if stop button was pressed"""
            if stop_flag["stop"]:
                print(f"\n[RO-OPT] Stop requested after iteration {len(cost_hist)}. Terminating optimization.")
                return True  # Return True to stop optimization
            return False  # Continue optimization

        # run Bayesian optimization with warm start (using gp_minimize directly)
        result = gp_minimize(
            func=objective,
            dimensions=param_space,
            n_calls=n_calls,
            n_random_starts=n_random_starts,
            random_state=random_state,
            callback=check_stop_callback,
            x0=x0_list if x0_list else None,  # Warm start with loaded points
            y0=y0_list if y0_list else None,
        )

        # extract best metrics
        if best_record["x"] is not None:
            best_cost = float(best_record["cost"])
            best_F = float(best_record["F"])
            best_Q = float(best_record["Q"])
            best_x = best_record["x"]
            best_params = best_record["params"]
        else:
            best_cost = float("nan")
            best_F = float("nan")
            best_Q = float("nan")
            best_x = result.x
            best_pulse_x = best_x[:n_pulse_dims]
            best_params = decode_pulse_params(best_pulse_x)

        print(f"[RO-OPT] Final best: F={best_F:.4f}, Q={best_Q:.4f}, cost={best_cost:.6f}")
        print(f"[RO-OPT] Checkpoint saved to: {checkpoint_path}")

        if optimize_SPA_pump_power and spa_index is not None and best_record["x"] is not None:
            best_spa_power = float(best_record["x"][spa_index])
            experiment.set_readout_SPA_pump_power(best_spa_power)
            best_params["spa_pump_power"] = best_spa_power

        if optimize_drive_frequency and drive_index is not None and best_record["x"] is not None:
            best_drive_frequency = float(best_record["x"][drive_index])
            best_params["drive_frequency"] = best_drive_frequency
        else:
            best_params["drive_frequency"] = base_drive_frequency

        best_params["F"] = best_F
        best_params["Q"] = best_Q
        best_params["cost"] = best_cost

        apply_pulse(best_params, persist=True)

        print("[RO-OPT] Best parameters:")
        for k, v in best_params.items():
            print(f"  {k}: {v}")

        # plot iterations
        if cost_hist:
            iters = np.arange(1, len(cost_hist) + 1)
            plt.figure(figsize=(6, 4))
            plt.plot(iters, cost_hist, marker="o")
            plt.xlabel("Iteration")
            plt.ylabel("Cost (1 - F·Q)")
            pulse_type = "CLEAR" if use_clear_pulse else "Square"
            plt.title(f"{pulse_type} Pulse Bayesian Optimization: Cost vs Iteration")
            plt.grid(True)
            plt.tight_layout()
            plt.show()

        return result, best_params

# Run optimization
result, best_params = optimize_clear_pulse_bayes(
    experiment=experiment,
    t_duration=300,
    A_steady_default=0.027,
    t_kick_default=50,
    optimize_A_steady=True,
    optimize_t_kick=True,
    optimize_SPA_pump_power=True,
    optimize_drive_frequency=False,
    optimize_t_duration=True,
    optimize_t_ringdown=True,
    use_clear_pulse=False,
    spa_pump_bounds=(8.0, 10.5),
    readout_op="readout_temp",
    n_calls=400,
    n_random_starts=15,
    random_state=0,
    MAX_AMPLITUDE=0.2,
    persist=False
)


In [ ]:
experiment.base_path

### readout weight optimization

In [ ]:
ro_op = "readout"
pulseOp = experiment.pulseOpMngr.get_pulseOp_by_element_op("resonator", ro_op)
division_clks = 1 # clock cycles per integration chunk
division_len = 4 * division_clks  # total cycles per "division" (I,Q demod window)
readout_len = pulseOp.length      # total readout length in clock cycles
num_division = int((readout_len) / (division_len))
time_list = np.arange(division_clks * 4, readout_len + 1, division_len)

num_division

In [ ]:
ro_op = "readout"
pulseOp = experiment.pulseOpMngr.get_pulseOp_by_element_op("resonator", ro_op)
division_clks = 1 # clock cycles per integration chunk
division_len = 4 * division_clks  # total cycles per "division" (I,Q demod window)
readout_len = pulseOp.length      # total readout length in clock cycles
num_division = int((readout_len) / (division_len))
time_list = np.arange(division_clks * 4, readout_len + 1, division_len)

print("Integration weights chunk-size length in clock cycles:", division_clks)
print("The readout has been sliced in the following number of divisions", num_division)
num_division*division_len

cos_w_key, sin_w_key, m_sin_w_key = "cos", "sin", "minus_sin"
result = experiment.readout_weights_optimization(ro_op, cos_w_key, sin_w_key, m_sin_w_key, num_division, n_avg=10000, persist=True, set_measureMacro=True)


In [ ]:
experiment.set_readout_SPA_pump_power(8)

### Readout Full Calibration

In [ ]:
ro_op = "readout"

detuning = experiment.attributes.ro_chi
drive_frequency = experiment.attributes.ro_fq
n_avg_weights      = 400_00
n_samples_disc     = 100_00
n_samples_butterfly = 100_00

k = 4
#M0_MAX_TRIALS = 2 * algorithms.samples_for_k_sigma_event(k, 0.9999)
M0_MAX_TRIALS = 1000
res = experiment.calibrate_readout_full(
    ro_op,
    drive_frequency,
    n_avg_weights=n_avg_weights,
    n_samples_disc=n_samples_disc,
    n_shots_butterfly=n_samples_butterfly,   # name kept; used as butterfly n_samples
    display_analysis=True,
    blob_k_g=k,
    # blob_k_e=k,  # optional, defaults to blob_k_g
    wopt_kwargs={"num_div": None},
    ge_kwargs={
        "gain": 1,
        "persist": True,
        "update_measureMacro": True,
    },
    bfly_kwargs={
        "M0_MAX_TRIALS": M0_MAX_TRIALS,
        "n_samples": n_samples_butterfly,
        # "prep_policy": "BLOBS",  # default
    },
)


In [ ]:
# ---- run ----
res = experiment.qubit_reset_benchmark(bit_size=1000, num_shots=1000)

# ---- simulation path ----
if hasattr(res, "mode") and (res.mode == "simulate"):
    print("SIMULATION: Skipping reset-benchmark analysis that expects real data.")
    sim = getattr(res, "sim_samples", None) or experiment.quaProgMngr.get_simulation_results()
    # Optional: visualize simulated waveforms
    # experiment.quaProgMngr._plot_sim_custom(sim)
else:
    # ---- hardware path ----
    out = res.output if hasattr(res, "output") else res  # tolerate Output or RunResult

    # fetch arrays
    target, state_M1, state_M2, I_M1, Q_M1, I_M2, Q_M2, Pe_star, eps_cond, eps_unc = out.extract(
        "target", "state_M1", "state_M2", "I_M1", "Q_M1", "I_M2", "Q_M2", "Pe_star", "eps_cond", "eps_unc"
    )

    # ===========================
    # Reset-benchmark analysis v2
    # ===========================
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # 0) flatten
    tgt  = np.asarray(target ,    dtype=bool).ravel()
    m1   = np.asarray(state_M1,   dtype=bool).ravel()
    m2   = np.asarray(state_M2,   dtype=bool).ravel()
    I1   = np.asarray(I_M1).ravel()
    Q1   = np.asarray(Q_M1).ravel()
    I2   = np.asarray(I_M2).ravel()
    Q2   = np.asarray(Q_M2).ravel()

    N_tot = tgt.size
    print(f"{N_tot} logical trials  ({target.shape[0]} shots × {target.shape[1]} bits)\n")

    # 1) headline metrics (recompute)
    Pe_star_calc  = m2.mean()
    need_reset    = m1
    eps_cond_calc = ((m2 & need_reset).sum() / need_reset.sum()) if need_reset.any() else np.nan
    eps_unc_calc  = Pe_star_calc

    print("---------- headline metrics --------------------------------")
    print(f"stored  Pe_star : {Pe_star :10.6e}   recomputed : {Pe_star_calc :10.6e}")
    print(f"stored  eps_cond: {eps_cond:10.6e}   recomputed : {eps_cond_calc:10.6e}")
    print(f"stored  eps_unc : {eps_unc :10.6e}   recomputed : {eps_unc_calc :10.6e}")
    print("-------------------------------------------------------------\n")

    # 2) contingency table (4 × 2)
    rows = pd.MultiIndex.from_product([["target_g","target_e"], ["M1_g","M1_e"]],
                                      names=["target","M1"])
    cols = ["M2_g","M2_e"]
    table = pd.DataFrame(0, index=rows, columns=cols)

    for b_t, b1, b2 in zip(tgt, m1, m2):
        table.loc[("target_e" if b_t else "target_g",
                   "M1_e"     if b1 else "M1_g"),
                  "M2_e" if b2 else "M2_g"] += 1

    print("Contingency table (counts):")
    try:
        display(table)  # Jupyter-friendly
    except Exception:
        print(table)

    # 3) quick visuals (optional)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].scatter(I1, Q1, s=2, alpha=0.4, label="M1 IQ")
    ax[0].set_title("M1 IQ")
    ax[0].set_xlabel("I"); ax[0].set_ylabel("Q"); ax[0].legend()

    ax[1].scatter(I2, Q2, s=2, alpha=0.4, label="M2 IQ")
    ax[1].set_title("M2 IQ")
    ax[1].set_xlabel("I"); ax[1].set_ylabel("Q"); ax[1].legend()
    fig.tight_layout()
    plt.show()


In [ ]:
step = max(1, N_tot // 20000)            # ≈5 % keep rate
sub   = np.zeros(N_tot, dtype=bool)
sub[::step] = True                     # True every 'step'-th index

g_mask_M1 = (~tgt) & sub              # ground-target shots (subsampled)
e_mask_M1 =  tgt  & sub               # excited-target shots
g_mask_M2 = (~m2) & sub               # post-reset ground
e_mask_M2 =  m2  & sub                # post-reset excited

fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

ax[0].scatter(I1[g_mask_M1], Q1[g_mask_M1],
              s=3, alpha=0.25, label="target g")
ax[0].scatter(I1[e_mask_M1], Q1[e_mask_M1],
              s=3, alpha=0.25, label="target e")
ax[0].set_title("IQ cloud – M1")
ax[0].set_xlabel("I"); ax[0].set_ylabel("Q"); ax[0].legend()

ax[1].scatter(I2[g_mask_M2], Q2[g_mask_M2],
              s=3, alpha=0.25, label="post-reset g")
ax[1].scatter(I2[e_mask_M2], Q2[e_mask_M2],
              s=3, alpha=0.25, label="post-reset e")
ax[1].set_title("IQ cloud – M2")
ax[1].set_xlabel("I"); ax[1].legend()

plt.tight_layout(); plt.show()

## SPA benchmarking and optimization

### SPA DC tune up

In [ ]:
dc_list = np.linspace(1.34, 1.5, 2)
ro_fq = experiment.attributes.ro_fq
n_avg = 5000
sample_fqs = [ro_fq]
#pump_detuning = 5e6 
#experiment.set_readout_SPA_pump_frequency(experiment.attributes.ro_fq*2+pump_detuning)
experiment.set_readout_SPA_pump_power(9)

runres = experiment.SPA_flux_optimization(dc_list, sample_fqs, n_avg, odc_param="voltage5", step=0.0005, delay_s=0.002)

mag, flux_dc_list = runres.output.extract("mag_matrix","flux_dc_list")
plt.plot(flux_dc_list, mag[0], "o-")
plt.xlabel("Flux DC (V)")
plt.ylabel("Readout magnitude (arb.)")
plt.title("SPA flux optimization")

In [ ]:
ro_fq = experiment.attributes.ro_fq
sample_fqs = [ro_fq]
rr = experiment.SPA_flux_optimization2(
    dc_list=[0, 2.0],           # just need range; we use min/max
    sample_fqs=sample_fqs,
    n_avg=2000,
    mode="auto_peak",
    scout_window=0.2,
    scout_step=0.01,
    refine_half_width=0.03,
    refine_step=0.002,
    lock_delta=0.001,
    approach_direction="up",
    approach_reset=-0.6,          # optional but helps hysteresis
)
locked_dc = rr.output["locked_dc"]
locked_S  = rr.output["locked_S"]


### SPA pump power, frequency optimization

In [ ]:
import numpy as np
CENTER_POWER_DBM = 9
FREQ_SPAN_HZ     = 25e6
POWER_SPAN_DB    = 3
NUM_FREQS        = 25
NUM_POWERS       = 10
SAMPLES_PER_RUN  = 20_000

pump_detunings = np.linspace(-FREQ_SPAN_HZ, 0, NUM_FREQS)
pump_powers    = np.linspace(
    CENTER_POWER_DBM - POWER_SPAN_DB / 2.0,
    CENTER_POWER_DBM + POWER_SPAN_DB / 2.0,
    NUM_POWERS,
)
metric = "butterfly"
readout_op = "readout"
if metric == "assignment_fidelity":
    res = experiment.SPA_pump_frequency_optimization(
        readout_op=readout_op,
        pump_powers=pump_powers,
        pump_detunings=pump_detunings,
        samples_per_run=SAMPLES_PER_RUN,          # default for this mode
        metric=metric,
        assignment_kwargs={
            # override / confirm samples_per_run for this mode:
            "samples_per_run": SAMPLES_PER_RUN,
            "disc_kwargs": {
                "b_plot": False,
                # "plots": ("hist", "raw_blob"),
                # "fig_title": "SPA discriminator sweep",
            },
        },
    )
    
elif metric == "butterfly":
    res = experiment.SPA_pump_frequency_optimization(
        readout_op=readout_op,
        drive_frequency=experiment.attributes.ro_fq,
        pump_powers=pump_powers,
        pump_detunings=pump_detunings,
        metric=metric,
        butterfly_kwargs={
            "show_analysis": False,
            "n_samples_disc": SAMPLES_PER_RUN,
            "n_samples_butterfly": 25_000,
            "M0_MAX_TRIALS": 500,
            "blob_k_g": 3.0,
            "blob_k_e": None,
            "k_g": 3,
            "k_e": 3,
        },
    )

In [ ]:
butterfly_matrix = res.output.extract("butterfly_matrix")
F_matrix = np.zeros((NUM_POWERS, NUM_FREQS))
Q_matrix = np.zeros((NUM_POWERS, NUM_FREQS))
for i in range(NUM_POWERS):
    for j in range(NUM_FREQS):
        butterfly = butterfly_matrix[i][j]
        F = butterfly["F"]
        Q = butterfly["Q"]
        F_matrix[i, j] = F
        Q_matrix[i, j] = Q
plotting.plot_hm(
    F_matrix, pump_detunings*1e-6, pump_powers,
    xlabel="Pump Detuning (MHz)", ylabel="Pump Power (dBm)",
    title="SPA Pump Frequency Optimization - Fidelity", barlabel="F"
)
plotting.plot_hm(
    Q_matrix, pump_detunings*1e-6, pump_powers,
    xlabel="Pump Detuning (MHz)", ylabel="Pump Power (dBm)",
    title="SPA Pump Frequency Optimization - QND", barlabel="QND"
)

w_Q, w_F = 0.5, 0.5
weighted_score = w_Q*Q_matrix + w_F*F_matrix
plotting.plot_hm(
    weighted_score, pump_detunings*1e-6, pump_powers,
    xlabel="Pump Detuning (MHz)", ylabel="Pump Power (dBm)",
    title=f"SPA Pump Frequency Optimization - weighted score (w_Q={w_Q}, w_F={w_F})", barlabel="QND"
)

argmax_index, score = algorithms.argmax_general(weighted_score)
opt_power = pump_powers[argmax_index[0]]
opt_detuning = pump_detunings[argmax_index[1]]
opt_SPA_freq = experiment.attributes.ro_fq*2 + opt_detuning

opt_F = F_matrix[argmax_index]
opt_Q = Q_matrix[argmax_index]
print("Optimal SPA pump frequency optimization results:")
print(f"Optimal SPA pump fidelity: {opt_F}")
print(f"Optimal SPA pump QND: {opt_Q}")
print(f"Optimal SPA pump frequency optimization score: {score}")
print(f"Optimal SPA pump power: {opt_power} dBm")
print(f"Optimal SPA pump detuning: {opt_detuning*1e-6} MHz")

print(f"Optimal SPA pump frequency: {opt_SPA_freq} Hz")

In [ ]:
experiment.attributes.ro_fq*2 + (-20e6)

### Readout_leakage_benchmarking

In [ ]:
m_max = 40
num_sequences = 15
control_bits = np.array([np.random.randint(0, 2, size=m_max) for _ in range(num_sequences)])
runres = experiment.qubit_readout_leakage_benchmarking(control_bits, num_sequences=num_sequences, n_avg=1000)
states = runres.output.extract("S")
cQED_plottings.plot_IQ(states, s=1)

In [ ]:
expected_bits = np.zeros((num_sequences, m_max+1), dtype=int)
expected_bits[:, 0] = -1
num_sequences, m_max = control_bits.shape
for i in range(num_sequences):
    for j in range(m_max):
        if control_bits[i][j] == 1:
            expected_bits[i][j+1] = expected_bits[i][j]*(-1)
        else:
            expected_bits[i][j+1] = expected_bits[i][j]

experimental_bits = np.where(states.real>0, +1, -1)

In [ ]:
def exponential_decay(x, tau):
    return np.exp(-x/tau)

E = expected_bits
X = experimental_bits

E_b = E[:, None, :]                   
corr_per_step = np.mean(E_b * X, axis=(0,1))

overall_corr = np.mean(E_b * X)       

print("corr_per_step:", corr_per_step)
print("overall_corr:", overall_corr)

steps = range(len(corr_per_step))
steps = [i for i in range(len(corr_per_step))]
custom_plot_opts = {"title": "expected bit vs experimental bit correlation", "xlabel": "m'th bit", "ylabel":"correlation"}
generalized_fit(steps, corr_per_step, exponential_decay, p0=[40], plotting=True, plot_options=custom_plot_opts)

In [ ]:
import matplotlib.pyplot as plt

# If your bits are –1/+1, shift them to 0/1 so the colormap is intuitive:
expct = (expected_bits + 1)/2
iter_index = 5
exptl = (experimental_bits[:, iter_index, :] + 1)/2

# 1) Compute the match boolean matrix (True where bits agree)
match = (expct == exptl)        # shape = (n_seqs, L), dtype=bool
match_int = match.astype(int)   # 1 = match, 0 = mismatch

# 2) Plot all three side‐by‐side
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

# Expected
im1 = ax1.imshow(expct,
                 aspect='auto', interpolation='nearest',
                 cmap='viridis', vmin=0, vmax=1)
ax1.set_title("Expected bits")
ax1.set_xlabel("Step")

# Experimental
im2 = ax2.imshow(exptl,
                 aspect='auto', interpolation='nearest',
                 cmap='viridis', vmin=0, vmax=1)
ax2.set_title("Experimental bits")
ax2.set_xlabel("Step")

# Match / Mismatch
im3 = ax3.imshow(match_int,
                 aspect='auto', interpolation='nearest',
                 cmap='Greys', vmin=0, vmax=1)
ax3.set_title("Match (1) / Mismatch (0)")
ax3.set_xlabel("Step")

fig.colorbar(im3, ax=[ax1, ax2, ax3], label="Bit value")
ax1.set_ylabel("Sequence index")
plt.show()

# Storage Cavity Experiments

## Storage Cavity Characterization

### default displacement and selective/unsel pi pulse generation

In [ ]:
disp_len = 100
disp_amp = 0.01
pulseOp_const_disp = PulseOp("storage", "const_disp", "const_disp_pulse", type="control", length=disp_len, I_wf_name="const_disp_I_wf", Q_wf_name="const_disp_Q_wf", I_wf=disp_amp, Q_wf=0)
experiment.register_pulse(pulseOp_const_disp, override=True, persist=True, save=True)



# ------------------------------
# SELECTIVE DRAG pulses (|g,n> <-> |e,n|)
# ------------------------------
sel_rlen = 1000

kaiser_amp = 0.0013420737712946228
beta = 7.967
sel_r180_I_wf, sel_r180_Q_wf = kaiser_pulse_waveforms(kaiser_amp, sel_rlen, beta)
pom = experiment.pulseOpMngr

# Optional: per-gate fine tweaks (start with none)

d_lambda_map = {
  "x180": -5.480646e4,
  "y180": -5.480646e4
}
d_alpha_map = {
    "x180":0.020317,
    "y180":0.020317
}
d_omega_map = {
  "x180": 2.489436e4,
  "y180": 2.489436e4
}

res = register_rotations_from_ref_iq(
    pom,
    sel_r180_I_wf,        # ref_I  (your x180 template I)
    sel_r180_Q_wf,         # ref_Q  (your x180 template Q)
    element="qubit",   # or whatever your qubit element name is
    rotations="all",
    make_r0=True,      # <-- bool, not "True"
    override=True,
    persist=True,
    d_lambda_map=d_lambda_map,
    d_alpha_map=d_alpha_map,
    d_omega_map=d_omega_map,
    prefix="sel_"
)

experiment.burn_pulses()
experiment.save_pulses()



### cavity spectroscopy

In [ ]:
rf_begin, rf_end = 5235e6, 5245e6
df = 100e3
runres = experiment.storage_spectroscopy("const_disp", rf_begin, rf_end, df, int(400*u.us), n_avg=100)

frequencies, S = runres.output.extract("frequencies", "S")
custom_plot_opts = {"figsize": (10, 6), "xlabel": "frequency (MHz)", "ylabel": "amplitude", "title": "Storage Spectroscopy Fit", "legend_fontsize": 16}
fit_prams = fitting.generalized_fit(frequencies*1e-6, S.real, cQED_models.resonator_spec_model, [5241, -7,-1, 1], plotting=True, plot_options=custom_plot_opts)

### storage ramsey

In [ ]:
delay_ticks = np.arange(4, 0.01e6, 0.04e3)
st_detune = 0.05
runres = experiment.storage_ramsey(delay_ticks, st_detune*u.MHz, "const_alpha", "sel_x180", n_avg=1000)
delay_ticks, states = runres.output.extract("delay_ticks", "S")
fit_p0 = [1, 300, 1, st_detune, -1.5, 0]
delay_ticks, states = runres.output.extract("delay_ticks", "S")
lower = [-np.inf]*6
upper = [ np.inf]*6
T2_fit_params = fitting.generalized_fit(4*delay_ticks*1e-3, states.real, cQED_models.T2_ramsey_model, fit_p0, plotting=True, bounds=(lower, upper),
                plot_options={"figsize": (12, 6), r"xlabel": "delays ($\mu s$)", "ylabel": "amplitudes", "title": "T2 Ramesy", "legend_fontsize": 16}, param_format="{:.4f}")


### storage chi ramsey

In [ ]:
disp = Displacement(2.0, build=True)
experiment.quaProgMngr.burn_pulse_to_qm(experiment.pulseOpMngr)
delay_ticks = np.arange(1, 250, 2, dtype=int)
fock_0_fq = experiment.attributes.qb_fq

runres = experiment.storage_chi_ramsey(fock_0_fq, delay_ticks, disp.name, n_avg=20000)

fit_p0 = [0.5, 0.5, 35, 0.1, 2.8, 0.4]
#output = Output.from_file("data/seq_1_device/storageChiRamsey/20250602_032503.npz")
delay_ticks, states1 = runres.output.extract("delay_ticks", "S")

def chi_ramsey_model(t, P0, A, T2_eff, nbar, chi, offset):
    """Eq. (1) – t in seconds, chi in rad/s."""
    return P0 + A * np.exp(-t / T2_eff) \
               * np.exp(-2 * nbar * np.sin(0.5 * 2*np.pi* chi * (t+offset))**2)
T2_fit_params = fitting.generalized_fit(4*delay_ticks*1e-3, states1.real, cQED_models.chi_ramsey_model, fit_p0, plotting=True,
                plot_options={"figsize": (16, 6), r"xlabel": "delays ($\mu s$)", "ylabel": "amplitudes", "title": "T2 Ramesy", "legend_fontsize": 16}, param_format="{:.4f}")

### kerr ramsey  

In [ ]:
delay_ticks = np.arange(4, 0.01e6, 0.04e3)
st_detune = 0.1*u.MHz
alpha_list = np.linspace(0, 4, 50) 
base_alpha = 10
output = experiment.storage_kerr_ramsey(delay_ticks, alpha_list, st_detune, base_alpha, n_avg=100)

In [ ]:
#output = Output.from_file("data/seq_1_device/storageKerrRamsey/20250530_005906.npz")
delays, states, alphas = output.extract("delays", "S", "alphas")
plotting.plot_hm(states.real, delays*1e-3, alphas, rf"delay ($\mu s$)", r"$\alpha$", "Kerr Ramsey", "states")

## fock resolved characterization

### alpha=1 displacement pulse generation

In [ ]:

alpha_len = 48
alpha_amp = 0.019580

pom = experiment.pulseOpMngr
pom.create_control_pulse(
    element="storage",
    op="const_alpha",
    pulse_name="const_alpha_pulse",
    length=alpha_len,
    I_wf_name="const_alpha_I_wf",
    Q_wf_name="const_alpha_Q_wf",
    I_samples=alpha_amp,
    Q_samples=0.0,
    persist=True,
    override=True,
)

experiment.attributes.b_alpha = 1
experiment.attributes.b_coherent_amp = alpha_amp
experiment.attributes.b_coherent_len = alpha_len 
experiment.save_attributes()

### fock resolved spectroscopy

In [ ]:

from concurrent.futures import wait
from qubox.analysis.cQED_plottings import display_fock_populations
attr = experiment.attributes
alpha = 1
disp = Displacement(alpha=alpha, build=True)
qb_rot = QubitRotation(np.pi, 0, build=True)
idle = Idle(wait_time=2000)
N = 6
experiment.burn_pulses()
fock_fqs = attr.get_fock_frequencies(N, from_chi=False)
fock_states = range(N)
def state_prep():
    qb_rot.play()
    disp.play()
    
runres = experiment.fock_resolved_spectroscopy3(fock_fqs, state_prep=state_prep, sel_r180="sel_x180", n_avg=10000, sel_r180_transfer_calibration=False)
I0, Q0, I_sel, Q_sel = runres.output.extract("I0", "Q0", "I_sel", "Q_sel")
S0 = I0 + 1j*Q0
S_sel = I_sel + 1j*Q_sel

w0g, w1e_sel, w1e_null = runres.output.extract("w0g", "w1e_sel", "w1e_null")

Pe_given_g_sel, Pg_given_e_sel = runres.output.extract("Pe_given_g_sel", "Pg_given_e_sel")
Pe_given_g_null, Pg_given_e_null = runres.output.extract("Pe_given_g_null", "Pg_given_e_null")
Pg0, Pe0 = runres.output.extract("Pg0", "Pe0")

In [ ]:
fit_params = display_fock_populations(fock_states, (Pe_given_g_sel-0)/1 , fit_alpha=True, max_alpha=1.3)
fit_params = display_fock_populations(fock_states, (Pg_given_e_sel-0)/1 , fit_alpha=True, max_alpha=1.3)
print(fit_params)
print(Pe_given_g_sel)
print(Pg_given_e_sel)
print(Pe_given_g_sel+Pg_given_e_sel)

In [ ]:
{'alpha': 1.0037441202954522, 'lambda': 1.0075022590276912, 'scale': 0.32143394073813125, 'offset': 0.3080964537537404}

In [ ]:
scale = 0.32143394073813125
offset = 0.3080964537537404
test_y_data = Pe_given_g_sel  
Pe_given_g_sel_corr = (test_y_data-offset)/scale
Pe_given_g_sel_corr

In [ ]:
cQED_models.coherent_population_model(fock_states, alpha)

In [ ]:
I0_cal, Q0_cal = runres.output.extract("I0_cal", "Q0_cal")
S0_cal = I0_cal + 1j*Q0_cal
S0g_cal = S0_cal[:, 0]
S0e_cal = S0_cal[:, 1]

Pg_ref = measureMacro.compute_Pe_from_S(S0g_cal.mean(axis=0))
Pe_ref = measureMacro.compute_Pe_from_S(S0e_cal.mean(axis=0))
print(f"Pg_M0_cal: {Pg_ref}")
print(f"Pe_M0_cal: {Pe_ref}")
I_cal, Q_cal = runres.output.extract("I_cal", "Q_cal")
S_cal = I_cal + 1j*Q_cal
Sg_cal = S_cal[:, 0]
Se_cal = S_cal[:, 1]

Pg_ref = measureMacro.compute_Pe_from_S(Sg_cal.mean(axis=0))
Pe_ref = measureMacro.compute_Pe_from_S(Se_cal.mean(axis=0))
print(f"Pg_M1_cal: {Pg_ref}")
print(f"Pe_M1_cal: {Pe_ref}")

In [ ]:
cQED_models.coherent_population_model(fock_states, alpha)

In [ ]:
runres.output.extract("Pn_given_g")

In [ ]:
P_n = runres.output.extract("Pflip")      # or "Pflip_raw"
P_n_norm = P_n / np.nansum(P_n)
P_n_norm

In [ ]:
Pg_n = runres.output.extract("Pg_n")      # or "Pg_n_raw"
Pg_n_cond = Pg_n / np.nansum(Pg_n)
Pg_n_cond

In [ ]:
runres.output.extract("Pe_n")

In [ ]:
runres.output.extract("Pe1_given_g0")

In [ ]:
w0_g_mean = np.nanmean(runres.output.extract("w0_g"), axis=0)
w0_g_mean

In [ ]:
cQED_models.coherent_population_model(fock_states, alpha)

In [ ]:

display_fock_populations(fock_states,runres.output.extract("w1_e").mean(axis=0), fit_alpha=True, max_alpha=1.3)
ideal_pop = cQED_models.coherent_population_model(fock_states, alpha)


In [ ]:
runres.output["Sg_n_main"]

In [ ]:
display_fock_populations(fock_states, runres.output["Sg_n"], fit_alpha=True, max_alpha=1.1)

### fock resolved power rabi

In [ ]:
from qubox.gates_legacy import Displacement
N = 6
attr = experiment.attributes
fock_fqs = attr.get_fock_frequencies(N, from_chi=False)
disp_n_list = []
for n in range(N):
    disp_gate = Displacement(np.sqrt(n), build=True)
    disp_n_list.append(disp_gate.op)

experiment.burn_pulses()
sel_qb_pulse = "sel_x180"

gains = np.linspace(0, 1.2, 11)
runres = experiment.fock_resolved_power_rabi(fock_fqs, gains, sel_qb_pulse, disp_n_list, n_avg=5000)
S_n = runres.output.extract("S")
fitted_params = []
for S in S_n:
    A0, g_pi0, offset0 = max(S.real) - min(S.real), 1, 1,
    p0 = (A0, g_pi0, offset0)

    fit_params = fitting.generalized_fit(
        gains,
        S.real,
        cQED_models.power_rabi_model,
        p0,
        plotting=False
    )
    fitted_params.append(fit_params[0])

for S, n in zip(S_n, range(N)):
    A0, g_pi0, offset0 = fitted_params[n]
    plt.scatter(gains, S.real)
    plt.plot(gains, cQED_models.power_rabi_model(gains, *fitted_params[n]), "--", label=f"|{n}> fit with g_pi={fitted_params[n][1]:.3f}")
plt.title("Fock-resolved power Rabi")
plt.xlabel("drive gain")
plt.ylabel("measured signal (arb.)")
plt.legend()

### fock resolved T1 relaxation

In [ ]:
from qubox.gates_legacy import Displacement
import matplotlib.pyplot as plt

attr = experiment.attributes
fock_levels = [1, 2, 3, 4] # Define the Fock levels to measure
fock_fqs = attr.get_fock_frequencies(fock_levels, from_chi=False)
alpha_vals = [alpha_for_max_fock_population(n) for n in fock_levels]
fock_disps = [Displacement(alpha, build=True).op for alpha in alpha_vals]

delay_end = 1000 * u.us 
dt = 20 * u.us
n_avg = 4000

experiment.burn_pulses()

# Run the experiment
runres = experiment.fock_resolved_T1_relaxation(
    fock_fqs, 
    fock_disps, 
    delay_end, 
    dt, 
    delay_begin=1*u.us, 
    n_avg=n_avg
)

# Extract Data
delay_ns, S = runres.output.extract("delays", "S")
delay_us = delay_ns * 1e-3
_colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', ['C0','C1','C2','C3','C4','C5'])

plt.figure(figsize=(10, 6))

for i, n_fock in enumerate(fock_levels):
    pop_n = np.asarray(S[i].real, dtype=float)

    tail = pop_n[-max(3, len(pop_n)//10):]
    offset_guess = float(np.median(tail))
    amp_guess = float(pop_n[0] - offset_guess)
    T1_guess = float(delay_us[-1] / 3)

    p0 = [amp_guess, T1_guess, offset_guess]

    try:
        popt, pcov = fitting.generalized_fit(
            delay_us,
            pop_n,
            cQED_models.T1_relaxation_model,
            p0,
            plotting=False,
            maxfev=20000,
        )

        if popt is None:
            raise RuntimeError("generalized_fit failed (popt is None)")

        fit_A, fit_T1, fit_offset = popt
        color = _colors[i % len(_colors)]

        # Plot data
        plt.plot(
            delay_us, pop_n, 'o',
            color=color, markersize=4, alpha=0.6,
            label=fr'|{n_fock}> data'
        )

        # Plot fit curve
        fit_curve = cQED_models.T1_relaxation_model(delay_us, *popt)
        plt.plot(
            delay_us, fit_curve, '-',
            color=color, linewidth=2,
            label=fr'|{n_fock}\rangle fit: $T_1$={fit_T1:.2f} $\mu$s'
        )

        print(f"|{n_fock}> T1: {fit_T1:.4f} us")

    except Exception as e:
        print(f"Fit/plot failed for Fock state |{n_fock}>: {e}")
        plt.plot(delay_us, pop_n, 'o--', alpha=0.5, label=fr'|{n_fock}\rangle (failed)')

plt.xlabel(r"Delay ($\mu$s)")
plt.ylabel("Signal / Population (real)")
plt.title("Fock-Resolved T1 Relaxation")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
    


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial

try:
    from scipy.optimize import least_squares
except ImportError as e:
    raise ImportError("This code needs scipy (scipy.optimize.least_squares).") from e


# ----------------------------
# Helpers
# ----------------------------
def infer_mu0_from_displacement_op(D_op: np.ndarray) -> float:
    """
    Infer mu0 = <n> produced by applying displacement operator D on vacuum:
        |psi> = D |0>, mu0 = <psi| a^† a |psi>.
    Works even if D includes small imperfections (as long as it's your model op).
    """
    D_op = np.asarray(D_op, dtype=np.complex128)
    dim = D_op.shape[0]

    # |0> basis vector
    vac = np.zeros(dim, dtype=np.complex128)
    vac[0] = 1.0

    psi = D_op @ vac  # displaced vacuum

    # number operator n_hat = diag(0,1,2,...)
    n_hat = np.diag(np.arange(dim, dtype=np.float64))
    mu0 = float(np.real(np.vdot(psi, n_hat @ psi)))
    return mu0


def poisson_pn(mu: np.ndarray, n: int) -> np.ndarray:
    """Poisson probability p_n(mu) for array mu."""
    # avoid numerical issues at mu ~ 0
    mu = np.clip(mu, 0.0, None)
    return np.exp(-mu) * (mu**n) / factorial(n)


def model_signal(t_us: np.ndarray, n: int, mu0: float, kappa_per_us: float, A: float, C: float) -> np.ndarray:
    """
    S(t) = A * p_n(mu0 * exp(-kappa t)) + C
    """
    mu_t = mu0 * np.exp(-kappa_per_us * t_us)
    return A * poisson_pn(mu_t, n) + C


# ----------------------------
# Build data arrays from your run
# ----------------------------
delay_ns, S = runres.output.extract("delays", "S")
delay_us = np.asarray(delay_ns, dtype=float) * 1e-3

# your chosen fock levels and displacement ops
# fock_levels = [1,2,3,4]
# fock_disps  = [Displacement(alpha, build=True).op for alpha in alpha_vals]

# make signals y_i(t)
y_list = [np.asarray(S[i].real, dtype=float) for i in range(len(fock_levels))]

# infer mu0_i from each displacement operator (recommended)
mu0_list = [infer_mu0_from_displacement_op(D) for D in fock_disps]

print("Inferred mu0 from displacement ops:")
for n, mu0 in zip(fock_levels, mu0_list):
    print(f"  target n={n}: mu0={mu0:.4f}  (expected ~{n} if Poisson-optimized)")

# ----------------------------
# Global fit: parameters = [kappa, A0..A_{M-1}, C0..C_{M-1}]
# ----------------------------
M = len(fock_levels)

def residuals(params: np.ndarray) -> np.ndarray:
    kappa = params[0]  # 1/us
    A = params[1:1+M]
    C = params[1+M:1+2*M]

    res = []
    for i in range(M):
        n = int(fock_levels[i])
        mu0 = float(mu0_list[i])
        yhat = model_signal(delay_us, n, mu0, kappa, A[i], C[i])
        res.append(y_list[i] - yhat)
    return np.concatenate(res)

# Initial guesses:
# kappa guess from your earlier ~300 us => kappa ~ 1/300 1/us
kappa0 = 1.0 / 300.0

A0 = []
C0 = []
for y in y_list:
    tail = np.median(y[-max(3, len(y)//10):])
    C0.append(float(tail))
    A0.append(float(max(y[0] - tail, 1e-6)))
A0 = np.asarray(A0, dtype=float)
C0 = np.asarray(C0, dtype=float)

p0 = np.concatenate([[kappa0], A0, C0])

# Bounds:
# kappa > 0, A free-ish but typically positive, C free
lower = np.concatenate([[0.0], np.full(M, -np.inf), np.full(M, -np.inf)])
upper = np.concatenate([[np.inf], np.full(M,  np.inf), np.full(M,  np.inf)])

fit = least_squares(residuals, p0, bounds=(lower, upper), max_nfev=20000)

kappa_fit = fit.x[0]
A_fit = fit.x[1:1+M]
C_fit = fit.x[1+M:1+2*M]

T1_cav_us = 1.0 / kappa_fit

print("\n=== Global Poisson-decay fit (corrects upper feeding) ===")
print(f"kappa = {kappa_fit:.6g} 1/us")
print(f"T1_cav = {T1_cav_us:.3f} us")
for n in [1,2,3,4]:
    print(f"Inferred pure |{n}> lifetime (if prepared as a true Fock state): T1_cav/n = {T1_cav_us/n:.3f} us")

# ----------------------------
# Plot
# ----------------------------
_colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', ['C0','C1','C2','C3','C4','C5'])
plt.figure(figsize=(10, 6))

for i, n in enumerate(fock_levels):
    y = y_list[i]
    yhat = model_signal(delay_us, int(n), float(mu0_list[i]), kappa_fit, A_fit[i], C_fit[i])
    color = _colors[i % len(_colors)]

    plt.plot(delay_us, y, 'o', color=color, markersize=4, alpha=0.6, label=fr'|{n}\rangle data')
    plt.plot(delay_us, yhat, '-', color=color, linewidth=2,
             label=fr'|{n}\rangle model (Poisson), $\mu_0$={mu0_list[i]:.2f}')

plt.xlabel(r"Delay ($\mu$s)")
plt.ylabel("Signal")
plt.title(fr"Fock-resolved decay fit with coherent-state feeding model: $T_{{1,cav}}$={T1_cav_us:.1f} $\mu$s")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### fock resolved T2 ramsey

In [ ]:
attr = experiment.attributes
fock_levels = [0,1,2,3,4,5,6,7]
fock_fqs = attr.get_fock_frequencies(fock_levels, from_chi=False)
fixed_detunings = 0.5e6
detunings = [fixed_detunings]*len(fock_levels)
alphas = [alpha_for_max_fock_population(n) for n in fock_levels]
disps = [Displacement(alpha, build=True).op for alpha in alphas]
delay_end = 4*u.us
dt = 100*u.ns
n_avg = 35000
experiment.burn_pulses()
runres = experiment.fock_resolved_qb_ramsey(fock_fqs, detunings, disps, delay_end, dt, n_avg=n_avg)
delays_ns, S = runres.output.extract("delays", "S")


In [ ]:
delays_us = delays_ns * 1e-3  # ns -> µs
fitted_detunings = []
for i, n in enumerate(fock_levels):
    fit_p0 = [0, 7, 1, fixed_detunings*1e-6, 0.4, 0]
    T2_fit_params = fitting.generalized_fit(
        delays_us,
        S[i].real,
        cQED_models.T2_ramsey_model,
        fit_p0,
        plotting=True,
        plot_options={
            "figsize": (10, 3),
            "xlabel": r"delays ($\mu s$)",
            "ylabel": "amplitudes",
            "title": f"T2 Ramsey condition on |{n}>",
            "legend_fontsize": 16,
        },
        param_format="{:.6f}",
    )
    A, T2_us, n, fitted_det_MHz, phi, offset = T2_fit_params[0]
    fitted_detunings.append(fitted_det_MHz*1e6)

In [ ]:
attr = experiment.attributes
for i, n in enumerate(fock_levels):
    fitted_detuning_n = fitted_detunings[i]
    imposed_detuning_n = detunings[i]
    correction = (imposed_detuning_n - fitted_detuning_n)
    print(f"correctioins |{n}>: {correction*1e-3} KHz")
    attr.fock_fqs[i] = attr.fock_fqs[i] + correction
experiment.save_attributes()

In [ ]:
attr = experiment.attributes
fock_fqs = attr.fock_fqs
n_list = range(len(fock_fqs))
p0 = [fock_fqs[0], attr.st_chi, 0, 0]
plot_options={
    "figsize": (10, 3),
},
model = cQED_models.number_split_frequency_model
fitting_params = fitting.generalized_fit(n_list, fock_fqs, model, p0, plotting=True,
        plot_options={
            "figsize": (10, 3),
            "xlabel": r"delays ($\mu s$)",
            "ylabel": "amplitudes",
            "title": f"T2 Ramsey condition on |{0}>",
            "legend_fontsize": 16,
        },
        param_format="{:.4f}",)
popt, pcov = fitting_params
fit_fq0, fit_chi, fit_chi2, fit_chi3 = popt
perr = np.sqrt(np.diag(pcov))
print(f"Fitted fock 0 frequency: {fit_fq0} ± {perr[0]}")
print(f"Fitted chi: {fit_chi} ± {perr[1]}")
print(f"Fitted chi2: {fit_chi2} ± {perr[2]}")
print(f"Fitted chi3: {fit_chi3} ± {perr[3]}")

#attr.st_chi = fit_chi
#attr.st_chi2 = fit_chi2
#attr.st_chi3 = fit_chi3
#experiment.save_attributes()

### fock resolved state tomography calibrations

In [ ]:
import numpy as np
from math import factorial

# ============================================================
# Helpers: fit affine map and apply correction
#   Fit:   s_meas ≈ A @ s_true + b
#   Corr:  s_corr = pinv(A) @ (s_meas - b)
# ============================================================

def _fit_affine_A_b(S_true, S_meas):
    S_true = np.asarray(S_true, float)
    S_meas = np.asarray(S_meas, float)
    K = S_true.shape[0]
    if S_true.shape != (K, 3) or S_meas.shape != (K, 3):
        raise ValueError(f"S_true and S_meas must be (K,3). Got {S_true.shape}, {S_meas.shape}")

    X = np.hstack([S_true, np.ones((K, 1))])   # (K,4)
    Y = S_meas                                 # (K,3)
    W, *_ = np.linalg.lstsq(X, Y, rcond=None)  # (4,3)

    A = W[:3, :].T                             # (3,3)
    b = W[3, :].copy()                         # (3,)
    return A, b

def _apply_affine_correction(s_meas, A, b, rcond=1e-12):
    s_meas = np.asarray(s_meas, float).reshape(3,)
    A = np.asarray(A, float).reshape(3, 3)
    b = np.asarray(b, float).reshape(3,)
    Ainv = np.linalg.pinv(A, rcond=rcond)
    return Ainv @ (s_meas - b)

def _normalize_dir(v, eps=1e-12):
    v = np.asarray(v, float).reshape(3,)
    n = np.linalg.norm(v)
    if n < eps:
        return np.array([np.nan, np.nan, np.nan], float)
    return v / n

def _fmt_vec(v, nd=4):
    v = np.asarray(v, float).reshape(3,)
    return f"({v[0]: .{nd}f}, {v[1]: .{nd}f}, {v[2]: .{nd}f})"


# ============================================================
# Coherent-state photon-number probability P(n)
#   P(n) = exp(-|α|^2) |α|^{2n} / n!
# Optionally renormalize over a finite set (e.g., the fitted n’s).
# ============================================================

def coherent_Pn(alpha, n, *, renorm_over=None):
    """
    Poisson P(n) for coherent state alpha.
    If renorm_over is an iterable of n's, renormalize so sum_{m in renorm_over} P(m)=1.
    """
    mu = float(np.abs(alpha) ** 2)
    n = int(n)
    Pn = np.exp(-mu) * (mu ** n) / factorial(n)

    if renorm_over is None:
        return float(Pn)

    Z = 0.0
    for m in renorm_over:
        m = int(m)
        Z += np.exp(-mu) * (mu ** m) / factorial(m)

    return float(Pn / Z) if Z > 0 else 0.0


# ============================================================
# 6 qubit states (your existing prep table)
# These are UNIT Bloch vectors r_true (conditioned).
# For your experiment outputs v_n = <σ Π_n>, the ideal target is:
#    v_true(n) = P(n) * r_true
# ============================================================

IDEAL_TRUE = {
    "g":  np.array([0.0,  0.0,  +1.0]),
    "e":  np.array([0.0,  0.0,  -1.0]),
    "+x": np.array([+1.0, 0.0,  0.0]),
    "-x": np.array([-1.0, 0.0,  0.0]),
    "+y": np.array([0.0,  +1.0, 0.0]),
    "-y": np.array([0.0,  -1.0, 0.0]),
}

STATE_PULSES = {
    "g":  [],
    "e":  [("x180", "qubit")],      # or ("y180","qubit")
    "+x": [("y90",  "qubit")],
    "-x": [("yn90", "qubit")],
    "+y": [("xn90", "qubit")],
    "-y": [("x90",  "qubit")],
}

STATE_ORDER_DEFAULT = ["g", "e", "+x", "-x", "+y", "-y"]


# ============================================================
# PART A: acquire raw 6-state tomo data (NO fitting here)
# ============================================================

def acquire_6state_tomo_data(
    *,
    experiment,
    fock_levels,
    states=STATE_ORDER_DEFAULT,
    n_avg=5000,
    sel_r180="sel_x180",
    from_chi=False,
    do_burn_pulses=True,
):
    """
    Returns dict with raw measured arrays for each state:
      data["meas"][state]["sxyz_n"] has shape (len(fock_levels), 3)

    NOTE: this assumes your experiment returns UNNORMALIZED vectors:
      sxyz_n[n] = (<σx Π_n>, <σy Π_n>, <σz Π_n>)   (up to conventions)
    """
    from qm.qua import play

    fock_levels = list(fock_levels)
    states = list(states)

    fock_fqs = experiment.attributes.get_fock_frequencies(fock_levels, from_chi=from_chi)

    if do_burn_pulses:
        experiment.burn_pulses()

    meas = {}

    for name in states:
        if name not in STATE_PULSES:
            raise ValueError(f"Unknown state '{name}'. Allowed: {list(STATE_PULSES.keys())}")

        pulses = STATE_PULSES[name]

        def state_prep(pulses=pulses):
            for pul, chan in pulses:
                play(pul, chan)

        runres = experiment.fock_resolved_state_tomography(
            fock_fqs=fock_fqs,
            state_prep=state_prep,
            n_avg=n_avg,
            sel_r180=sel_r180
        )

        sx_n, sy_n, sz_n = runres.output.extract("sigma_x_n", "sigma_y_n", "sigma_z_n")
        sx_n = np.asarray(sx_n, float)
        sy_n = np.asarray(sy_n, float)
        sz_n = np.asarray(sz_n, float)

        if sx_n.shape[0] != len(fock_levels):
            raise RuntimeError(
                f"State {name}: expected len={len(fock_levels)} but got {sx_n.shape[0]}."
            )

        sxyz_n = np.stack([sx_n, sy_n, sz_n], axis=1)
        meas[name] = dict(sx_n=sx_n, sy_n=sy_n, sz_n=sz_n, sxyz_n=sxyz_n)

    return dict(
        meta=dict(n_avg=n_avg, sel_r180=sel_r180, from_chi=from_chi),
        states=states,
        fock_levels=fock_levels,
        meas=meas,
    )


# ============================================================
# Acquire raw 6-state tomo data WITH coherent cavity prep
#   - uses Displacement(alpha, build=True)
#   - in state_prep: disp.play() then qubit pulses
#   - burns pulses ONCE per call (for a single alpha, do_burn_pulses=True once)
# ============================================================

def acquire_6state_tomo_data_with_coherent(
    *,
    experiment,
    fock_levels,
    alpha=1.0,
    states=STATE_ORDER_DEFAULT,
    n_avg=10000,
    sel_r180="sel_x180",
    from_chi=False,
    do_burn_pulses=True,
):
    """
    Same as acquire_6state_tomo_data, but prepends a cavity displacement.

    Requires your environment has:
      disp = Displacement(alpha, build=True)
      disp.play()

    NOTE: You said: if we add a new displacement pulse, we need burn_pulses.
    This function burns ONCE at the beginning (good for fixed alpha).
    """
    from qm.qua import play

    # You said you have this class available in your codebase.
    # If it's in another module, import it above or here.
    disp = Displacement(alpha, build=True)

    fock_levels = list(fock_levels)
    states = list(states)

    fock_fqs = experiment.attributes.get_fock_frequencies(fock_levels, from_chi=from_chi)

    if do_burn_pulses:
        experiment.burn_pulses()

    meas = {}

    for name in states:
        if name not in STATE_PULSES:
            raise ValueError(f"Unknown state '{name}'. Allowed: {list(STATE_PULSES.keys())}")

        pulses = STATE_PULSES[name]

        def state_prep(pulses=pulses):
            # 1) cavity coherent prep
            disp.play()

            # 2) qubit prep
            for pul, chan in pulses:
                play(pul, chan)

        runres = experiment.fock_resolved_state_tomography(
            fock_fqs=fock_fqs,
            state_prep=state_prep,
            n_avg=n_avg,
            sel_r180=sel_r180,
        )

        sx_n, sy_n, sz_n = runres.output.extract("sigma_x_n", "sigma_y_n", "sigma_z_n")
        sx_n = np.asarray(sx_n, float)
        sy_n = np.asarray(sy_n, float)
        sz_n = np.asarray(sz_n, float)

        if sx_n.shape[0] != len(fock_levels):
            raise RuntimeError(
                f"State {name}: expected len={len(fock_levels)} but got {sx_n.shape[0]}."
            )

        sxyz_n = np.stack([sx_n, sy_n, sz_n], axis=1)
        meas[name] = dict(sx_n=sx_n, sy_n=sy_n, sz_n=sz_n, sxyz_n=sxyz_n)

    return dict(
        meta=dict(
            n_avg=n_avg,
            sel_r180=sel_r180,
            from_chi=from_chi,
            alpha=float(alpha),
        ),
        states=states,
        fock_levels=fock_levels,
        meas=meas,
    )


# ============================================================
# PART B: fit A,b for ONE chosen fock level (offline)
#
# IMPORTANT for your case:
#   - Your measured vectors are v_n = <σ Π_n> (NOT conditioned).
#   - So the ideal truth for coherent prep should be v_true(n)=P(n)*r_true.
#   - That’s what use_Pn_scaling does.
# ============================================================

def fit_affine_for_one_fock_level(
    data,
    *,
    target_fock_level=0,
    ideal_true=IDEAL_TRUE,
    rcond_pinv=1e-12,
    require_full_rank=True,

    # NEW (key):
    use_Pn_scaling=False,
    Pn_value=None,          # if None, try meta['alpha'] with coherent_Pn
    Pn_renorm_over=None,    # e.g. [0,1,2] when you only calibrate those
):
    states = list(data["states"])
    fock_levels = list(data["fock_levels"])
    meas = data["meas"]
    meta = data.get("meta", {})

    if target_fock_level not in fock_levels:
        raise ValueError(f"target_fock_level={target_fock_level} not in acquired fock_levels={fock_levels}")

    idx = fock_levels.index(target_fock_level)

    # (K,3) unit Bloch vectors r_true
    S_true = np.stack([ideal_true[s] for s in states], axis=0)

    # If measured is <σ Π_n>, and cavity is coherent with known P(n), scale truth by P(n)
    if use_Pn_scaling:
        if Pn_value is None:
            if "alpha" not in meta:
                raise ValueError("use_Pn_scaling=True needs Pn_value or data['meta']['alpha'].")
            alpha = meta["alpha"]
            Pn_value = coherent_Pn(alpha, target_fock_level, renorm_over=Pn_renorm_over)
        S_true = float(Pn_value) * S_true  # now represents v_true(n)=P(n)*r_true

    # (K,3) measured vectors v_meas(n)
    S_meas = np.stack([meas[s]["sxyz_n"][idx, :] for s in states], axis=0)

    A, b = _fit_affine_A_b(S_true, S_meas)

    # diagnostics
    svals = np.linalg.svd(A, compute_uv=False)
    cond = float(svals[0] / max(svals[-1], 1e-30))
    rank = int(np.sum(svals > (svals[0] * 1e-12)))
    pred = (S_true @ A.T) + b.reshape(1, 3)
    rms = float(np.sqrt(np.mean((S_meas - pred) ** 2)))

    if require_full_rank and rank < 3:
        raise RuntimeError(
            f"Fit produced rank(A)={rank} (<3) at fock n={target_fock_level}. "
            f"Try require_full_rank=False to inspect results."
        )

    meas_raw = {s: meas[s]["sxyz_n"][idx, :].copy() for s in states}
    meas_corr = {s: _apply_affine_correction(meas_raw[s], A, b, rcond=rcond_pinv) for s in states}

    # For convenience: store the (possibly scaled) truth used in the fit
    true = {s: (S_true[states.index(s)].copy()) for s in states}

    # Direction diagnostics (direction-only is what you care about)
    dir_raw  = {s: _normalize_dir(meas_raw[s]) for s in states}
    dir_corr = {s: _normalize_dir(meas_corr[s]) for s in states}
    dir_true = {s: _normalize_dir(true[s]) for s in states}

    return dict(
        n=int(target_fock_level),
        A=A,
        b=b,
        meas_raw=meas_raw,
        meas_corr=meas_corr,
        true=true,
        dir_raw=dir_raw,
        dir_corr=dir_corr,
        dir_true=dir_true,
        diagnostics=dict(rms_fit=rms, condA=cond, rankA=rank, svals=svals),
        meta=meta,
        Pn_value=(float(Pn_value) if use_Pn_scaling else None),
        Pn_renorm_over=(list(Pn_renorm_over) if Pn_renorm_over is not None else None),
    )


# ============================================================
# Fit A_n,b_n for multiple fock levels
# ============================================================

def fit_affine_for_multiple_fock_levels(
    data,
    *,
    target_fock_levels=(0, 1, 2),
    ideal_true=IDEAL_TRUE,
    rcond_pinv=1e-12,
    require_full_rank=True,
    use_Pn_scaling=False,
    Pn_renorm_over=None,
):
    cal_by_n = {}
    for n in target_fock_levels:
        cal_by_n[int(n)] = fit_affine_for_one_fock_level(
            data,
            target_fock_level=int(n),
            ideal_true=ideal_true,
            rcond_pinv=rcond_pinv,
            require_full_rank=require_full_rank,
            use_Pn_scaling=use_Pn_scaling,
            Pn_value=None,
            Pn_renorm_over=Pn_renorm_over,
        )
    return cal_by_n


# ============================================================
# Pretty print
# ============================================================

def print_one_fock_calibration(cal, states=STATE_ORDER_DEFAULT):
    n = cal["n"]
    A, b = cal["A"], cal["b"]
    diag = cal["diagnostics"]
    meta = cal.get("meta", {})

    print("=" * 80)
    hdr = f"Affine calibration for fock n = {n}"
    if "alpha" in meta:
        hdr += f" (coherent alpha={meta['alpha']})"
    if cal.get("Pn_value") is not None:
        hdr += f"  [Pn_used={cal['Pn_value']:.6f}]"
    print(hdr)

    print("A =")
    print(A)
    print("b =", b)
    print(f"Diagnostics: rms_fit={diag['rms_fit']:.3e}, rankA={diag['rankA']}, condA={diag['condA']:.3e}")

    print("\nRaw vs Corrected vs True-used-in-fit (v_n = <σ Π_n>):")
    for s in states:
        if s not in cal["meas_raw"]:
            continue
        raw = cal["meas_raw"][s]
        corr = cal["meas_corr"][s]
        tru = cal["true"][s]
        print(f"  {s:>2s}: raw {_fmt_vec(raw)} | corr {_fmt_vec(corr)} | true-fit {_fmt_vec(tru)}")

    print("\nDirection-only (normalized):")
    for s in states:
        if s not in cal["dir_raw"]:
            continue
        dr = cal["dir_raw"][s]
        dc = cal["dir_corr"][s]
        dt = cal["dir_true"][s]
        print(f"  {s:>2s}: dir_raw {_fmt_vec(dr)} | dir_corr {_fmt_vec(dc)} | dir_true {_fmt_vec(dt)}")

def print_multi_fock_calibration(cal_by_n, states=STATE_ORDER_DEFAULT):
    for n in sorted(cal_by_n.keys()):
        print_one_fock_calibration(cal_by_n[n], states=states)


# ============================================================
# Apply {A_n,b_n} to a new measured sxyz_n array
# - returns corrected unnormalized vectors v_corr(n)
# - optionally returns corrected directions per n (normalize at end)
# ============================================================

def apply_calibration_to_measured_sxyz_n(
    sxyz_n,
    cal_by_n,
    fock_levels,
    *,
    rcond_pinv=1e-12,
    return_dirs=False,
):
    """
    sxyz_n: array shape (len(fock_levels), 3),
            assumed to be v_meas(n) = <σ Π_n> (unnormalized).
    cal_by_n: dict n -> calibration dict containing A,b
    """
    sxyz_n = np.asarray(sxyz_n, float)
    fock_levels = list(fock_levels)

    if sxyz_n.shape != (len(fock_levels), 3):
        raise ValueError(f"Expected sxyz_n shape {(len(fock_levels),3)} but got {sxyz_n.shape}")

    out = np.zeros_like(sxyz_n, dtype=float)
    out_dir = np.zeros_like(sxyz_n, dtype=float) if return_dirs else None

    for i, n in enumerate(fock_levels):
        n = int(n)
        if n not in cal_by_n:
            raise KeyError(f"No calibration for n={n}. Available: {sorted(cal_by_n.keys())}")

        A = cal_by_n[n]["A"]
        b = cal_by_n[n]["b"]
        out[i, :] = _apply_affine_correction(sxyz_n[i, :], A, b, rcond=rcond_pinv)
        if return_dirs:
            out_dir[i, :] = _normalize_dir(out[i, :])

    return (out, out_dir) if return_dirs else out


# ============================================================
# Example usage:
#   - coherent-state calibration using alpha=1
#   - calibrate up to n=2 => three sets (A_0,b_0), (A_1,b_1), (A_2,b_2)
#   - burn_pulses only once for this alpha
#
# IMPORTANT: use_Pn_scaling=True so the truth matches v_n=<σ Π_n>
# ============================================================


# You asked: calibrate up to n=2
fock_levels = [0, 1, 2]

# 1) Acquire once with coherent prep alpha=1
data_alpha1 = acquire_6state_tomo_data_with_coherent(
    experiment=experiment,
    fock_levels=fock_levels,
    alpha=1.0,
    states=STATE_ORDER_DEFAULT,
    n_avg=10000,
    sel_r180="sel_x180",
    from_chi=False,
    do_burn_pulses=True,   # burns ONCE for this alpha
)




In [ ]:
cal_by_n = fit_affine_for_multiple_fock_levels(
    data_alpha1,
    target_fock_levels=(0, 1, 2),
    use_Pn_scaling=True,
    Pn_renorm_over=[0, 1, 2],
    require_full_rank=True,
    rcond_pinv=1e-12,
)

print(cal_by_n)
# ============================================================
# Residual plots: (ideal - raw) and (ideal - corrected)
# Works for ONE calibration dict cal_n (for a specific n)
# ============================================================

def plot_residuals_raw_vs_corrected(
    cal_n,
    *,
    states=("g", "e", "+x", "-x", "+y", "-y"),
    title_prefix=None,
    show=True,
):
    """
    cal_n must have:
      cal_n["meas_raw"][state]  -> (3,)
      cal_n["meas_corr"][state] -> (3,)
      cal_n["true"][state]      -> (3,)
      cal_n["n"] optional
    """
    states = list(states)
    n = cal_n.get("n", None)
    if title_prefix is None:
        title_prefix = f"Fock n={n}" if n is not None else "Residuals"

    # Stack
    raw  = np.stack([cal_n["meas_raw"][s]  for s in states], axis=0)   # (K,3)
    corr = np.stack([cal_n["meas_corr"][s] for s in states], axis=0)   # (K,3)
    tru  = np.stack([cal_n["true"][s]      for s in states], axis=0)   # (K,3)

    # residuals you requested
    res_raw  = tru - raw
    res_corr = tru - corr

    x = np.arange(len(states))
    comps = ["X", "Y", "Z"]

    # ----------------------------
    # (A) Component-wise residuals (3 subplots)
    # ----------------------------
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharex=True, sharey=False)

    for ci, ax in enumerate(axes):
        ax.axhline(0, linewidth=1, alpha=0.3)

        ax.plot(x, res_raw[:, ci],  marker="o", linestyle="-",  label="ideal - raw")
        ax.plot(x, res_corr[:, ci], marker="s", linestyle="--", label="ideal - corrected")

        ax.set_title(f"Residual in {comps[ci]}")
        ax.set_xticks(x)
        ax.set_xticklabels(states, rotation=45, ha="right")
        ax.grid(True, alpha=0.3)

        # symmetric y-lims per component
        m = np.nanmax(np.abs(np.r_[res_raw[:, ci], res_corr[:, ci]]))
        if np.isfinite(m) and m > 0:
            ax.set_ylim(-1.1 * m, 1.1 * m)
        else:
            ax.set_ylim(-0.1, 0.1)

    axes[0].set_ylabel("Residual (ideal - measured)")
    axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2)
    fig.suptitle(f"{title_prefix}: Residuals per component", y=1.05)
    plt.tight_layout()
    if show:
        plt.show()

    # ----------------------------
    # (B) Residual vector magnitude per state
    # ----------------------------
    mag_raw  = np.linalg.norm(res_raw,  axis=1)
    mag_corr = np.linalg.norm(res_corr, axis=1)

    plt.figure(figsize=(7.2, 3.8))
    plt.axhline(0, linewidth=1, alpha=0.3)
    plt.plot(x, mag_raw,  marker="o", linestyle="-",  label="||ideal - raw||")
    plt.plot(x, mag_corr, marker="s", linestyle="--", label="||ideal - corrected||")
    plt.xticks(x, states, rotation=45, ha="right")
    plt.ylabel("Residual magnitude")
    plt.title(f"{title_prefix}: Residual magnitude per state")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    if show:
        plt.show()

    # ----------------------------
    # (C) Improvement factor: mag_raw / mag_corr
    # ----------------------------
    with np.errstate(divide="ignore", invalid="ignore"):
        improve = mag_raw / mag_corr

    plt.figure(figsize=(7.2, 3.8))
    plt.axhline(1.0, linewidth=1, alpha=0.3)
    plt.plot(x, improve, marker="o", linestyle="-")
    plt.xticks(x, states, rotation=45, ha="right")
    plt.ylabel("Improvement factor  (||ideal-raw|| / ||ideal-corr||)")
    plt.title(f"{title_prefix}: How much correction helped (>1 is good)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if show:
        plt.show()

    return dict(
        n=n,
        res_raw=res_raw,
        res_corr=res_corr,
        mag_raw=mag_raw,
        mag_corr=mag_corr,
        improve=improve,
        states=states,
    )


# ============================================================
# Convenience: plot for each n in cal_by_n
#   cal_by_n is dict like {0: cal_n0, 1: cal_n1, 2: cal_n2, ...}
# ============================================================

def plot_results_for_each_n(
    cal_by_n,
    *,
    ns=None,
    states=("g", "e", "+x", "-x", "+y", "-y"),
):
    """
    For each n, makes the same 3 plot groups as your template.
    Returns a dict n -> plot_data (residual arrays, magnitudes, etc.)
    """
    if ns is None:
        ns = sorted(cal_by_n.keys())
    out = {}
    for n in ns:
        cal_n = cal_by_n[int(n)]
        out[int(n)] = plot_residuals_raw_vs_corrected(
            cal_n,
            states=states,
            title_prefix=f"Fock n={cal_n.get('n', n)}",
            show=True,
        )
    return out


_ = plot_results_for_each_n(
    cal_by_n,
ns=[0, 1, 2],
    states=("g", "e", "+x", "-x", "+y", "-y"),
)

print_multi_fock_calibration(cal_by_n)
# Build the affine_n dictionary in the format expected by measureMacro
affine_n_dict = {}
for n, cal_data in cal_by_n.items():
    affine_n_dict[str(n)] = {
        "A": cal_data["A"],  # 3x3 matrix
        "b": cal_data["b"]   # 3-element vector
    }

out = {"affine_n": affine_n_dict}
measureMacro._update_readout_quality(out)

print("Saved affine transformations for Fock levels:", list(affine_n_dict.keys()))
print("\nVerify saved data:")
saved_affine = measureMacro._ro_quality_params.get("affine_n")
if saved_affine:
    for n in sorted(saved_affine.keys(), key=lambda x: int(x) if x.isdigit() else x):
        print(f"\nn = {n}:")
        print(f"  A shape: {saved_affine[n]['A'].shape}")
        print(f"  b shape: {saved_affine[n]['b'].shape}")

experiment.save_measureMacro_state()

### frst

In [ ]:
import matplotlib.pyplot as plt
from qm.qua import play, align
# --- 1. Settings ---
n_avg = 5000
alpha_target = 1

disp = Displacement(alpha_target, build=True)

experiment.burn_pulses()
def state_prep():
    disp.play()
    
fock_levels = [0,1,2]
fock_fqs = experiment.attributes.get_fock_frequencies(fock_levels, from_chi=False)
# --- 4. Run Experiment ---
print(f"Running Fock-resolved Tomography for alpha ~ {alpha_target}...")
runres = experiment.fock_resolved_state_tomography(
    fock_fqs=fock_fqs,
    state_prep=state_prep,
    n_avg=n_avg,
    sel_r180="sel_x180",
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# BAR-PLOT VISUALIZATION (fixed y-lims for components)
#   - Component plots (x,y,z): y-axis fixed to [-1, 1]
#   - Direction-only plots:    y-axis fixed to [-1, 1]
#   - Delta plots:             auto (since delta can be outside [-1,1])
#   - Magnitude plot:          auto
# ============================================================

# ----------------------------
# User inputs
# ----------------------------
fock_levels = [0, 1, 2]   # MUST match your run
rcond_pinv = 1e-12
YLIM_SIGMA = (-1.0, 1.0)  # requested fixed y-axis for sigma components

# ----------------------------
# Helpers
# ----------------------------
def _pinv3(A, rcond=1e-12):
    return np.linalg.pinv(np.asarray(A, float).reshape(3, 3), rcond=rcond)

def _apply_affine(v_meas, A, b, rcond=1e-12):
    v_meas = np.asarray(v_meas, float).reshape(3,)
    A = np.asarray(A, float).reshape(3, 3)
    b = np.asarray(b, float).reshape(3,)
    return _pinv3(A, rcond=rcond) @ (v_meas - b)

def _normalize_dir(v, eps=1e-12):
    v = np.asarray(v, float).reshape(3,)
    n = np.linalg.norm(v)
    if n < eps:
        return np.array([np.nan, np.nan, np.nan], float)
    return v / n

def _assert_affine_shapes(affine_n_dict, fock_levels):
    for n in fock_levels:
        key = str(int(n))
        if key not in affine_n_dict:
            raise KeyError(
                f"Missing affine calibration for n={n}. "
                f"Keys available: {list(affine_n_dict.keys())}"
            )
        A = np.asarray(affine_n_dict[key]["A"])
        b = np.asarray(affine_n_dict[key]["b"])
        if A.shape != (3, 3):
            raise ValueError(f"affine_n[{key}]['A'] must be (3,3), got {A.shape}")
        if b.shape != (3,):
            raise ValueError(f"affine_n[{key}]['b'] must be (3,), got {b.shape}")

def _grouped_bars(ax, x, yA, yB, *, width=0.35, labelA="raw", labelB="corrected"):
    ax.bar(x - width/2, yA, width, label=labelA)
    ax.bar(x + width/2, yB, width, label=labelB)

# ============================================================
# 0) Load affine_n
# ============================================================
affine_n = measureMacro._ro_quality_params["affine_n"]
_assert_affine_shapes(affine_n, fock_levels)

# ============================================================
# 1) Extract raw <σ Π_n> from runres
# ============================================================
sx, sy, sz = runres.output.extract("sigma_x_n", "sigma_y_n", "sigma_z_n")
sx = np.asarray(sx, float)
sy = np.asarray(sy, float)
sz = np.asarray(sz, float)

K = len(fock_levels)
if sx.shape[0] != K or sy.shape[0] != K or sz.shape[0] != K:
    raise ValueError(
        f"Length mismatch: got lens sx={sx.shape[0]}, sy={sy.shape[0]}, sz={sz.shape[0]} "
        f"but fock_levels len={K}"
    )

v_meas = np.column_stack([sx, sy, sz])  # (K,3) raw unnormalized <σ Π_n>

# ============================================================
# 2) Apply per-n affine correction + compute directions
# ============================================================
v_corr = np.zeros_like(v_meas)
v_raw_dir  = np.zeros_like(v_meas)
v_corr_dir = np.zeros_like(v_meas)

for i, n in enumerate(fock_levels):
    key = str(int(n))
    A_n = affine_n[key]["A"]
    b_n = affine_n[key]["b"]
    v_corr[i, :] = _apply_affine(v_meas[i, :], A_n, b_n, rcond=rcond_pinv)

    v_raw_dir[i, :]  = _normalize_dir(v_meas[i, :])
    v_corr_dir[i, :] = _normalize_dir(v_corr[i, :])

dv = v_corr - v_meas

# ============================================================
# 3) PLOTS (component y-lims fixed to [-1,1])
# ============================================================
def plot_grouped_bars_components(v_raw, v_corr, fock_levels, *, title, ylims=(-1, 1)):
    fock_levels = list(fock_levels)
    x = np.arange(len(fock_levels))
    comps = ["x", "y", "z"]

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True, sharey=True)
    for ci, ax in enumerate(axes):
        _grouped_bars(ax, x, v_raw[:, ci], v_corr[:, ci], labelA="raw", labelB="corrected")
        ax.axhline(0, linewidth=1, alpha=0.3)
        ax.set_title(rf"{comps[ci]} component")
        ax.set_xticks(x)
        ax.set_xticklabels([str(n) for n in fock_levels])
        ax.set_ylim(*ylims)
        ax.grid(True, axis="y", alpha=0.3)

    axes[0].set_ylabel("value")
    axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2)
    fig.suptitle(title, y=1.05)
    plt.tight_layout()
    plt.show()

def plot_delta_bars_components(dv, fock_levels, *, title, ylims=None):
    fock_levels = list(fock_levels)
    x = np.arange(len(fock_levels))
    comps = ["x", "y", "z"]

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True)
    for ci, ax in enumerate(axes):
        ax.bar(x, dv[:, ci], width=0.55)
        ax.axhline(0, linewidth=1, alpha=0.3)
        ax.set_title(rf"{comps[ci]} component")
        ax.set_xticks(x)
        ax.set_xticklabels([str(n) for n in fock_levels])
        if ylims is not None:
            ax.set_ylim(*ylims)
        ax.grid(True, axis="y", alpha=0.3)

    axes[0].set_ylabel("corrected - raw")
    fig.suptitle(title, y=1.05)
    plt.tight_layout()
    plt.show()

def plot_grouped_bars_magnitude(v_raw, v_corr, fock_levels, *, title):
    fock_levels = list(fock_levels)
    x = np.arange(len(fock_levels))
    mag_raw = np.linalg.norm(v_raw, axis=1)
    mag_cor = np.linalg.norm(v_corr, axis=1)

    plt.figure(figsize=(7.0, 3.8))
    _grouped_bars(plt.gca(), x, mag_raw, mag_cor, width=0.35, labelA="raw", labelB="corrected")
    plt.axhline(0, linewidth=1, alpha=0.3)
    plt.xticks(x, [str(n) for n in fock_levels])
    plt.ylabel("magnitude")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# --- (A) Raw vs corrected <σ Π_n> component-wise (FIXED y-axis) ---
plot_grouped_bars_components(
    v_meas, v_corr, fock_levels,
    title=r"Raw vs corrected unnormalized $\langle \sigma_a \Pi_n \rangle$ (components)",
    ylims=YLIM_SIGMA
)

# --- (B) Delta (corrected - raw) component-wise (auto by default) ---
plot_delta_bars_components(
    dv, fock_levels,
    title=r"Correction delta: $\langle \sigma_a \Pi_n \rangle_{\rm corr}-\langle \sigma_a \Pi_n \rangle_{\rm raw}$ (components)",
    ylims=None
)

# --- (C) Magnitudes of unnormalized vectors ---
plot_grouped_bars_magnitude(
    v_meas, v_corr, fock_levels,
    title=r"Vector magnitude $||\langle \vec{\sigma}\Pi_n\rangle||$: raw vs corrected"
)

# --- (D) Direction-only grouped bars (FIXED y-axis) ---
plot_grouped_bars_components(
    v_raw_dir, v_corr_dir, fock_levels,
    title=r"Direction-only (unit) $\hat r(n)$: raw vs corrected (components)",
    ylims=(-1.0, 1.0)
)


## Cavity State Manipulation

In [ ]:
experiment.attributes.st_chi = -2.833037135e6
experiment.attributes.st_chi2 = -23.643895e3
experiment.attributes.st_chi3 = -0.209126e3
experiment.save_attributes()

### st chi measurement (num split spec)

In [ ]:
def choose_displacements(n: int,
                         alpha_cap: float = 6.0,    # upper-bound for |α|
                         eps_abs: float  = 0.10):   # fixed |ε|  (linear regime)
    alpha_amp = np.sqrt(n + 0.5)
    alpha_amp = min(alpha_amp, alpha_cap)
    return alpha_amp, eps_abs

fock_num = 7
disp_pulses = []
for n in range(fock_num):
    alpha_amp, _ = choose_displacements(n, alpha_cap=6)
    disp = Displacement(alpha=alpha_amp, build=True)
    disp_pulses.append(disp.op)
experiment.quaProgMngr.burn_pulse_to_qm(experiment.pulseOpMngr)

attr = experiment.attributes
qb_fq = 6150.406474328e6
chi = attr.st_chi
chi2 = attr.st_chi2
rf_centers = [qb_fq + n*chi + chi2*n*(n-1) for n in range(fock_num)]

rf_centers = [6.15041468e+09, 6.14758293e+09, 6.14466950e+09, 6.14172336e+09,
 6.13876699e+09, 6.13576344e+09, 6.13266730e+09, 6.12956476e+09]
rf_centers = attr.get_fock_frequencies(fock_num, from_chi=False)
rf_spans = [0.5e6 for i in range(fock_num)]

df = 25e3
runres = experiment.num_splitting_spectroscopy(rf_centers, rf_spans, df, n_avg=10000, disp_pulses=disp_pulses)
frequencies, S, phases = runres.output.extract("frequencies", "S", "Phases")

In [ ]:
experiment.attributes.fock_fqs = [6.15040247e+09, 6.14755113e+09, 6.14470040e+09, 6.14177439e+09,
 6.13879641e+09, 6.13571374e+09, 6.13267529e+09, 6.12952045e+09]
experiment.save_attributes()

### displacement calibration

In [ ]:
from qm.qua import play, amp, align
from qubox.analysis.cQED_plottings import display_fock_populations
attr = experiment.attributes
N = 7
fock_fqs = attr.get_fock_frequencies(N, from_chi=False)
fock_levels = range(N)
gain_list = np.linspace(1, 1.9, 10)
I, Q = experiment.pulseOpMngr.get_pulse_waveforms("const_alpha_pulse", include_volatile=True)
base_alpha_amp = I
amplitudes = base_alpha_amp * gain_list
alpha_list = []
for gain in gain_list:
    def state_prep():
        play("const_alpha"*amp(gain), attr.st_el)
        align()
    runres = experiment.fock_resolved_spectroscopy(fock_fqs, state_prep=state_prep, n_avg=50000)
    fock_fqs, S = runres.output.extract("probe_fqs", "S")
    S_mean = np.mean(S, axis=0)
    fitted_alpha = display_fock_populations(fock_levels, S_mean.real, plotting=True, fit_alpha=True, normalize=False, max_alpha=gain*1+0.3)
    alpha_list.append(fitted_alpha)

p0 = [1, 0]
custom_plot_opts = {"title": "Storage coherent state amplitude calibration", "xlabel": "Coherent pulse amplitude (V)", "ylabel": "Fitted alpha"}
params = fitting.generalized_fit(gain_list*base_alpha_amp, alpha_list, models.linear_model, p0=p0, plot_options=custom_plot_opts, plotting=True)
m, b = params[0]
y = 1
x = (y - b)/m
print(f"To prepare a coherent state with alpha={y}, set the coherent pulse amplitude to {x:.6f} V")

### num_splitting_spectroscopy

In [ ]:

attr = experiment.attributes
offset = abs(3*attr.st_chi)
rf_centers = attr.qb_fq - offset
rf_spans = 4*offset
df = 200e3
disp = Displacement(alpha=1.0, build=True)
experiment.burn_pulses()
def state_prep():
    disp.play()
runres = experiment.num_splitting_spectroscopy(rf_centers, rf_spans, df, state_prep=state_prep, n_avg=5000)
frequencies, S, Phases = runres.output.extract("frequencies", "S", "Phases")
plt.plot(frequencies*1e-6, S.real)
plt.xlabel("Frequency (MHz)")
plt.ylabel("Measured signal (arb.)")
plt.title("Num-splitting spectroscopy with displacement alpha=1.0")


### SQR gate test

In [ ]:
# ...existing code...
from qubox.gates_legacy import SQR

sqr_test = SQR([np.pi,np.pi/2], [0, 0], build=True)
experiment.pulseOpMngr.display_op(sqr_test.target, sqr_test.op, freq_range=(-4,4))

### Fock |1⟩ Preparation 

In [ ]:
# ======================================================================
# FOCK STATE |1⟩ PREPARATION - Complete Analysis
# Gate structure: D-SNAP-D (3 gates total)
# ======================================================================

# Gate parameters from 2015 paper implementation
L_f1 = 1
thetas_f1 = [0 for _ in range(L_f1)]
thetas_f1[0] = -3.13587488

# Define gates
disp1_f1 = Displacement(alpha=-1.097269+1j*0.312139, build=True)
disp2_f1 = Displacement(alpha=0.559134-1j*0.159680, build=True)
snap_f1 = SNAP(thetas_f1, build=True, include_unselective=False)

# Display SNAP operation
attr = experiment.attributes
print("SNAP Gate for |1⟩ preparation:")
res_f1 = pom.display_op("qubit", snap_f1.op, time_window=[1, 2000])

# Define state preparation sequence
def state_prep_f1():
    disp1_f1.play()
    snap_f1.play()
    disp2_f1.play()

# Probe frequencies for Fock states 0-8
fock_states_f1 = np.arange(0, 9, dtype=int)
probe_fqs_f1 = attr.get_fock_frequencies(fock_states_f1, from_chi=True)

# Burn pulses to hardware
experiment.burn_pulses()

# Run Fock-resolved spectroscopy with double post-selection
print("\nRunning Fock-resolved spectroscopy for |1⟩ preparation...")
runres_f1 = experiment.fock_resolved_spectroscopy(
    probe_fqs_f1, 
    state_prep=state_prep_f1, 
    n_avg=10000
)

# Extract all results including acceptance rates
probe_fqs_f1, S_f1, Sg_n_f1, Sg_n_norm_f1, Pg_n_f1 = runres_f1.output.extract(
    "probe_fqs", "S", "Sg_n", "Sg_n_norm", "Pg_n"
)
acceptance_rate_g_f1, acceptance_rate_e_f1 = runres_f1.output.extract(
    "acceptance_rate_g", "acceptance_rate_e"
)
Pe_n_counts_f1 = runres_f1.output.get("Pe_n_counts")

# Display comprehensive results
print(f"\n{'='*70}")
print(f"FOCK STATE |1⟩ SPECTROSCOPY RESULTS")
print(f"{'='*70}")
print(f"\nPost-Selection Statistics:")
print(f"  First post-selection (ground state) mean rate:  {acceptance_rate_g_f1.mean():.2%}")
print(f"  Second post-selection (excited state) mean rate: {acceptance_rate_e_f1.mean():.2%}")
print(f"\nPost-selection rates per Fock state:")
for n, rate_g, rate_e in zip(fock_states_f1, acceptance_rate_g_f1, acceptance_rate_e_f1):
    print(f"  |{n}⟩: 1st={rate_g:.2%}, 2nd={rate_e:.2%}")

print(f"\nPopulation Distribution (double post-selected):")
for n, pop in enumerate(Pg_n_f1):
    marker = " ← TARGET" if n == 1 else ""
    print(f"  |{n}⟩: {pop:.6f}{marker}")

print(f"\nQuality Metrics:")
print(f"  Population in target state |1⟩: {Pg_n_f1[1]:.6f}")
print(f"  Total detected excited counts: {Pe_n_counts_f1.sum():.0f}")
print(f"{'='*70}\n")

# Plot single post-selection results (normalized signal)
display_fock_populations(
    fock_states_f1, 
    Sg_n_norm_f1.real, 
    normalize=True, 
    title="Fock |1⟩ - Single Post-Selection (Normalized Signal)"
)

# Plot double post-selection results (actual population distribution)
display_fock_populations(
    fock_states_f1, 
    Pg_n_f1, 
    normalize=True, 
    title="Fock |1⟩ - Double Post-Selection (Population Distribution)",
    fit_alpha=True  # Optional: fit to coherent state model
)

# Optional: Compare acceptance rates across Fock states
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot acceptance rates
ax1.plot(fock_states_f1, acceptance_rate_g_f1 * 100, 'o-', label='1st (ground)', linewidth=2)
ax1.plot(fock_states_f1, acceptance_rate_e_f1 * 100, 's-', label='2nd (excited)', linewidth=2)
ax1.set_xlabel('Fock state |n⟩', fontsize=12)
ax1.set_ylabel('Acceptance rate (%)', fontsize=12)
ax1.set_title('Post-Selection Acceptance Rates - |1⟩ Prep', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.set_xticks(fock_states_f1)

# Plot excited counts per Fock state
ax2.bar(fock_states_f1, Pe_n_counts_f1, color='coral', edgecolor='black')
ax2.set_xlabel('Fock state |n⟩', fontsize=12)
ax2.set_ylabel('Excited counts', fontsize=12)
ax2.set_title('Doubly Post-Selected Excited Counts - |1⟩ Prep', fontsize=14)
ax2.set_xticks(fock_states_f1)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

print("✓ Analysis complete for Fock |1⟩ preparation!")


### Fock |2⟩ Preparation 

In [ ]:
# ======================================================================
# FOCK STATE |2⟩ PREPARATION - Optimized 6-gate sequence
# Target fidelity: 0.990893 | Gate structure: D-SNAP-D-SNAP-D-SNAP
# ======================================================================

# Gate 1: Displacement α = -1.100617-0.905922j
disp1_f2 = Displacement(alpha=-1.100617-0.905922j, build=True)

# Gate 2: SNAP with n_active = 4 (first 4 non-zero thetas)
thetas_snap1 = [1.77699875, 2.28381248, -0.56870374, -0.4720737]
snap1_f2 = SNAP(thetas_snap1, build=True, include_unselective=False)

# Gate 3: Displacement α = 0.771348-0.812763j
disp2_f2 = Displacement(alpha=0.771348-0.812763j, build=True)

# Gate 4: SNAP with n_active = 4
thetas_snap2 = [1.15231966, -0.86403778, -0.39368209, -0.84862988]
snap2_f2 = SNAP(thetas_snap2, build=True, include_unselective=False)

# Gate 5: Displacement α = -0.366437+1.183297j
disp3_f2 = Displacement(alpha=-0.366437+1.183297j, build=True)

# Gate 6: SNAP with n_active = 4 (final phase correction, near 2π)
thetas_snap3 = [6.28314373, 6.28314373, 6.28317538, 6.28314373]
snap3_f2 = SNAP(thetas_snap3, build=True, include_unselective=False)

# Define state preparation sequence
def state_prep_f2():
    disp1_f2.play()
    snap1_f2.play()
    disp2_f2.play()
    snap2_f2.play()
    disp3_f2.play()
    #snap3_f2.play()

# Probe frequencies for Fock states 0-8
fock_states_f2 = np.arange(0, 9, dtype=int)
probe_fqs_f2 = attr.get_fock_frequencies(fock_states_f2, from_chi=True)

# Burn pulses to hardware
experiment.burn_pulses()

# Run Fock-resolved spectroscopy
print("\nRunning Fock-resolved spectroscopy for |2⟩ preparation...")
runres_f2 = experiment.fock_resolved_spectroscopy(
    probe_fqs_f2, 
    state_prep=state_prep_f2, 
    n_avg=10000
)

# Extract results
probe_fqs_f2, S_f2, Sg_n_norm_f2, Pg_n_f2 = runres_f2.output.extract(
    "probe_fqs", "S", "Sg_n_norm", "Pg_n"
)
acceptance_rate_g_f2, acceptance_rate_e_f2 = runres_f2.output.extract(
    "acceptance_rate_g", "acceptance_rate_e"
)

# Display results
print(f"\n{'='*70}")
print(f"FOCK STATE |2⟩ SPECTROSCOPY RESULTS")
print(f"{'='*70}")
print(f"First post-selection (ground state) rate: {acceptance_rate_g_f2.mean():.2%}")
print(f"Second post-selection (excited state) rate: {acceptance_rate_e_f2.mean():.2%}")
print(f"\nPopulation in target state |2⟩: {Pg_n_f2[2]:.6f}")
print(f"{'='*70}\n")

# Plot single post-selection results (normalized signal)
display_fock_populations(
    fock_states_f2, 
    Sg_n_norm_f2.real, 
    normalize=True, 
    title="Fock |2⟩ - Single Post-Selection Pop |n⟩"
)

# Plot double post-selection results (actual population distribution)
display_fock_populations(
    fock_states_f2, 
    Pg_n_f2, 
    normalize=True, 
    title="Fock |2⟩ - Double Post-Selection Pop |n⟩"
)


### SNAP rotation optimization

In [ ]:
import numpy as np

fock_levels = [0, 1, 2, 3]

thetas = [0.0 for _ in range(len(fock_levels))]
thetas[0] = np.pi

alpha = 1.0
disp = Displacement(alpha=alpha, build=True)

T_sel = 1e-6
lam0 = np.pi / (2 * T_sel)

scale_n = np.ones(len(fock_levels))
scale_n = np.array([0.9, 1.08, 0.98, 0.93])
scale_bounds = [0.9, 1.1]
d_lambda_n = (scale_n - 1.0) * lam0

d_alpha_n = np.zeros(len(fock_levels))
d_alpha_n = [0.074, 0.3, -0.14, -0.23]
d_alpha_bounds = [-0.1 * np.pi, 0.1 * np.pi]

unopt_snap = SNAP(
    thetas,
    build=True,
    include_unselective=True,
    d_lambda=d_lambda_n,
    d_alpha=d_alpha_n,
    op="test_snap"
)

experiment.burn_pulses()
attr = experiment.attributes

# |c_n|^2 weights for the chosen alpha
fock_levels_arr = np.asarray(fock_levels, dtype=int)
pop_n = cQED_models.coherent_population_model(fock_levels_arr, alpha)  # |c_n|^2

def eps2_from_Pg_joint(Pg_joint, pop, clip=False):
    """
    Compute |eps_n|^2 from P(g,n)=|c_n|^2*(1-|eps_n|^2/4).
    """
    r = Pg_joint / pop
    if clip:
        r = np.clip(r, 0.0, 1.0)   # protect against noise making r>1 or <0
    return 4.0 * (1.0 - r)         # |eps_n|^2

# Define state preparation sequence
def state_prep():
    disp.play()
    unopt_snap.play()

fock_fqs = attr.get_fock_frequencies(fock_levels, from_chi=False)
runres = experiment.fock_resolved_spectroscopy(fock_fqs, state_prep=state_prep, n_avg=20000)

# Extract using the NEW keys from the patched post-processor
Pg_n_joint, Sg_n = runres.output.extract("Pg_n_joint_from_S", "Sg_n")

Pg_n_joint = np.asarray(Pg_n_joint, dtype=float)
print(Pg_n_joint)
# Compute eps^2 and eps magnitude (optional)
eps2_n = np.abs(eps2_from_Pg_joint(Pg_n_joint, pop_n, clip=False))
eps_mag_n = np.sqrt(eps2_n)  # |eps_n|

# Weighting choices:
# - If you want "expected contribution" weighting, pop_n is natural.
weighted_eps2 = pop_n * eps2_n
cost = abs(np.nansum(weighted_eps2))

print("Sanity: sum_n P(g,n) ≈ P(g)")
print("sum Pg_n_joint =", np.nansum(Pg_n_joint))

for n, epsmag, eps2 in zip(fock_levels_arr, eps_mag_n, eps2_n):
    print(f"n={n}: |eps_n|={epsmag:.4f}   |eps_n|^2={eps2:.4f}")

print("Cost =", cost)


In [ ]:
display_fock_populations(fock_levels, Sg_n.real, fit_alpha=True, max_alpha=1.2)

In [ ]:
cQED_plottings.plot_IQ(runres.output.extract("S_0"), s=1) 

In [ ]:
Pg_n_joint, Sg_n, ref_S = runres.output.extract("Pg_n_joint_from_S", "Sg_n", "ref_sel_r180_S")
test_ref_S = (8.855251595377923e-06+0.00010204588919878005j)
measureMacro.compute_Pe_from_S(ref_S)

In [ ]:
import numpy as np
import logging
from qubox.optimization.optimization import bayesian_optimize
from qubox.logging_config import temporarily_set_levels
from skopt.space import Real

fock_levels = [0, 1, 2, 3]

thetas = [0.0 for _ in range(len(fock_levels))]
thetas[0] = np.pi

alpha = 1.0
disp = Displacement(alpha=alpha, build=True)

T_sel = 1e-6
lam0 = np.pi / (2 * T_sel)

attr = experiment.attributes

# |c_n|^2 weights for the chosen alpha
fock_levels_arr = np.asarray(fock_levels, dtype=int)
pop_n = cQED_models.coherent_population_model(fock_levels_arr, alpha)  # |c_n|^2

# Thresholds for noise filtering
MIN_POPULATION = 0.05  # Ignore levels with |c_n|^2 < 5%

# Setup loggers for suppression
qubox_logger = logging.getLogger("qubox")
qm_logger = logging.getLogger("qm")

def eps2_from_Pg_joint(Pg_joint, pop, clip=False):
    """
    Compute |eps_n|^2 from P(g,n)=|c_n|^2*(1-|eps_n|^2/4).
    """
    r = Pg_joint / pop
    if clip:
        r = np.clip(r, 0.0, 1.0)   # protect against noise making r>1 or <0
    return 4.0 * (1.0 - r)         # |eps_n|^2

# Evaluation counter and history
eval_counter = [0]
cost_history = []

def objective_function(params):
    """
    Cost function for Bayesian optimization.
    
    Params are flattened: [scale_n[0], ..., scale_n[4], d_alpha_n[0], ..., d_alpha_n[4]]
    """
    eval_counter[0] += 1
    n_levels = len(fock_levels)
    
    # Extract scale_n and d_alpha_n from flat params
    scale_n = np.array(params[:n_levels])
    d_alpha_n = np.array(params[n_levels:])
    
    # Compute d_lambda_n from scale_n
    d_lambda_n = (scale_n - 1.0) * lam0
    
    # Build SNAP gate with current parameters
    snap_gate = SNAP(
        thetas,
        build=True,
        include_unselective=True,
        d_lambda=d_lambda_n,
        d_alpha=d_alpha_n,
        op="snap_opt"
    )
    
    # Define state preparation sequence
    def state_prep():
        disp.play()
        snap_gate.play()
    
    
    # Run experiment with suppressed logging
    with temporarily_set_levels([qubox_logger, qm_logger], logging.WARNING):
        experiment.burn_pulses()
        fock_fqs = attr.get_fock_frequencies(fock_levels, from_chi=False)
        runres = experiment.fock_resolved_spectroscopy(fock_fqs, state_prep=state_prep, n_avg=25)
    
    # Extract joint probabilities and acceptance rates
    Pg_n_joint = np.asarray(runres.output.extract("Pg_n_joint_from_S"), dtype=float)
    acc_g = np.asarray(runres.output.extract("acceptance_rate_g"), dtype=float)
    acc_e = np.asarray(runres.output.extract("acceptance_rate_e"), dtype=float)
    
    # Compute epsilon^2 for each level
    eps2_n = eps2_from_Pg_joint(Pg_n_joint, pop_n, clip=False)
    
    # Filter out noise-dominated levels
    valid_mask = pop_n > MIN_POPULATION
    eps2_filtered = np.where(valid_mask, eps2_n, 0.0)
    
    # Cost: weighted by population
    cost = np.abs(np.nansum(pop_n * eps2_filtered))
    
    # Extract acceptance rates for monitoring only
    mean_acc_g = np.nanmean(acc_g)
    mean_acc_e = np.nanmean(acc_e)
    
    # Store cost in history
    cost_history.append(cost)
    
    # Print progress
    print(f"\n{'='*80}")
    print(f"Evaluation #{eval_counter[0]}")
    print(f"{'='*80}")
    print(f"Parameters:")
    print(f"  scale_n:   {scale_n}")
    print(f"  d_alpha_n: {d_alpha_n}")
    print(f"\nResults:")
    print(f"  Pg_n_joint:  {Pg_n_joint}")
    print(f"  eps^2_n:     {eps2_n}")
    print(f"  Acceptance:  g={mean_acc_g:.3f}, e={mean_acc_e:.3f}")
    print(f"\nCOST: {cost:.6f}")
    print(f"{'='*80}\n")
    
    return cost

# Define parameter space
# scale_n bounds: [0.9, 1.1] for each Fock level
# d_alpha_n bounds: [-0.1*pi, 0.1*pi] for each Fock level
param_space = []
for i in range(len(fock_levels)):
    param_space.append(Real(0.9, 1.1, name=f"scale_n[{i}]"))
for i in range(len(fock_levels)):
    param_space.append(Real(-0.1*np.pi, 0.1*np.pi, name=f"d_alpha_n[{i}]"))

# Run Bayesian optimization
print("Starting Bayesian Optimization...")
print(f"Optimizing {len(param_space)} parameters over Fock levels {fock_levels}")
print(f"Population weights: {pop_n}\n")

result = bayesian_optimize(
    objective_func=objective_function,
    param_space=param_space,
    n_calls=180,              # Total evaluations
    n_random_starts=20,      # Random exploration before GP model
    random_state=42
)

# Extract and display best result
n_levels = len(fock_levels)
best_scale_n = np.array(result.x[:n_levels])
best_d_alpha_n = np.array(result.x[n_levels:])
best_d_lambda_n = (best_scale_n - 1.0) * lam0

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE")
print("="*80)
print(f"Best cost achieved: {result.fun:.6f}")
print(f"\nOptimal parameters:")
print(f"  scale_n:   {best_scale_n}")
print(f"  d_alpha_n: {best_d_alpha_n}")
print(f"  d_lambda_n: {best_d_lambda_n}")
print("="*80)

# Store best parameters for later use
optimized_snap_params = {
    'scale_n': best_scale_n,
    'd_alpha_n': best_d_alpha_n,
    'd_lambda_n': best_d_lambda_n,
    'cost': result.fun
}

# Plot optimization progress
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
iterations = np.arange(1, len(cost_history) + 1)
ax.plot(iterations, cost_history, 'o-', linewidth=2, markersize=6, label='Cost')

# Mark the best point
best_iter = np.argmin(cost_history) + 1
best_cost = np.min(cost_history)
ax.plot(best_iter, best_cost, 'r*', markersize=20, label=f'Best (iter {best_iter}, cost={best_cost:.6f})')

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Cost (Population-weighted |ε|²)', fontsize=12)
ax.set_title('Bayesian Optimization Progress', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nTotal evaluations: {len(cost_history)}")
print(f"Best cost: {best_cost:.6f} at iteration {best_iter}")
print(f"Final cost: {cost_history[-1]:.6f}")
print(f"Improvement: {(cost_history[0] - best_cost)/cost_history[0]*100:.1f}%")


### SNAP rot opt fast

In [ ]:
import numpy as np
import logging
from skopt.space import Real
from qubox.optimization.optimization import bayesian_optimize
from qubox.logging_config import temporarily_set_levels

# -----------------------------
# FAST MODE SETTINGS
# -----------------------------
POP_THRESH = 0.10          # only optimize / measure Fock levels with pop > 10%
N_AVG_COARSE = 3000        # stage-1
N_AVG_FINE   = 20000       # stage-2
N_CALLS_COARSE = 50
N_CALLS_FINE   = 40
N_RANDOM_STARTS_COARSE = 12
N_RANDOM_STARTS_FINE   = 8

# -----------------------------
# USER INPUTS (your original)
# -----------------------------
fock_levels_full = [0, 1, 2, 3]

alpha = 1.0
disp = Displacement(alpha=alpha, build=True)

T_sel = 1e-6
lam0 = np.pi / (2 * T_sel)

attr = experiment.attributes

# |c_n|^2 for all candidate levels
fock_levels_arr_full = np.asarray(fock_levels_full, dtype=int)
pop_full = cQED_models.coherent_population_model(fock_levels_arr_full, alpha)

# pick levels to actually optimize/measure
valid_mask = pop_full > POP_THRESH
fock_levels = list(np.asarray(fock_levels_full, dtype=int)[valid_mask])
pop_n = np.asarray(pop_full[valid_mask], dtype=float)

if len(fock_levels) == 0:
    raise RuntimeError(
        f"No Fock levels pass POP_THRESH={POP_THRESH}. "
        f"pop_full={pop_full}, alpha={alpha}"
    )

print(f"FAST MODE levels: {fock_levels} with weights pop_n={pop_n}")

# thetas for those levels (example: pi on n=0 if present, else all 0)
thetas = np.zeros(len(fock_levels), dtype=float)
if 0 in fock_levels:
    thetas[fock_levels.index(0)] = np.pi

# Precompute frequencies once (do NOT do this inside objective)
fock_fqs = attr.get_fock_frequencies(fock_levels, from_chi=False)

# Setup loggers for suppression
qubox_logger = logging.getLogger("qubox")
qm_logger = logging.getLogger("qm")


def eps2_from_Pg_joint(Pg_joint, pop, clip=False):
    """
    Compute |eps_n|^2 from P(g,n)=|c_n|^2*(1-|eps_n|^2/4).
    clip=True prevents noise from producing unphysical eps^2 < 0.
    """
    r = Pg_joint / pop
    if clip:
        r = np.clip(r, 0.0, 1.0)
    return abs(4.0 * (1.0 - r))


# Burn pulses ONCE (big speedup if your stack was doing it per-eval)
with temporarily_set_levels([qubox_logger, qm_logger], logging.WARNING):
    experiment.burn_pulses()

# Eval counter + history
eval_counter = [0]
cost_history = []


def make_objective(n_avg: int):
    """
    Returns an objective that runs with a fixed n_avg (multi-fidelity).
    Parameterization ("fast mode"):
      scale_n   = s0 + s1*n
      d_alpha_n = a0 + a1*n
    So only 4 params total: [s0, s1, a0, a1]
    """
    n_arr = np.asarray(fock_levels, dtype=float)

    def objective(params):
        eval_counter[0] += 1
        s0, s1, a0, a1 = params

        # Build per-n arrays from low-order model
        scale_n = s0 + s1 * n_arr
        d_alpha_n = a0 + a1 * n_arr

        # Convert to d_lambda_n for SNAP
        d_lambda_n = (scale_n - 1.0) * lam0

        snap_gate = SNAP(
            thetas,
            build=True,
            include_unselective=True,
            d_lambda=d_lambda_n,
            d_alpha=d_alpha_n,
            op="snap_opt_fast",
        )

        def state_prep():
            disp.play()
            snap_gate.play()

        with temporarily_set_levels([qubox_logger, qm_logger], logging.WARNING):
            experiment.burn_pulses()
            runres = experiment.fock_resolved_spectroscopy(
                fock_fqs,
                state_prep=state_prep,
                n_avg=n_avg,
            )

        Pg_n_joint = np.asarray(runres.output.extract("Pg_n_joint_from_S"), dtype=float)

        # Use clip=True to avoid "fake good" values when Pg/pop > 1 from noise
        eps2_n = eps2_from_Pg_joint(Pg_n_joint, pop_n, clip=False)

        # Cost: population-weighted eps^2
        cost = float(np.nansum(pop_n * eps2_n))
        cost_history.append(cost)

        print(f"\n{'='*80}")
        print(f"Evaluation #{eval_counter[0]}  (n_avg={n_avg})")
        print(f"{'='*80}")
        print(f"Params: s0={s0:.6f}, s1={s1:.6f}, a0={a0:.6f}, a1={a1:.6f}")
        print(f"Derived arrays on n={fock_levels}:")
        print(f"  scale_n:   {scale_n}")
        print(f"  d_alpha_n: {d_alpha_n}")
        print(f"Results:")
        print(f"  Pg_n_joint: {Pg_n_joint}")
        print(f"  eps^2_n:    {eps2_n}")
        print(f"COST: {cost:.6f}")
        print(f"{'='*80}\n")

        return cost

    return objective


# -----------------------------
# STAGE 1: COARSE OPT
# -----------------------------
param_space_coarse = [
    Real(0.90, 1.10, name="s0"),
    Real(-0.05, 0.05, name="s1"),
    Real(-0.10*np.pi, 0.10*np.pi, name="a0"),
    Real(-0.05*np.pi, 0.05*np.pi, name="a1"),
]

print("\nStarting FAST MODE Bayesian Optimization (COARSE)...")
print(f"Levels: {fock_levels}")
print(f"Weights pop_n: {pop_n}")

result_coarse = bayesian_optimize(
    objective_func=make_objective(N_AVG_COARSE),
    param_space=param_space_coarse,
    n_calls=N_CALLS_COARSE,
    n_random_starts=N_RANDOM_STARTS_COARSE,
    random_state=42,
)

best_s0, best_s1, best_a0, best_a1 = result_coarse.x
print("\n" + "="*80)
print("COARSE COMPLETE")
print("="*80)
print(f"Best coarse cost: {result_coarse.fun:.6f}")
print(f"Best coarse params: s0={best_s0}, s1={best_s1}, a0={best_a0}, a1={best_a1}")
print("="*80)

# -----------------------------
# STAGE 2: FINE OPT (tighten bounds around coarse best)
# -----------------------------
def tighten(x, frac, lo, hi):
    w = (hi - lo) * frac
    return max(lo, x - w), min(hi, x + w)

s0_lo, s0_hi = tighten(best_s0, 0.10, 0.90, 1.10)
s1_lo, s1_hi = tighten(best_s1, 0.20, -0.05, 0.05)
a0_lo, a0_hi = tighten(best_a0, 0.10, -0.10*np.pi, 0.10*np.pi)
a1_lo, a1_hi = tighten(best_a1, 0.20, -0.05*np.pi, 0.05*np.pi)

param_space_fine = [
    Real(s0_lo, s0_hi, name="s0"),
    Real(s1_lo, s1_hi, name="s1"),
    Real(a0_lo, a0_hi, name="a0"),
    Real(a1_lo, a1_hi, name="a1"),
]

print("\nStarting FAST MODE Bayesian Optimization (FINE)...")
print("FINE bounds:")
print(f"  s0 in [{s0_lo:.6f}, {s0_hi:.6f}]")
print(f"  s1 in [{s1_lo:.6f}, {s1_hi:.6f}]")
print(f"  a0 in [{a0_lo:.6f}, {a0_hi:.6f}]")
print(f"  a1 in [{a1_lo:.6f}, {a1_hi:.6f}]")

result_fine = bayesian_optimize(
    objective_func=make_objective(N_AVG_FINE),
    param_space=param_space_fine,
    n_calls=N_CALLS_FINE,
    n_random_starts=N_RANDOM_STARTS_FINE,
    random_state=43,
)

best_s0, best_s1, best_a0, best_a1 = result_fine.x

# Build final arrays on chosen levels
n_arr = np.asarray(fock_levels, dtype=float)
best_scale_n = best_s0 + best_s1 * n_arr
best_d_alpha_n = best_a0 + best_a1 * n_arr
best_d_lambda_n = (best_scale_n - 1.0) * lam0

print("\n" + "="*80)
print("FAST MODE OPTIMIZATION COMPLETE")
print("="*80)
print(f"Best fine cost achieved: {result_fine.fun:.6f}")
print("Best params:")
print(f"  s0={best_s0}, s1={best_s1}, a0={best_a0}, a1={best_a1}")
print(f"Derived per-level parameters on n={fock_levels}:")
print(f"  scale_n:    {best_scale_n}")
print(f"  d_alpha_n:  {best_d_alpha_n}")
print(f"  d_lambda_n: {best_d_lambda_n}")
print("="*80)

optimized_snap_params_fast = {
    "fock_levels": fock_levels,
    "pop_n": pop_n,
    "s0": best_s0,
    "s1": best_s1,
    "a0": best_a0,
    "a1": best_a1,
    "scale_n": best_scale_n,
    "d_alpha_n": best_d_alpha_n,
    "d_lambda_n": best_d_lambda_n,
    "cost": result_fine.fun,
}

# -----------------------------
# Plot progress
# -----------------------------
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
iters = np.arange(1, len(cost_history) + 1)
ax.plot(iters, cost_history, 'o-', linewidth=2, markersize=6, label='Cost')

best_iter = int(np.argmin(cost_history) + 1)
best_cost = float(np.min(cost_history))
ax.plot(best_iter, best_cost, 'r*', markersize=20,
        label=f'Best (iter {best_iter}, cost={best_cost:.6f})')

ax.set_xlabel('Iteration')
ax.set_ylabel('Cost (weighted |ε|²)')
ax.set_title('FAST MODE Optimization Progress')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nTotal evaluations: {len(cost_history)}")
print(f"Best cost: {best_cost:.6f} at iteration {best_iter}")
print(f"Final cost: {cost_history[-1]:.6f}")


### SNAP phase optimization

## Cavity Gate Optimization

## Advance Cavity Characterization

### Kerr phase evolution

In [ ]:

def choose_displacements(n: int,
                         alpha_cap: float = 6.0,    # upper-bound for |α|
                         eps_abs: float  = 0.10):   # fixed |ε|  (linear regime)
    alpha_amp = np.sqrt(n + 0.5)
    alpha_amp = min(alpha_amp, alpha_cap)
    return alpha_amp, eps_abs

for n in range(10):
    attr = experiment.attributes 
    fock_centers = attr.get_fock_frequencies(20)
    theta_array = np.linspace(0, 2*np.pi, 20) 
    delay_clks = np.arange(1, 10000, 800, dtype=int)

    #delay_clks = np.array([750])
    alpha_amp, eps_amp = choose_displacements(n)
    disp_alpha   = Displacement(alpha = alpha_amp)  # state-prep
    disp_epsilon = Displacement(alpha = eps_amp)    # probe
    print(f"[info] |α| = {alpha_amp:.3f}, |ε| = {eps_amp:.3f}")

    if not (0 <= n < len(fock_centers)-1):
        raise ValueError(f"i must be 0..{len(fock_centers)-2}")


    f_n  = fock_centers[n]
    f_np = fock_centers[n+1]

    probe_fqs = np.array([f_n])
    experiment.save_unique_identifier = f"dynamic_disp_{n}_phase_evolution_070725_"
    result = experiment.storage_phase_evolution(n, probe_fqs, theta_array, None, delay_clks, n_avg=7500)


In [ ]:
states, delays, thetas = output.extract("S", "delays_clk", "thetas")
parity = np.real(states)

def fringe_model(theta, A, B, phi):
    return A + B*np.cos(theta + phi)

Nd, Nn, Nθ = parity.shape
phases = np.zeros((Nd, Nn))

for i in range(Nd):
    for j in range(Nn):
        y = parity[i, j, :]                       # a 1‑D slice vs θ
        # initial guesses : A=0, B=0.5, phi=0
        popt, _ = curve_fit(fringe_model, thetas, y, p0=[0, 0.5, 0])
        phases[i, j] = popt[2] 

phases_unwrapped = phases
dt = 4
plt.figure()
for j in range(Nn):
    plt.plot(delays*dt, phases_unwrapped[:,j],
             label=f"|{j}>→|{j+1}>")
    # optional: overplot linear fit
    m, c = np.polyfit(delays*dt, phases_unwrapped[:,j], 1)
    plt.plot(delays*dt, m*delays*dt + c, "--", color="C%d"%j)
plt.xlabel(r"$\Delta T\;(\mu s)$")
plt.ylabel("phase (rad)")
plt.legend(ncol=2, fontsize=8)
plt.title("Phase evolution  (qubit in |g〉)")
plt.tight_layout()
plt.show()

In [ ]:
S = states[:, 0, :].real
T = delay_ticks * 4
θ = theta_array
φ = []                          # extracted phase per ΔT
for row in S:                   # iterate over delay (fixed ΔT)
    # least-squares fit  A cos(θ - φ) + B
    A = np.vstack([np.cos(θ), np.sin(θ), np.ones_like(θ)]).T
    c, _, _, _ = np.linalg.lstsq(A, row, rcond=None)[0:4]
    phase = np.arctan2(-c[1], c[0])    # minus sign because of cos shift
    φ.append(phase)
φ = np.unwrap(φ)    
slope, intercept = np.polyfit(T, φ, 1)
Δ_rad_s = slope                    # rad s⁻¹
Δ_Hz    = Δ_rad_s / (2*np.pi) 

# Quantum Circuit Experiments

## Sequential Simulation

### Benchmarking <ZZ>

In [ ]:
from qubox.gates import *
from holographic_sim.holographicSim import holographic_sim_cached

import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Helper: coerce to CMatrix convention used by holographic_sim
# ----------------------------
def to_column_stochastic_2x2(M, name="matrix", atol=1e-3):
    """
    Ensure 2x2 matrix is column-stochastic: M[row, col] = P(row | true=col),
    so columns sum to 1. If it looks row-stochastic, transpose it.
    """
    A = np.asarray(M, dtype=float)
    if A.shape != (2, 2):
        raise ValueError(f"{name} must be shape (2,2), got {A.shape}")

    col_sums = A.sum(axis=0)
    row_sums = A.sum(axis=1)

    col_ok = np.allclose(col_sums, 1.0, atol=atol)
    row_ok = np.allclose(row_sums, 1.0, atol=atol)

    # If it's row-stochastic but not column-stochastic, transpose.
    if row_ok and not col_ok:
        A = A.T
        col_sums = A.sum(axis=0)

    # Normalize columns defensively (handles small numerical drift)
    col_sums = np.where(col_sums == 0, 1.0, col_sums)
    A = A / col_sums[None, :]

    return A


# ----------------------------
# Build gate list U_gates
# ----------------------------
pi_ratio = 3
theta = np.pi/pi_ratio
rot = QubitRotation(
    theta, 0,
    b_x180_pulse="x180_pulse",
    b_y180_pulse="y180_pulse",
    build=False
)

U = GateArray([rot], op_prefix="U")
U.build()

attr = experiment.attributes
N = 5
U_gates = [U] * N

fock_dim = 20
U_matrix_list = [gate.get_unitaries(fock_dim) for gate in U_gates]

experiment.burn_pulses()

# ----------------------------
# Observable
# ----------------------------
X = np.array([[0, 1],
              [1, 0]], dtype=np.complex128)

Y = np.array([[0, -1j],
              [1j,  0]], dtype=np.complex128)

Z = np.array([[1, 0],
              [0, -1]], dtype=np.complex128)

# ----------------------------
# Run settings
# ----------------------------
shot_nums = 10000
total_pairs = N - 1

dt_holo = 16e-9

T1 = getattr(attr, "qb_T1_relax", None)
T2 = getattr(attr, "qb_T2_echo", None)
T1_in_s = T1 * 1e-9 if T1 is not None else None  # assuming attr in ns
T2_in_s = T2 * 1e-9 if T2 is not None else None  # assuming attr in ns

T1_in_s = T2_in_s = 1
# Pull confusion/transition from your calibrated dict
ro_quality_dict = measureMacro._ro_quality_params
confusion_matrix_raw  = ro_quality_dict["confusion_matrix"]
transition_matrix_raw = ro_quality_dict["transition_matrix"]

# Coerce into holographic_sim CMatrix convention
confusion_matrix  = to_column_stochastic_2x2(confusion_matrix_raw,  name="confusion_matrix")
transition_matrix = to_column_stochastic_2x2(transition_matrix_raw, name="transition_matrix")

# ----------------------------
# Storage (per pair i,i+1)
# ----------------------------
holo_ideal_samples   = np.zeros((total_pairs, shot_nums), dtype=np.complex128)
holo_with_deco_samples = np.zeros((total_pairs, shot_nums), dtype=np.complex128)

holo_ideal_mean   = np.zeros(total_pairs, dtype=float)
holo_with_deco_mean = np.zeros(total_pairs, dtype=float)
seq_mean            = np.zeros(total_pairs, dtype=float)

holo_ideal_sem   = np.zeros(total_pairs, dtype=float)
holo_with_deco_sem = np.zeros(total_pairs, dtype=float)
seq_sem            = np.zeros(total_pairs, dtype=float)

states_array = []

for i in range(total_pairs):
    # --- operator list for ZZ on sites i and i+1
    op_list = [None] * N
    op_list[i] = Z
    op_list[i + 1] = Z

    # (1) holo: no decoherence, WITH confusion + transition
    C0 = holographic_sim_cached(
        U_matrix_list,
        op_list,
        shot_nums=shot_nums
    )
    holo_ideal_samples[i] = C0

    # (2) holo: with decoherence, WITH confusion + transition
    holo_kwargs = dict(
        shot_nums=shot_nums,
        dt=dt_holo,
        confusion=confusion_matrix,
        transition=transition_matrix,
    )
    if T1_in_s is not None:
        holo_kwargs["T1"] = T1_in_s
    if T2_in_s is not None:
        holo_kwargs["T2"] = T2_in_s

    C1 = holographic_sim_cached(U_matrix_list, op_list, **holo_kwargs)
    holo_with_deco_samples[i] = C1

    # Convert complex samples -> real expectation (ZZ should be real)
    c0r = np.real(C0)
    c1r = np.real(C1)

    holo_ideal_mean[i] = c0r.mean()
    holo_with_deco_mean[i] = c1r.mean()
    holo_ideal_sem[i] = c0r.std(ddof=1) / np.sqrt(shot_nums)
    holo_with_deco_sem[i] = c1r.std(ddof=1) / np.sqrt(shot_nums)

    # (3) sequential simulation (hardware / sim stack already includes readout imperfections)
    measurement_gates = [Measure(axis="none", target=attr.ro_el) for _ in range(N)]
    measurement_gates[i] = Measure(axis="z", target=attr.ro_el)
    measurement_gates[i + 1] = Measure(axis="z", target=attr.ro_el)

    runres = experiment.sequential_simulation(U_gates, measurement_gates, shot_nums)
    states = runres.output.extract("states")
    states_array.append(states)

    zz_each_shot = np.prod(analysis_tools.bools_to_sigma_z(states) * (-1), axis=1)
    seq_mean[i] = zz_each_shot.mean()
    seq_sem[i] = zz_each_shot.std(ddof=1) / np.sqrt(shot_nums)

# ----------------------------
# Plot comparison
# ----------------------------
x = np.arange(total_pairs)

plt.figure(figsize=(10, 5))
plt.errorbar(x, holo_ideal_mean,   yerr=holo_ideal_sem,   fmt="o-", capsize=3,
             label="holo (ideal)")
plt.errorbar(x, holo_with_deco_mean, yerr=holo_with_deco_sem, fmt="o-", capsize=3,
             label="holo (decoherence + measurement error)")
plt.errorbar(x, seq_mean,            yerr=seq_sem,            fmt="o-", capsize=3,
             label="sequential simulation")

plt.axhline(0, linewidth=1)
plt.xlabel(r"Pair index i (correlator is $Z_i Z_{i+1}$)")
plt.ylabel(r"$\langle Z_i Z_{i+1}\rangle$")
plt.title(f"ZZ correlators, N={N}, shots={shot_nums} with R($\pi$/{pi_ratio})")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
experiment.quaProgMngr.serialize_program()

### benchmarking X, Y, Z

In [ ]:
from qubox.gates import *
# IMPORTANT: import your NEW simulator module
from holographic_sim_updated.holographicSim import *

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import textwrap

# -----------------------------------------------------------------------------
# Helper: ensure 2x2 matrices are in column-stochastic convention:
#   M[row, col] = P(observed=row | true=col)  => columns sum to 1
# If user calibration returns row-stochastic, transpose it.
# -----------------------------------------------------------------------------
def op_label(op, width=14):
    name = getattr(op, "op", None)
    if name is None:
        return "op(?)"
    return textwrap.fill(str(name), width=width)

# =============================================================================
# 1) Build the experiment gates (for real hardware run)
# =============================================================================
operation_hw = QubitRotation(theta=np.pi, phi=0)
U = GateArray([operation_hw], op_prefix="U")
U.build()
experiment.burn_pulses()

attr = experiment.attributes

# Use your requested values
dt = 16e-9  # 16 ns (seconds)
T1 = float(attr.qb_T1_relax) * 1e-9  # attr is ns -> seconds
T2 = float(attr.qb_T2_ramsey) * 1e-9 # attr is ns -> seconds

# Readout calibration (USE THESE for simulation + comparison)
ro_cal = measureMacro.get_readout_calibration()
C_meas = ro_cal["confusion_matrix"]
T_meas = ro_cal["transition_matrix"]

# =============================================================================
# 2) Experimental: measure <X_i>, <Y_i>, <Z_i> by running the real sequence
#    (your existing pattern: only one site is measured in axis ax each run)
# =============================================================================
N = 10
shot_nums = 10_000
U_gates = [U] * N

axes_exp = ["x", "y", "z"]
exp_means = {ax.upper(): np.zeros(N, dtype=float) for ax in axes_exp}

for ax in axes_exp:
    axU = ax.upper()
    for i in range(N):
        measurement_gates = [Measure(axis="none", target=attr.ro_el) for _ in range(N)]
        measurement_gates[i] = Measure(axis=ax, target=attr.ro_el)

        runres = experiment.sequential_simulation(U_gates, measurement_gates, shot_nums)
        states = runres.output.extract("states")   # usually (shots,) or (shots,1)

        # bool -> sigma_z: True -> -1, False -> +1
        sigma_z_mean = np.where(states, -1, 1).mean()

        exp_means[axU][i] = float(sigma_z_mean)

# =============================================================================
# 3) Simulation: build Kraus channel per site using T1/T2 + dt,
#    then get <X_i>, <Y_i>, <Z_i> using holographic_sim with C_meas/T_meas
# =============================================================================
operation_sim = QubitRotation(np.pi, 0, build=False)
K = operation_sim.get_kraus(dt=dt, T1=T1, T2=T2, n_max=0)  # single-qubit channel
channel_list = [K] * N
latency_channel_sys = build_T1T2_kraus(dt=368e-9, T1=T1, T2=T2)


axes_sim = [("X", PAULI_X), ("Y", PAULI_Y), ("Z", PAULI_Z)]
sim_means = {ax: np.zeros(N, dtype=float) for ax, _ in axes_sim}

for ax_name, pauli in axes_sim:
    for i in tqdm(range(N), desc=f"Sim {ax_name}"):
        op_list = [None] * N
        op_list[i] = pauli

        samples = holographic_sim(
            channel_list,
            op_list,
            shot_nums=shot_nums,
            C_meas=C_meas,
            T_meas=T_meas,
            latency_channel_sys=latency_channel_sys,  
            show_progress=False,
            parallel_process=True,
        )
        sim_means[ax_name][i] = float(np.real_if_close(samples).mean())

# =============================================================================
# 4) Plot: overlay experiment vs simulation for X/Y/Z on each site
#    - Sim: line + marker
#    - Exp: scatter-only points, SAME color as sim, different marker shape
# =============================================================================
fig_w = max(7.0, 0.85 * N)
fig_h = 4.8
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

x = np.arange(N)

exp_marker = {"X": "s", "Y": "D", "Z": "^"}
sim_marker = {"X": "o", "Y": "o", "Z": "o"}

for ax_name in ["X", "Y", "Z"]:
    (sim_line,) = ax.plot(
        x,
        sim_means[ax_name],
        marker=sim_marker[ax_name],
        linestyle="-",
        label=f"Sim {ax_name}",
    )
    c = sim_line.get_color()

    ax.scatter(
        x,
        exp_means[ax_name],
        marker=exp_marker[ax_name],
        color=c,
        label=f"Exp {ax_name}",
        zorder=5,
    )

# ✅ x-axis is now just site index i
ax.set_xlabel(r"Site $i$")
ax.set_ylabel(r"$\langle K_i \rangle$")
ax.set_title(
    "Single-site expectation values: Simulation vs Experiment\n"
    rf"$dt={dt*1e9:.0f}\,$ns, "
    rf"$T_1={T1*1e6:.2f}\,\mu s, "
    rf"$T_2={T2*1e6:.2f}\,\mu s$"
)

ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in x])   # or just omit this line entirely
ax.margins(x=0.02)
ax.legend(ncols=2)
fig.tight_layout()
plt.show()


### benchmarking ZZ correlation

In [ ]:
from qubox.gates import *
from holographic_sim_updated.holographicSim import (
    holographic_sim,
    holographic_sim_cached,
)

import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Helper: coerce to column-stochastic convention (columns sum to 1)
# ----------------------------
def to_column_stochastic_2x2(M, name="matrix", atol=1e-3):
    """
    Ensure 2x2 matrix is column-stochastic: M[row, col] = P(row | true=col),
    so columns sum to 1. If it looks row-stochastic, transpose it.
    """
    A = np.asarray(M, dtype=float)
    if A.shape != (2, 2):
        raise ValueError(f"{name} must be shape (2,2), got {A.shape}")

    col_sums = A.sum(axis=0)
    row_sums = A.sum(axis=1)

    col_ok = np.allclose(col_sums, 1.0, atol=atol)
    row_ok = np.allclose(row_sums, 1.0, atol=atol)

    if row_ok and not col_ok:
        A = A.T
        col_sums = A.sum(axis=0)

    col_sums = np.where(col_sums == 0, 1.0, col_sums)
    A = A / col_sums[None, :]
    return A


# ----------------------------
# Build gate list U_gates
# ----------------------------
pi_ratio = 1
theta = np.pi / pi_ratio
rot = QubitRotation(
    theta, 0,
    b_x180_pulse="x180_pulse",
    b_y180_pulse="y180_pulse",
    build=False
)

U = GateArray([rot], op_prefix="U")
U.build()

attr = experiment.attributes
N = 10
U_gates = [U] * N

fock_dim = 5
U_matrix_list = [gate.get_unitaries(fock_dim) for gate in U_gates]

experiment.burn_pulses()

# ----------------------------
# Observable
# ----------------------------
Z = np.array([[1, 0],
              [0, -1]], dtype=np.complex128)

# ----------------------------
# Run settings
# ----------------------------
shot_nums = 5000
total_pairs = N - 1

# ----------------------------
# Measurement + reset parameters
# ----------------------------
# Perfect readout (no misclassification)
C_perfect = np.eye(2, dtype=float)

# Perfect reset-to-g (independent of reported bit)
R_reset_perfect = np.array([[1.0, 1.0],
                            [0.0, 0.0]], dtype=float)

# Measurement error only: use your calibrated confusion, but keep perfect reset
ro_quality_dict = measureMacro._ro_quality_params
confusion_raw = ro_quality_dict.get("confusion_matrix", None)
C_meas = to_column_stochastic_2x2(confusion_raw, name="confusion_matrix") if confusion_raw is not None else C_perfect

# Reset error: use your benchmarked effective reset matrix (assumed exists)
R_reset = np.array([[0.99, 0.99],[0.01, 0.1]])
R_reset = to_column_stochastic_2x2(R_reset, name="R_reset")

# In the new simulator API, you can pass a single 2x2 matrix for C_by_basis or R_by_basis.
# That single matrix is applied to X/Y/Z automatically.
C_by_basis_perfect = C_perfect
R_by_basis_perfect = R_reset_perfect
C_by_basis_measerr = C_meas
R_by_basis_measerr = R_reset_perfect
C_by_basis_both    = C_meas
R_by_basis_both    = R_reset

# Because reset requires a measurement every step, provide default reset measurement as Z.
reset_op_list = [Z] * N

# ----------------------------
# Storage (per pair i,i+1)
# ----------------------------
ideal_samples = np.zeros((total_pairs, shot_nums), dtype=np.complex128)
measerr_samples = np.zeros((total_pairs, shot_nums), dtype=np.complex128)
botherr_samples = np.zeros((total_pairs, shot_nums), dtype=np.complex128)

ideal_mean = np.zeros(total_pairs, dtype=float)
measerr_mean = np.zeros(total_pairs, dtype=float)
botherr_mean = np.zeros(total_pairs, dtype=float)
exp_mean = np.zeros(total_pairs, dtype=float)

ideal_sem = np.zeros(total_pairs, dtype=float)
measerr_sem = np.zeros(total_pairs, dtype=float)
botherr_sem = np.zeros(total_pairs, dtype=float, )
exp_sem = np.zeros(total_pairs, dtype=float)

states_array = []

for i in range(total_pairs):
    # Correlator: measure Z at sites i and i+1; elsewhere still measure Z for reset
    op_list = [None] * N
    op_list[i] = Z
    op_list[i + 1] = Z

    # ------------------------------------------------------------
    # (1) IDEAL: no errors at all (perfect readout + perfect reset)
    # Use cached exact sampler (fast for small N; warns if branching large)
    # ------------------------------------------------------------
    C0 = holographic_sim_cached(
        U_matrix_list,
        op_list,
        reset_op_list=reset_op_list,
        d=2,
        shot_nums=shot_nums,
        C_by_basis=C_by_basis_perfect,
        R_by_basis=R_by_basis_perfect,
        warn_branch_count=1000,
    )
    ideal_samples[i] = C0

    # ------------------------------------------------------------
    # (2) MEASUREMENT ERROR ONLY: confusion matrix, but perfect reset
    # Use Monte Carlo (fast, scalable)
    # ------------------------------------------------------------
    C1 = holographic_sim(
        U_matrix_list,
        op_list,
        reset_op_list=reset_op_list,
        d=2,
        shot_nums=shot_nums,
        C_by_basis=C_by_basis_measerr,
        R_by_basis=R_by_basis_measerr,
        parallel_process=False,
    )
    measerr_samples[i] = C1

    # ------------------------------------------------------------
    # (3) MEASUREMENT + RESET ERROR: confusion + benchmarked R_reset
    # ------------------------------------------------------------
    C2 = holographic_sim(
        U_matrix_list,
        op_list,
        reset_op_list=reset_op_list,
        d=2,
        shot_nums=shot_nums,
        C_by_basis=C_by_basis_both,
        R_by_basis=R_by_basis_both,
        parallel_process=False,
    )
    botherr_samples[i] = C2

    # Convert samples -> real expectation (ZZ should be real)
    c0r = np.real(C0)
    c1r = np.real(C1)
    c2r = np.real(C2)

    ideal_mean[i] = c0r.mean()
    measerr_mean[i] = c1r.mean()
    botherr_mean[i] = c2r.mean()

    ideal_sem[i] = c0r.std(ddof=1) / np.sqrt(shot_nums)
    measerr_sem[i] = c1r.std(ddof=1) / np.sqrt(shot_nums)
    botherr_sem[i] = c2r.std(ddof=1) / np.sqrt(shot_nums)

    # ------------------------------------------------------------
    # (4) ACTUAL EXPERIMENT / SEQ SIM STACK
    # ------------------------------------------------------------
    measurement_gates = [Measure(axis="none", target=attr.ro_el) for _ in range(N)]
    measurement_gates[i] = Measure(axis="z", target=attr.ro_el)
    measurement_gates[i + 1] = Measure(axis="z", target=attr.ro_el)

    runres = experiment.sequential_simulation(U_gates, measurement_gates, shot_nums)
    states = runres.output.extract("states")
    states_array.append(states)

    zz_each_shot = np.prod(analysis_tools.bools_to_sigma_z(states) * (-1), axis=1)
    exp_mean[i] = zz_each_shot.mean()
    exp_sem[i]  = zz_each_shot.std(ddof=1) / np.sqrt(shot_nums)

# ----------------------------
# Plot comparison
# ----------------------------
x = np.arange(total_pairs)

plt.figure(figsize=(10, 5))
plt.errorbar(x, ideal_mean,   yerr=ideal_sem,   fmt="o-", capsize=3, label="(1) holo ideal (cached)")
plt.errorbar(x, measerr_mean, yerr=measerr_sem, fmt="o-", capsize=3, label="(2) holo + meas err")
plt.errorbar(x, botherr_mean, yerr=botherr_sem, fmt="o-", capsize=3, label="(3) holo + meas+reset err")
plt.errorbar(x, exp_mean,     yerr=exp_sem,     fmt="o-", capsize=3, label="(4) sequential simulation / experiment")

plt.axhline(0, linewidth=1)
plt.xlabel(r"Pair index i (correlator is $Z_i Z_{i+1}$)")
plt.ylabel(r"$\langle Z_i Z_{i+1}\rangle$")
plt.title(f"ZZ correlators benchmark, N={N}, shots={shot_nums} (new protocol)")
plt.legend()
plt.tight_layout()
plt.show()


### benchmarking XX correlation

In [ ]:
from qubox.gates import *
from holographic_sim_updated.holographicSim import (
    holographic_sim,
    holographic_sim_cached,
)

import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Helper: coerce to column-stochastic convention (columns sum to 1)
# ----------------------------
def to_column_stochastic_2x2(M, name="matrix", atol=1e-3):
    """
    Ensure 2x2 matrix is column-stochastic: M[row, col] = P(row | true=col),
    so columns sum to 1. If it looks row-stochastic, transpose it.
    """
    A = np.asarray(M, dtype=float)
    if A.shape != (2, 2):
        raise ValueError(f"{name} must be shape (2,2), got {A.shape}")

    col_sums = A.sum(axis=0)
    row_sums = A.sum(axis=1)

    col_ok = np.allclose(col_sums, 1.0, atol=atol)
    row_ok = np.allclose(row_sums, 1.0, atol=atol)

    if row_ok and not col_ok:
        A = A.T
        col_sums = A.sum(axis=0)

    col_sums = np.where(col_sums == 0, 1.0, col_sums)
    A = A / col_sums[None, :]
    return A


# ----------------------------
# Build gate list U_gates
# ----------------------------
pi_ratio = 2
theta = np.pi / pi_ratio
rot = QubitRotation(
    theta, 0,
    b_x180_pulse="x180_pulse",
    b_y180_pulse="y180_pulse",
    build=False,
)

U = GateArray([rot], op_prefix="U")
U.build()

attr = experiment.attributes
N = 5
U_gates = [U] * N

fock_dim = 5
U_matrix_list = [gate.get_unitaries(fock_dim) for gate in U_gates]

experiment.burn_pulses()


# ----------------------------
# Observables
# ----------------------------
X = np.array([[0, 1],
              [1, 0]], dtype=np.complex128)

Z = np.array([[1,  0],
              [0, -1]], dtype=np.complex128)


# ----------------------------
# Run settings
# ----------------------------
shot_nums   = 5000
total_pairs = N - 1


# ----------------------------
# Measurement + reset parameters (new protocol, matches your ZZ/XX style)
# ----------------------------
# Perfect readout (no misclassification)
C_perfect = np.eye(2, dtype=float)

# Perfect reset-to-g (independent of reported bit)
R_reset_perfect = np.array([[1.0, 1.0],
                            [0.0, 0.0]], dtype=float)

# Measurement error only: use your calibrated confusion, but keep perfect reset
ro_quality_dict = measureMacro._ro_quality_params
confusion_raw = ro_quality_dict.get("confusion_matrix", None)
C_meas = to_column_stochastic_2x2(confusion_raw, name="confusion_matrix") if confusion_raw is not None else C_perfect

# Reset error: use your benchmarked effective reset matrix
# NOTE: replace this placeholder with your *measured / benchmarked* reset matrix.
R_reset = np.array([[0.90, 0.90],
                    [0.10, 0.10]], dtype=float)
R_reset = to_column_stochastic_2x2(R_reset, name="R_reset")

# New simulator API: single 2x2 for C_by_basis / R_by_basis, applied to X/Y/Z automatically
C_by_basis_perfect = C_perfect
R_by_basis_perfect = R_reset_perfect

C_by_basis_measerr = C_meas
R_by_basis_measerr = R_reset_perfect

C_by_basis_both    = C_meas
R_by_basis_both    = R_reset

# Reset requires a measurement every step; default reset measurement = Z
reset_op_list = [Z] * N


# ----------------------------
# Storage (per pair i,i+1)
# ----------------------------
ideal_samples   = np.zeros((total_pairs, shot_nums), dtype=np.complex128)
measerr_samples = np.zeros((total_pairs, shot_nums), dtype=np.complex128)
botherr_samples = np.zeros((total_pairs, shot_nums), dtype=np.complex128)

ideal_mean   = np.zeros(total_pairs, dtype=float)
measerr_mean = np.zeros(total_pairs, dtype=float)
botherr_mean = np.zeros(total_pairs, dtype=float)
exp_mean     = np.zeros(total_pairs, dtype=float)

ideal_sem   = np.zeros(total_pairs, dtype=float)
measerr_sem = np.zeros(total_pairs, dtype=float)
botherr_sem = np.zeros(total_pairs, dtype=float)
exp_sem     = np.zeros(total_pairs, dtype=float)

states_array = []


for i in range(total_pairs):
    # Correlator: measure X at sites i and i+1; elsewhere still measure Z for reset
    op_list = [None] * N
    op_list[i]     = X
    op_list[i + 1] = X

    # ------------------------------------------------------------
    # (1) IDEAL: no errors at all (perfect readout + perfect reset)
    # Cached exact sampler
    # ------------------------------------------------------------
    C0 = holographic_sim_cached(
        U_matrix_list,
        op_list,
        reset_op_list=reset_op_list,
        d=2,
        shot_nums=shot_nums,
        C_by_basis=C_by_basis_perfect,
        R_by_basis=R_by_basis_perfect,
        warn_branch_count=1000,
    )
    ideal_samples[i] = C0

    # ------------------------------------------------------------
    # (2) MEASUREMENT ERROR ONLY: confusion matrix, but perfect reset
    # Monte Carlo
    # ------------------------------------------------------------
    C1 = holographic_sim(
        U_matrix_list,
        op_list,
        reset_op_list=reset_op_list,
        d=2,
        shot_nums=shot_nums,
        C_by_basis=C_by_basis_measerr,
        R_by_basis=R_by_basis_measerr,
        parallel_process=False,
    )
    measerr_samples[i] = C1

    # ------------------------------------------------------------
    # (3) MEASUREMENT + RESET ERROR: confusion + benchmarked R_reset
    # ------------------------------------------------------------
    C2 = holographic_sim(
        U_matrix_list,
        op_list,
        reset_op_list=reset_op_list,
        d=2,
        shot_nums=shot_nums,
        C_by_basis=C_by_basis_both,
        R_by_basis=R_by_basis_both,
        parallel_process=False,
    )
    botherr_samples[i] = C2

    # Convert samples -> real expectation (XX should be real)
    c0r = np.real(C0)
    c1r = np.real(C1)
    c2r = np.real(C2)

    ideal_mean[i]   = c0r.mean()
    measerr_mean[i] = c1r.mean()
    botherr_mean[i] = c2r.mean()

    ideal_sem[i]   = c0r.std(ddof=1) / np.sqrt(shot_nums)
    measerr_sem[i] = c1r.std(ddof=1) / np.sqrt(shot_nums)
    botherr_sem[i] = c2r.std(ddof=1) / np.sqrt(shot_nums)

    # ------------------------------------------------------------
    # (4) ACTUAL EXPERIMENT / SEQ SIM STACK
    # ------------------------------------------------------------
    measurement_gates = [Measure(axis="none", target=attr.ro_el) for _ in range(N)]
    measurement_gates[i]     = Measure(axis="x", target=attr.ro_el)
    measurement_gates[i + 1] = Measure(axis="x", target=attr.ro_el)

    runres = experiment.sequential_simulation(U_gates, measurement_gates, shot_nums)
    states = runres.output.extract("states")
    states_array.append(states)

    # same convention as your ZZ script (expects "states" are g/e outcomes post basis-rotation)
    xx_each_shot = np.prod(analysis_tools.bools_to_sigma_z(states) * (-1), axis=1)
    exp_mean[i] = xx_each_shot.mean()
    exp_sem[i]  = xx_each_shot.std(ddof=1) / np.sqrt(shot_nums)


# ----------------------------
# Plot comparison
# ----------------------------
x = np.arange(total_pairs)

plt.figure(figsize=(10, 5))
plt.errorbar(x, ideal_mean,   yerr=ideal_sem,   fmt="o-", capsize=3, label="(1) holo ideal (cached)")
plt.errorbar(x, measerr_mean, yerr=measerr_sem, fmt="o-", capsize=3, label="(2) holo + meas err")
plt.errorbar(x, botherr_mean, yerr=botherr_sem, fmt="o-", capsize=3, label="(3) holo + meas+reset err")
plt.errorbar(x, exp_mean,     yerr=exp_sem,     fmt="o-", capsize=3, label="(4) sequential simulation / experiment")

plt.axhline(0, linewidth=1)
plt.xlabel(r"Pair index i (correlator is $X_i X_{i+1}$)")
plt.ylabel(r"$\langle X_i X_{i+1}\rangle$")
plt.title(f"XX correlators benchmark, N={N}, shots={shot_nums} (new protocol), R($\\pi$/{pi_ratio})")
plt.legend()
plt.tight_layout()
plt.show()


### benchmarking YY correlation

In [ ]:
from qubox.gates import *
from holographic_sim_updated.holographicSim import (
    holographic_sim,
    holographic_sim_cached,
)

import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Helper: coerce to column-stochastic convention (columns sum to 1)
# ----------------------------
def to_column_stochastic_2x2(M, name="matrix", atol=1e-3):
    """
    Ensure 2x2 matrix is column-stochastic: M[row, col] = P(row | true=col),
    so columns sum to 1. If it looks row-stochastic, transpose it.
    """
    A = np.asarray(M, dtype=float)
    if A.shape != (2, 2):
        raise ValueError(f"{name} must be shape (2,2), got {A.shape}")

    col_sums = A.sum(axis=0)
    row_sums = A.sum(axis=1)

    col_ok = np.allclose(col_sums, 1.0, atol=atol)
    row_ok = np.allclose(row_sums, 1.0, atol=atol)

    if row_ok and not col_ok:
        A = A.T
        col_sums = A.sum(axis=0)

    col_sums = np.where(col_sums == 0, 1.0, col_sums)
    A = A / col_sums[None, :]
    return A


# ----------------------------
# Build gate list U_gates
# ----------------------------
pi_ratio = 2
theta = np.pi / pi_ratio
rot = QubitRotation(
    theta, 0,
    b_x180_pulse="x180_pulse",
    b_y180_pulse="y180_pulse",
    build=False,
)

U = GateArray([rot], op_prefix="U")
U.build()

attr = experiment.attributes
N = 5
U_gates = [U] * N

fock_dim = 5
U_matrix_list = [gate.get_unitaries(fock_dim) for gate in U_gates]

experiment.burn_pulses()


# ----------------------------
# Observables
# ----------------------------
Y = np.array([[0, -1j],
              [1j,  0]], dtype=np.complex128)

Z = np.array([[1,  0],
              [0, -1]], dtype=np.complex128)


# ----------------------------
# Run settings
# ----------------------------
shot_nums   = 5000
total_pairs = N - 1


# ----------------------------
# Measurement + reset parameters (new API)
# ----------------------------
# Perfect readout (no misclassification)
C_perfect = np.eye(2, dtype=float)

# Perfect reset-to-g (independent of reported bit)
R_reset_perfect = np.array([[1.0, 1.0],
                            [0.0, 0.0]], dtype=float)

# Measurement error only: use your calibrated confusion, but keep perfect reset
ro_quality_dict = measureMacro._ro_quality_params
confusion_raw = ro_quality_dict.get("confusion_matrix", None)
C_meas = to_column_stochastic_2x2(confusion_raw, name="confusion_matrix") if confusion_raw is not None else C_perfect

# Reset error: use your benchmarked effective reset matrix
# NOTE: replace this placeholder with your *measured / benchmarked* reset matrix.
R_reset = np.array([[0.90, 0.90],
                    [0.10, 0.10]], dtype=float)
R_reset = to_column_stochastic_2x2(R_reset, name="R_reset")

# In the new simulator API, you can pass a single 2x2 matrix for C_by_basis or R_by_basis.
# That single matrix is applied to X/Y/Z automatically.
C_by_basis_perfect = C_perfect
R_by_basis_perfect = R_reset_perfect

C_by_basis_measerr = C_meas
R_by_basis_measerr = R_reset_perfect

C_by_basis_both    = C_meas
R_by_basis_both    = R_reset

# Because reset requires a measurement every step, provide default reset measurement as Z.
reset_op_list = [Z] * N


# ----------------------------
# Storage (per pair i,i+1)
# ----------------------------
ideal_samples   = np.zeros((total_pairs, shot_nums), dtype=np.complex128)
measerr_samples = np.zeros((total_pairs, shot_nums), dtype=np.complex128)
botherr_samples = np.zeros((total_pairs, shot_nums), dtype=np.complex128)

ideal_mean   = np.zeros(total_pairs, dtype=float)
measerr_mean = np.zeros(total_pairs, dtype=float)
botherr_mean = np.zeros(total_pairs, dtype=float)
exp_mean     = np.zeros(total_pairs, dtype=float)

ideal_sem   = np.zeros(total_pairs, dtype=float)
measerr_sem = np.zeros(total_pairs, dtype=float)
botherr_sem = np.zeros(total_pairs, dtype=float)
exp_sem     = np.zeros(total_pairs, dtype=float)

states_array = []


for i in range(total_pairs):
    # Correlator: measure Y at sites i and i+1; elsewhere still measure Z for reset
    op_list = [None] * N
    op_list[i]     = Y
    op_list[i + 1] = Y

    # ------------------------------------------------------------
    # (1) IDEAL: no errors at all (perfect readout + perfect reset)
    # Use cached exact sampler
    # ------------------------------------------------------------
    C0 = holographic_sim_cached(
        U_matrix_list,
        op_list,
        reset_op_list=reset_op_list,
        d=2,
        shot_nums=shot_nums,
        C_by_basis=C_by_basis_perfect,
        R_by_basis=R_by_basis_perfect,
        warn_branch_count=1000,
    )
    ideal_samples[i] = C0

    # ------------------------------------------------------------
    # (2) MEASUREMENT ERROR ONLY: confusion matrix, but perfect reset
    # Use Monte Carlo
    # ------------------------------------------------------------
    C1 = holographic_sim(
        U_matrix_list,
        op_list,
        reset_op_list=reset_op_list,
        d=2,
        shot_nums=shot_nums,
        C_by_basis=C_by_basis_measerr,
        R_by_basis=R_by_basis_measerr,
        parallel_process=False,
    )
    measerr_samples[i] = C1

    # ------------------------------------------------------------
    # (3) MEASUREMENT + RESET ERROR: confusion + benchmarked R_reset
    # ------------------------------------------------------------
    C2 = holographic_sim(
        U_matrix_list,
        op_list,
        reset_op_list=reset_op_list,
        d=2,
        shot_nums=shot_nums,
        C_by_basis=C_by_basis_both,
        R_by_basis=R_by_basis_both,
        parallel_process=False,
    )
    botherr_samples[i] = C2

    # Convert samples -> real expectation (YY should be real)
    c0r = np.real(C0)
    c1r = np.real(C1)
    c2r = np.real(C2)

    ideal_mean[i]   = c0r.mean()
    measerr_mean[i] = c1r.mean()
    botherr_mean[i] = c2r.mean()

    ideal_sem[i]   = c0r.std(ddof=1) / np.sqrt(shot_nums)
    measerr_sem[i] = c1r.std(ddof=1) / np.sqrt(shot_nums)
    botherr_sem[i] = c2r.std(ddof=1) / np.sqrt(shot_nums)

    # ------------------------------------------------------------
    # (4) ACTUAL EXPERIMENT / SEQ SIM STACK
    # ------------------------------------------------------------
    measurement_gates = [Measure(axis="none", target=attr.ro_el) for _ in range(N)]
    measurement_gates[i]     = Measure(axis="y", target=attr.ro_el)
    measurement_gates[i + 1] = Measure(axis="y", target=attr.ro_el)

    runres = experiment.sequential_simulation(U_gates, measurement_gates, shot_nums)
    states = runres.output.extract("states")
    states_array.append(states)

    # same convention as your ZZ script (expects "states" are g/e outcomes post basis-rotation)
    yy_each_shot = np.prod(analysis_tools.bools_to_sigma_z(states) * (-1), axis=1)
    exp_mean[i] = yy_each_shot.mean()
    exp_sem[i]  = yy_each_shot.std(ddof=1) / np.sqrt(shot_nums)


# ----------------------------
# Plot comparison
# ----------------------------
x = np.arange(total_pairs)

plt.figure(figsize=(10, 5))
plt.errorbar(x, ideal_mean,   yerr=ideal_sem,   fmt="o-", capsize=3, label="(1) holo ideal (cached)")
plt.errorbar(x, measerr_mean, yerr=measerr_sem, fmt="o-", capsize=3, label="(2) holo + meas err")
plt.errorbar(x, botherr_mean, yerr=botherr_sem, fmt="o-", capsize=3, label="(3) holo + meas+reset err")
plt.errorbar(x, exp_mean,     yerr=exp_sem,     fmt="o-", capsize=3, label="(4) sequential simulation / experiment")

plt.axhline(0, linewidth=1)
plt.xlabel(r"Pair index i (correlator is $Y_i Y_{i+1}$)")
plt.ylabel(r"$\langle Y_i Y_{i+1}\rangle$")
plt.title(f"YY correlators benchmark, N={N}, shots={shot_nums} (new protocol), R($\\pi$/{pi_ratio})")
plt.legend()
plt.tight_layout()
plt.show()


## Cluster state unitary gate decompostion evolution

### cluster state

In [ ]:
experiment.attributes.qb_fq = 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# If you need these explicitly (depends on your environment):
# from qubox.analysis import analysis_tools

from holographic_sim_updated.holographicSim import *  # holographic_sim, build_T1T2_kraus, PAULI_*, etc.

# ============================================================
# 0) Config
# ============================================================
path = "cluster_state_gate_data.json"

shot_nums = 10_000
N = 1
axes = ["X", "Y", "Z"]
operators_to_sim = {"X": PAULI_X, "Y": PAULI_Y, "Z": PAULI_Z}

latency_dt = 368e-9  # 368 ns

# ============================================================
# 1) Load gates (JSON + matrix npz)
# ============================================================
mgr  = experiment.pulseOpMngr
attr = experiment.attributes

gate_objs = list(reversed(load_gates(path, mgr, attr, False)))

data = np.load("cluster_state_u.npz", allow_pickle=True)
U_mats = list(reversed([data[key] for key in data.files]))  # (d,d) matrices

num = min(len(gate_objs), len(U_mats))
gate_names = [gate_objs[k].__class__.__name__ for k in range(num)]
x = np.arange(num)

# ============================================================
# 2) Readout calibration + latency channel
# ============================================================
ro_cal = measureMacro.get_readout_calibration()
C_meas = np.asarray(ro_cal["confusion_matrix"], dtype=float)
T_meas = np.asarray(ro_cal["transition_matrix"], dtype=float)

# attrs are ns -> seconds
T1 = float(attr.qb_T1_relax) * 1e-9
T2 = float(attr.qb_T2_ramsey) * 1e-9

latency_channel_sys = build_T1T2_kraus(dt=latency_dt, T1=T1, T2=T2)

print("=== Using calibration ===")
print("C_meas:\n", C_meas)
print("T_meas:\n", T_meas)
print(f"Latency: dt={latency_dt*1e9:.1f} ns, T1={T1*1e6:.3f} us, T2={T2*1e6:.3f} us")
print("=========================")

# ============================================================
# 3) Helper: matrix-prefix simulation INCLUDING C/T/latency
# ============================================================
def simulate_prefix_channel(prefix_channel, op_matrix):
    channel_list = [prefix_channel] * N
    op_list = [op_matrix]
    samples = holographic_sim(
        channel_list,
        op_list,
        shot_nums=shot_nums,
        show_progress=False,
        parallel_process=True,
        # match experiment measurement/reset knobs
        C_meas=C_meas,
        T_meas=T_meas,
        latency_channel_sys=latency_channel_sys,
    )
    return float(np.real_if_close(samples.mean()))

# ============================================================
# 4) Storage
# ============================================================
results_exp = {ax: np.zeros(num, dtype=float) for ax in axes}  # sequential_sim (experiment)
results_U   = {ax: np.zeros(num, dtype=float) for ax in axes}  # matrix sim (U_mats + C/T/latency)

# Optional: raw states from experiment
states_by_axis_by_k = {ax: [] for ax in axes}

# ============================================================
# 5) Prefix sweep
# ============================================================
print(f"Running prefix sweep: num_prefixes={num}, shots={shot_nums}, N={N}")

U_total = None  # prefix unitary (matrix)
burned_once = False

for k in tqdm(range(1, num + 1), desc="Prefix sweep (experiment vs matrix sim w/ C/T/latency)"):
    # --------------------------------------------------------
    # (A) Build prefix U_gate from JSON (rebuild each k)
    # --------------------------------------------------------
    prefix_gate_list = gate_objs[:k]
    U_gate = GateArray(prefix_gate_list, op_prefix="U")
    U_gate.build()

    # Usually burning once is enough; if you see missing waveform errors,
    # move burn_pulses() inside the loop (every k).
    if not burned_once:
        experiment.burn_pulses()
        burned_once = True

    U_gates = [U_gate] * N
    experiment.burn_pulses()
    # --------------------------------------------------------
    # (B) Experiment sequential_sim: <X>,<Y>,<Z>
    # --------------------------------------------------------
    for ax in axes:
        measurement_gates = [Measure(axis="none", target=attr.ro_el) for _ in range(N)]
        measurement_gates[0] = Measure(axis=ax, target=attr.ro_el)

        runres = experiment.sequential_simulation(U_gates, measurement_gates, shot_nums)
        states = runres.output.extract("states")  # shape (shot_nums, N)

        states_by_axis_by_k[ax].append(states)

        sigma = analysis_tools.bools_to_sigma_z(states) * (-1)  # -> ±1 convention
        results_exp[ax][k - 1] = float(sigma[:, 0].mean())

    # --------------------------------------------------------
    # (C) Matrix-prefix sim: build U_total then simulate with C/T/latency
    # --------------------------------------------------------
    if U_total is None:
        U_total = U_mats[0]
    else:
        U_total = U_mats[k - 1] @ U_total  # your convention

    for ax in axes:
        results_U[ax][k - 1] = simulate_prefix_channel(U_total, operators_to_sim[ax])

# ============================================================
# 6) Plot 1: expectations with 3 rows x 2 cols legend
#    (col1 = simulated, col2 = experiment)
# ============================================================
fig, axp = plt.subplots(figsize=(max(10, 0.7 * num), 6))

handles, labels = [], []

for name in ["X", "Y", "Z"]:
    # simulated (column 1)
    (line,) = axp.plot(
        x, results_U[name],
        linestyle="-", marker="o", markersize=3,
    )
    c = line.get_color()

    # experiment (column 2)
    scat = axp.scatter(
        x, results_exp[name],
        color=c, marker="^", s=35, zorder=5,
    )

    # order matters: row-wise fill for ncols=2
    handles += [line, scat]
    labels  += [f"{name}: matrix (cluster_state_u.npz)",
                f"{name}: sequential_sim (experiment)"]

axp.set_title(f"Prefix expectations: experiment vs matrix sim (with C/T/latency)\n(N={N}, shots={shot_nums})")
axp.set_xlabel("Gate prefix (by name)")
axp.set_ylabel("Expectation value")
axp.grid(True, alpha=0.3)
axp.set_xticks(x)
axp.set_xticklabels(gate_names, rotation=35, ha="right")

axp.legend(
    handles, labels,
    ncols=2,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.18),  # above axes; remove if you want it inside
    columnspacing=1.6,
    handletextpad=0.6,
    frameon=True,
)

fig.tight_layout()
plt.show()

# ============================================================
# 7) Plot 2: absolute differences
# ============================================================
fig, axd = plt.subplots(figsize=(max(10, 0.7 * num), 4.5))

for name in ["X", "Y", "Z"]:
    axd.plot(
        x,
        np.abs(results_exp[name] - results_U[name]),
        marker="o",
        markersize=3,
        label=f"|Δ<{name}>|",
    )

axd.set_title("Absolute difference: experiment vs matrix sim (with C/T/latency)")
axd.set_xlabel("Gate prefix (by name)")
axd.set_ylabel("Absolute difference")
axd.grid(True, alpha=0.3)
axd.set_xticks(x)
axd.set_xticklabels(gate_names, rotation=35, ha="right")
axd.legend()

fig.tight_layout()
plt.show()




# Test

### reference seq sim 

In [ ]:
reference_clks+0*rotation_clks

In [ ]:
TOF = experiment.quaProgMngr.qm_config["elements"]["resonator"]["time_of_flight"]
instrinsic_gap = 88
calculated_gap = TOF + instrinsic_gap

In [ ]:
#Test Tomography

### Test active qb reset 

In [ ]:
from qm.qua import *
with program() as prog:
    v1 = declare(fixed, )
    v2 = declare(fixed, )
    v3 = declare(fixed, )
    v4 = declare(fixed, )
    v5 = declare(fixed, )
    v6 = declare(fixed, )
    v7 = declare(bool, )
    v8 = declare(bool, )
    v9 = declare(bool, )
    v10 = declare(int, )
    v11 = declare(int, )
    v12 = declare(bool, )
    with for_(v10,0,(v10<20000),(v10+1)):
        assign(v11, 0)
        assign(v12, False)
        with while_(((v12^True)&(v11<1))):
            measure("readout"*amp(1), "resonator", dual_demod.full("rot_cos", "rot_sin", v1), dual_demod.full("rot_m_sin", "rot_cos", v2))
            align()
            assign(v7, (v1>0.00010764933863591362))
            measure("readout"*amp(1), "resonator", dual_demod.full("rot_cos", "rot_sin", v3), dual_demod.full("rot_m_sin", "rot_cos", v4))
            align()
            assign(v8, (v3>0.00010764933863591362))
            measure("readout"*amp(1), "resonator", dual_demod.full("rot_cos", "rot_sin", v5), dual_demod.full("rot_m_sin", "rot_cos", v6))
            align()
            assign(v9, (v5>0.00010764933863591362))
            assign(v12, (((((v1-4.4091386668958633e-05)*(v1-4.4091386668958633e-05))+((v2-0.00033789569649799004)*(v2-0.00033789569649799004)))<=6.180137024162879e-08)&(((((v1-0.000175650449365668)*(v1-0.000175650449365668))+((v2-0.00033745360198310137)*(v2-0.00033745360198310137)))<=6.33728504268862e-08)^True)))
            with if_((v12^True)):
                play("x180", "qubit", condition=(v1>0.00010764933863591362))
                align()
            assign(v11, (v11+1))
        r1 = declare_stream()
        save(v7, r1)
        save(v8, r1)
        save(v9, r1)
        r2 = declare_stream()
        save(v1, r2)
        r3 = declare_stream()
        save(v2, r3)
        r4 = declare_stream()
        save(v3, r4)
        r5 = declare_stream()
        save(v4, r5)
        r6 = declare_stream()
        save(v5, r6)
        r7 = declare_stream()
        save(v6, r7)
        r8 = declare_stream()
        save(v12, r8)
        r9 = declare_stream()
        save(v11, r9)
        assign(v11, 0)
        assign(v12, False)
        wait(1000)
        with while_(((v12^True)&(v11<256))):
            measure("readout"*amp(1), "resonator", dual_demod.full("rot_cos", "rot_sin", v1), dual_demod.full("rot_m_sin", "rot_cos", v2))
            align()
            assign(v7, (v1>0.00010764933863591362))
            measure("readout"*amp(1), "resonator", dual_demod.full("rot_cos", "rot_sin", v3), dual_demod.full("rot_m_sin", "rot_cos", v4))
            align()
            assign(v8, (v3>0.00010764933863591362))
            measure("readout"*amp(1), "resonator", dual_demod.full("rot_cos", "rot_sin", v5), dual_demod.full("rot_m_sin", "rot_cos", v6))
            align()
            assign(v9, (v5>0.00010764933863591362))
            assign(v12, (((((v1-0.000175650449365668)*(v1-0.000175650449365668))+((v2-0.00033745360198310137)*(v2-0.00033745360198310137)))<=6.33728504268862e-08)&(((((v1-4.4091386668958633e-05)*(v1-4.4091386668958633e-05))+((v2-0.00033789569649799004)*(v2-0.00033789569649799004)))<=6.180137024162879e-08)^True)))
            with if_((v12^True)):
                play("x180", "qubit", condition=(v1<0.00010764933863591362))
                align()
            assign(v11, (v11+1))
        save(v7, r1)
        save(v8, r1)
        save(v9, r1)
        save(v1, r2)
        save(v2, r3)
        save(v3, r4)
        save(v4, r5)
        save(v5, r6)
        save(v6, r7)
        save(v12, r8)
        save(v11, r9)
        r10 = declare_stream()
        save(v10, r10)
        wait(10000, )
    with stream_processing():
        r1.buffer(3).buffer(2).buffer(20000).save("states")
        r2.buffer(2).buffer(20000).save("I0")
        r3.buffer(2).buffer(20000).save("Q0")
        r4.buffer(2).buffer(20000).save("I1")
        r5.buffer(2).buffer(20000).save("Q1")
        r6.buffer(2).buffer(20000).save("I2")
        r7.buffer(2).buffer(20000).save("Q2")
        r8.buffer(2).buffer(20000).save("accept")
        r9.buffer(2).buffer(20000).save("tries")
        r10.save("iteration")

sim = experiment.quaProgMngr.simulate(
    prog,
    duration=8000,
    plot=True,
    plot_params={"time_unit": "ns", "which": "both", "title": "Debug"},
    controllers=("con1",),
    t_begin=0,
    t_end=2000,
)